<a href="https://colab.research.google.com/github/KojiKaiwa/D365-Finance-Automation/blob/main/D365_Data_Processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# 1. Excelファイルの読み込み
df = pd.read_excel('test_data.xlsx')

# 2. データの先頭5行を表示して中身を確認
print("--- データの先頭5行 ---")
display(df.head())

# 3. 基本的な統計量（件数、合計、平均など）を表示
print("\n--- データの統計情報 ---")
display(df.describe())

In [1]:
import pandas as pd

# ------------------------------------------------------------------
# 1. 各システムから出力されたデータを想定
# ------------------------------------------------------------------

# ① 加盟店売上データ（決済システム側：売上と計算上の手数料）
# 手数料（Fee）は一律3%と仮定。Net_Amount（差引入金予定額）= Gross_Amount - Fee
sales_data = {
    'Transaction_ID': ['TXN-001', 'TXN-002', 'TXN-003', 'TXN-004'],
    'Merchant_ID': ['MCH-101', 'MCH-102', 'MCH-103', 'MCH-104'],
    'Gross_Amount': [10000, 20000, 30000, 50000],
    'Expected_Fee': [300, 600, 900, 1500],
    'Expected_Net': [9700, 19400, 29100, 48500]
}
df_sales = pd.DataFrame(sales_data)

# ② 銀行入金データ（ファームバンキング/入金口座側）
# あえて TXN-002 の入金額を違わせ（手数料計算バグを想定）、TXN-004 は入金なし（未入金）にしています
bank_data = {
    'Transaction_ID': ['TXN-001', 'TXN-002', 'TXN-003'], # TXN-004が口座に未入金
    'Actual_Net': [9700, 19000, 29100] # TXN-002は計算と合わない（400円不足）
}
df_bank = pd.DataFrame(bank_data)

print("--- 監査対象：決済システム売上データ ---")
print(df_sales)

# ------------------------------------------------------------------
# 2. 今日のテーマ：トランザクションの外部結合（Left Join）と消込
# ------------------------------------------------------------------
# 売上データをベースに、実際の銀行入金データを Transaction_ID で結合します
df_recon = pd.merge(df_sales, df_bank, on='Transaction_ID', how='left', indicator=True)


# ------------------------------------------------------------------
# 3. 決済ビジネスにおける2大監査アラートの抽出
# ------------------------------------------------------------------

# 🚨 アラート①：システム未入金エラー（売上はあるが、銀行入金データが紐づかない）
# _merge が 'left_only' のレコードを抽出します
unpaid_errors = df_recon[df_recon['_merge'] == 'left_only']

# 🚨 アラート②：金額不一致エラー（入金はあるが、予定額と実際の入金額が1円でもズレている）
# 入金があり（both）、かつ予定額（Expected_Net）と実際の額（Actual_Net）が異なるもの
amount_mismatches = df_recon[
    (df_recon['_merge'] == 'both') &
    (df_recon['Expected_Net'] != df_recon['Actual_Net'])
]

# ------------------------------------------------------------------
# 4. 実行結果の出力
# ------------------------------------------------------------------
print("\n🚨 【監査アラート1】銀行未入金（消込エラー） 🚨")
if not unpaid_errors.empty:
    print(unpaid_errors[['Transaction_ID', 'Merchant_ID', 'Expected_Net']])
else:
    print("未入金データはありません。")

print("\n🚨 【監査アラート2】入金金額不一致（手数料等の計算異常） 🚨")
if not amount_mismatches.empty:
    # 差額（Variance）を計算して追加表示
    amount_mismatches['Variance'] = amount_mismatches['Expected_Net'] - amount_mismatches['Actual_Net']
    print(amount_mismatches[['Transaction_ID', 'Merchant_ID', 'Expected_Net', 'Actual_Net', 'Variance']])
else:
    print("金額の不一致はありません。")

--- 監査対象：決済システム売上データ ---
  Transaction_ID Merchant_ID  Gross_Amount  Expected_Fee  Expected_Net
0        TXN-001     MCH-101         10000           300          9700
1        TXN-002     MCH-102         20000           600         19400
2        TXN-003     MCH-103         30000           900         29100
3        TXN-004     MCH-104         50000          1500         48500

🚨 【監査アラート1】銀行未入金（消込エラー） 🚨
  Transaction_ID Merchant_ID  Expected_Net
3        TXN-004     MCH-104         48500

🚨 【監査アラート2】入金金額不一致（手数料等の計算異常） 🚨
  Transaction_ID Merchant_ID  Expected_Net  Actual_Net  Variance
1        TXN-002     MCH-102         19400     19000.0     400.0


/tmp/ipykernel_1895/4206337940.py:63: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  amount_mismatches['Variance'] = amount_mismatches['Expected_Net'] - amount_mismatches['Actual_Net']


In [2]:
import pandas as pd

# ------------------------------------------------------------------
# 1. 各システムから出力されたデータを想定
# ------------------------------------------------------------------

# ① 加盟店売上データ（決済システム側：売上と計算上の手数料）
# 手数料（Fee）は一律3%と仮定。Net_Amount（差引入金予定額）= Gross_Amount - Fee
sales_data = {
    'Transaction_ID': ['TXN-001', 'TXN-002', 'TXN-003', 'TXN-004'],
    'Merchant_ID': ['MCH-101', 'MCH-102', 'MCH-103', 'MCH-104'],
    'Gross_Amount': [10000, 20000, 30000, 50000],
    'Expected_Fee': [300, 600, 900, 1500],
    'Expected_Net': [9700, 19400, 29100, 48500]
}
df_sales = pd.DataFrame(sales_data)

# ② 銀行入金データ（ファームバンキング/入金口座側）
# あえて TXN-002 の入金額を違わせ（手数料計算バグを想定）、TXN-004 は入金なし（未入金）にしています
bank_data = {
    'Transaction_ID': ['TXN-001', 'TXN-002', 'TXN-003'], # TXN-004が口座に未入金
    'Actual_Net': [9700, 19000, 29100] # TXN-002は計算と合わない（400円不足）
}
df_bank = pd.DataFrame(bank_data)

print("--- 監査対象：決済システム売上データ ---")
print(df_sales)

# ------------------------------------------------------------------
# 2. 今日のテーマ：トランザクションの外部結合（Left Join）と消込
# ------------------------------------------------------------------
# 売上データをベースに、実際の銀行入金データを Transaction_ID で結合します
df_recon = pd.merge(df_sales, df_bank, on='Transaction_ID', how='left', indicator=True)


# ------------------------------------------------------------------
# 3. 決済ビジネスにおける2大監査アラートの抽出
# ------------------------------------------------------------------

# 🚨 アラート①：システム未入金エラー（売上はあるが、銀行入金データが紐づかない）
# _merge が 'left_only' のレコードを抽出します
unpaid_errors = df_recon[df_recon['_merge'] == 'left_only']

# 🚨 アラート②：金額不一致エラー（入金はあるが、予定額と実際の入金額が1円でもズレている）
# 入金があり（both）、かつ予定額（Expected_Net）と実際の額（Actual_Net）が異なるもの
amount_mismatches = df_recon[
    (df_recon['_merge'] == 'both') &
    (df_recon['Expected_Net'] != df_recon['Actual_Net'])
].copy()

# amount_mismatches = df_recon[
#     (df_recon['_merge'] == 'both') &
#     (df_recon['Expected_Net'] != df_recon['Actual_Net'])
# ]

# ------------------------------------------------------------------
# 4. 実行結果の出力
# ------------------------------------------------------------------
print("\n🚨 【監査アラート1】銀行未入金（消込エラー） 🚨")
if not unpaid_errors.empty:
    print(unpaid_errors[['Transaction_ID', 'Merchant_ID', 'Expected_Net']])
else:
    print("未入金データはありません。")

print("\n🚨 【監査アラート2】入金金額不一致（手数料等の計算異常） 🚨")
if not amount_mismatches.empty:
    # 差額（Variance）を計算して追加表示
    amount_mismatches['Variance'] = amount_mismatches['Expected_Net'] - amount_mismatches['Actual_Net']
    print(amount_mismatches[['Transaction_ID', 'Merchant_ID', 'Expected_Net', 'Actual_Net', 'Variance']])
else:
    print("金額の不一致はありません。")

--- 監査対象：決済システム売上データ ---
  Transaction_ID Merchant_ID  Gross_Amount  Expected_Fee  Expected_Net
0        TXN-001     MCH-101         10000           300          9700
1        TXN-002     MCH-102         20000           600         19400
2        TXN-003     MCH-103         30000           900         29100
3        TXN-004     MCH-104         50000          1500         48500

🚨 【監査アラート1】銀行未入金（消込エラー） 🚨
  Transaction_ID Merchant_ID  Expected_Net
3        TXN-004     MCH-104         48500

🚨 【監査アラート2】入金金額不一致（手数料等の計算異常） 🚨
  Transaction_ID Merchant_ID  Expected_Net  Actual_Net  Variance
1        TXN-002     MCH-102         19400     19000.0     400.0


In [ ]:
import pandas as pd

# ------------------------------------------------------------------
# 1. 各システムから出力されたデータを想定（キー：PO番号、品目コード）
# ------------------------------------------------------------------

# ① 発注データ（PO）: 購買システム
po_data = {
    'PO_Number': ['PO-001', 'PO-002', 'PO-003'],
    'Item_Code': ['ITEM-A', 'ITEM-B', 'ITEM-C'],
    'PO_Qty': [100, 50, 10],
    'PO_Amount': [100000, 50000, 20000]
}
df_po = pd.DataFrame(po_data)

# ② 入庫データ（GR）: サプライチェーン/倉庫管理システム (SCM)
# あえて PO-002 の納品数量を「40個（10個不足）」にしています
gr_data = {
    'PO_Number': ['PO-001', 'PO-002', 'PO-003'],
    'Item_Code': ['ITEM-A', 'ITEM-B', 'ITEM-C'],
    'GR_Qty': [100, 40, 10], # PO-002が不一致
}
df_gr = pd.DataFrame(gr_data)

# ③ 請求データ（INV）: 会計/購買債務システム (AP)
# あえて PO-003 の請求金額を「25,000円（過大請求）」にしています
inv_data = {
    'PO_Number': ['PO-001', 'PO-002', 'PO-003'],
    'Item_Code': ['ITEM-A', 'ITEM-B', 'ITEM-C'],
    'INV_Amount': [100000, 50000, 25000] # PO-003が不一致
}
df_inv = pd.DataFrame(inv_data)

# ------------------------------------------------------------------
# 2. 今日のテーマ：複数テーブルの多角的結合（マルチJOIN）
# ------------------------------------------------------------------
# 複合キー（PO番号 ＋ 品目コード）で3つのテーブルを順番に内結合（Inner Join）します

# まずPOとGRを結合
df_merged = pd.merge(df_po, df_gr, on=['PO_Number', 'Item_Code'], how='inner')

# 次に、その結果とINVを結合して1つの大きな照合用テーブルを作る
df_match_table = pd.merge(df_merged, df_inv, on=['PO_Number', 'Item_Code'], how='inner')


# ------------------------------------------------------------------
# 3. 3面照合（3-Way Matching）の監査ロジック
# ------------------------------------------------------------------
# 条件A: 発注数量（PO_Qty）と 入庫数量（GR_Qty）が一致しているか
# 条件B: 発注金額（PO_Amount）と 請求金額（INV_Amount）が一致しているか

df_match_table['Qty_Match'] = df_match_table['PO_Qty'] == df_match_table['GR_Qty']
df_match_table['Amount_Match'] = df_match_table['PO_Amount'] == df_match_table['INV_Amount']

print("=== 3面照合 総合検証テーブル ===")
print(df_match_table[['PO_Number', 'Item_Code', 'Qty_Match', 'Amount_Match']])


# ------------------------------------------------------------------
# 4. PMO/監査用：不一致（エラー）レコードの抽出
# ------------------------------------------------------------------
# 数量、または金額のどちらかが False（不一致）のものを探す（COBOLのOR条件判定）
errors = df_match_table[(df_match_table['Qty_Match'] == False) | (df_match_table['Amount_Match'] == False)]

print("\n🚨 【要確認】3面照合エラー（検収・請求不整合リターン） 🚨")
if not errors.empty:
    print(errors[['PO_Number', 'Item_Code', 'PO_Qty', 'GR_Qty', 'PO_Amount', 'INV_Amount']])
else:
    print("すべてのデータが完全に3面照合されました。内部統制は健全です。")

=== 3面照合 総合検証テーブル ===
  PO_Number Item_Code  Qty_Match  Amount_Match
0    PO-001    ITEM-A       True          True
1    PO-002    ITEM-B      False          True
2    PO-003    ITEM-C       True         False

🚨 【要確認】3面照合エラー（検収・請求不整合リターン） 🚨
  PO_Number Item_Code  PO_Qty  GR_Qty  PO_Amount  INV_Amount
1    PO-002    ITEM-B      50      40      50000       50000
2    PO-003    ITEM-C      10      10      20000       25000


In [ ]:
import pandas as pd

# ==========================================
# 準備：前回のクレンジング済ベンダーマスター
# ==========================================
vendor_data = {
    'VendorAccount': ['VEND001', 'VEND002', 'VEND003'],
    'VendorName': ['株式会社ABC', '有限会社東京商事', '西日本インダストリ'],
    'TaxRegistrationNumber': ['11111', '22222', '33333']
}
df_vendors = pd.DataFrame(vendor_data)

# ==========================================
# 準備：D365から出力された発注書（PO）データ
# （監査対象：あえて不整合なデータを混ぜています）
# ==========================================
po_data = {
    'PurchaseOrder': ['PO26-0001', 'PO26-0002', 'PO26-0003', 'PO26-0004'],
    'VendorAccount': ['VEND001', 'VEND002', 'VEND099', 'VEND003'], # VEND099はマスター未登録
    'Amount': [500000, 1200000, 350000, 80000]
}
df_po = pd.DataFrame(po_data)

print("--- 監査対象：発注書（PO）データ ---")
print(df_po)

# ==========================================
# 今日のテーマ：データの結合（JOIN）とクロスチェック
# ==========================================

# 1. 外部結合（Left Join）を実行
# 発注書（PO）をベースに、ベンダーマスターを「VendorAccount」をキーにして結合します
# indicator=True をつけると、どちらのテーブルにデータが存在したかの判定フラグ（_merge）が作られます
df_audit = pd.merge(df_po, df_vendors, on='VendorAccount', how='left', indicator=True)

print("\n--- 結合直後の状態（確認用） ---")
print(df_audit)

# 2. 監査チェック：マスターに存在しない不正なPOを抽出（COBOLの不一致レコード抽出に相当）
# _merge カラムが 'left_only' になっているものが「マスター未登録」のデータです
unregistered_po = df_audit[df_audit['_merge'] == 'left_only']

print("\n🚨 【監査アラート】ベンダーマスター未登録の発注書 🚨")
if not unregistered_po.empty:
    # 監査レポート用に必要な列だけをピックアップして表示
    print(unregistered_po[['PurchaseOrder', 'VendorAccount', 'Amount']])
else:
    print("不整合データはありません。すべての発注書は正当なベンダーに関連付けられています。")

# ==========================================
# 今日の仕上げ：監査レポート（Excel）への出力
# ==========================================

# 1. 監査アラートデータがあるか確認
if not unregistered_po.empty:

    # 2. 【実務の工夫】レポート用に列を整理・並び替え
    # 不要な '_merge' や空白になったマスター側の列を除外し、見やすい順番にします
    report_columns = ['PurchaseOrder', 'VendorAccount', 'Amount']
    df_report = unregistered_po[report_columns]

    # 3. Excelファイルとして保存
    excel_report_name = 'Audit_Alert_Report.xlsx'

    df_report.to_excel(
        excel_report_name,
        sheet_name='Unregistered_PO_Alerts', # シート名
        index=False                          # 行番号（インデックス）は出力しない
    )

    print(f"【成功】監査アラートレポートを保存しました: {excel_report_name}")
    print("※Colabの左メニューのフォルダアイコンからダウンロードしてください。")

else:
    print("不整合データがないため、レポートは出力されませんでした。")

--- 監査対象：発注書（PO）データ ---
  PurchaseOrder VendorAccount   Amount
0     PO26-0001       VEND001   500000
1     PO26-0002       VEND002  1200000
2     PO26-0003       VEND099   350000
3     PO26-0004       VEND003    80000

--- 結合直後の状態（確認用） ---
  PurchaseOrder VendorAccount   Amount VendorName TaxRegistrationNumber  \
0     PO26-0001       VEND001   500000    株式会社ABC                 11111   
1     PO26-0002       VEND002  1200000   有限会社東京商事                 22222   
2     PO26-0003       VEND099   350000        NaN                   NaN   
3     PO26-0004       VEND003    80000  西日本インダストリ                 33333   

      _merge  
0       both  
1       both  
2  left_only  
3       both  

🚨 【監査アラート】ベンダーマスター未登録の発注書 🚨
  PurchaseOrder VendorAccount  Amount
2     PO26-0003       VEND099  350000
【成功】監査アラートレポートを保存しました: Audit_Alert_Report.xlsx
※Colabの左メニューのフォルダアイコンからダウンロードしてください。


In [ ]:
import pandas as pd
import re

# 1. テスト用の擬似ベンダーマスターデータ（スペース混入や重複がある状態）
data = {
    'VendorAccount': ['VEND001', 'VEND002 ', 'VEND001', 'VEND003', 'VEND004'],
    'VendorName': ['株式会社  ＡＢＣ', '東京商事 有限会社', '株式会社 ABC', '  西日本インダストリ  ', '東京商事 有限会社'],
    'TaxRegistrationNumber': ['1234567890123', '9876543210123', '1234567890123', '5555555555555', '9876543210123']
}
df = pd.DataFrame(data)
print("--- クレンジング前のデータ ---")
print(df)

# ==========================================
# 昨日のテーマ：不要スペース自動除去 と 重複一括排除
# ==========================================

# 機能A: 不要スペースの自動除去（文字列型の全カラムに適用）
def clean_spaces(text):
    if not isinstance(text, str):
        return text
    # 1. 前後のスペース（半角・全角）を削除 (Trim)
    text = text.strip()
    # 2. 文字列の途中にある「連続したスペース（半角・全角）」を半角スペース1個に統合
    text = re.sub(r'[\s　]+', ' ', text)
    return text

# データフレーム全体（文字列カラム）に適用
for col in df.columns:
    df[col] = df[col].astype(str).apply(clean_spaces)

# 機能B: 重複データの一括排除
# 「VendorAccount（ベンダーコード）」または「TaxRegistrationNumber（登録番号/法人番号）」が重複しているものを排除
# keep='first' で最初に見つかったレコードを残します
df_cleaned = df.drop_duplicates(subset=['VendorAccount'], keep='first')
# さらに法人番号などの一意のキーでも重複を排除したい場合
df_cleaned = df_cleaned.drop_duplicates(subset=['TaxRegistrationNumber'], keep='first')

print("\n--- クレンジング・重複排除後のデータ ---")
print(df_cleaned)

--- クレンジング前のデータ ---
  VendorAccount     VendorName TaxRegistrationNumber
0       VEND001      株式会社  ＡＢＣ         1234567890123
1      VEND002       東京商事 有限会社         9876543210123
2       VEND001       株式会社 ABC         1234567890123
3       VEND003    西日本インダストリ           5555555555555
4       VEND004      東京商事 有限会社         9876543210123

--- クレンジング・重複排除後のデータ ---
  VendorAccount VendorName TaxRegistrationNumber
0       VEND001   株式会社 ＡＢＣ         1234567890123
1       VEND002  東京商事 有限会社         9876543210123
3       VEND003  西日本インダストリ         5555555555555


In [ ]:
import pandas as pd
import re

# 1. 前株・後株が混在し、ベンダーコードが異なるデータを想定
data = {
    'VendorAccount': ['VEND005', 'VEND006', 'VEND007', 'VEND008'],
    'VendorName': ['株式会社ABC', 'ABC株式会社', '有限会社東京商事', '東京商事（有）'],
    'TaxRegistrationNumber': ['11111', '11111', '22222', '22222'] # 同一企業
}
df = pd.DataFrame(data)
print("--- クレンジング前のデータ ---")
print(df)

# ==========================================
# テキスト処理：前株・後株・法人格記号の除去パターン
# ==========================================
# (株) や （有）、株式会社、有限会社 などにマッチする正規表現パターン
company_types = r'(株式会社|有限会社|合同会社|＼(株＼)|＼(有＼)|（株）|（有）)'

def extract_pure_name(text):
    if not isinstance(text, str):
        return text
    # 正規表現で法人格の部分を「空文字（削除）」に置換
    # textの前後や途中、どこにあっても削除します
    cleaned = re.sub(company_types, '', text)
    # 念のため、前後に残ったスペースをトリミング
    return cleaned.strip()

# 「名寄せ判定用」の新しいカラムを作成して適用
df['MatchKey'] = df['VendorName'].apply(extract_pure_name)

print("\n--- 中間状態（MatchKeyを作成） ---")
print(df[['VendorName', 'MatchKey']])

# 「MatchKey」が同じものは重複とみなして排除（最初のレコードを残す）
df_cleaned = df.drop_duplicates(subset=['MatchKey'], keep='first')

# 不要になった中間カラムを削除（ドロップ）
df_final = df_cleaned.drop(columns=['MatchKey'])

print("\n--- 重複排除後の最終データ ---")
print(df_final)

--- クレンジング前のデータ ---
  VendorAccount VendorName TaxRegistrationNumber
0       VEND005    株式会社ABC                 11111
1       VEND006    ABC株式会社                 11111
2       VEND007   有限会社東京商事                 22222
3       VEND008    東京商事（有）                 22222

--- 中間状態（MatchKeyを作成） ---
  VendorName MatchKey
0    株式会社ABC      ABC
1    ABC株式会社      ABC
2   有限会社東京商事     東京商事
3    東京商事（有）     東京商事

--- 重複排除後の最終データ ---
  VendorAccount VendorName TaxRegistrationNumber
0       VEND005    株式会社ABC                 11111
2       VEND007   有限会社東京商事                 22222


In [ ]:
import pandas as pd

# （前回の処理で「df_final」が完成している前提です）

# ==========================================
# 今日の仕上げ：Dynamics 365用ファイル保存
# ==========================================

# パターン1: Excelファイル (.xlsx) として出力
# Dynamics 365のデータエンティティはExcel形式が最もトラブルが少なくて推奨されます。
excel_file_name = 'D365_VendorMaster_Cleaned.xlsx'

df_final.to_excel(
    excel_file_name,
    sheet_name='Vendors', # シート名（Dynamicsのエンティティ名に合わせると便利）
    index=False           # 左端の行番号（0, 2...）をファイルに書き出さない設定（超重要！）
)
print(f"【成功】Excelファイルを保存しました: {excel_file_name}")


# パターン2: CSVファイル (.csv) として出力
# CSVにする場合は、D365インポート時の日本語文字化けを防ぐため「UTF-8 (BOM付き)」にします。
csv_file_name = 'D365_VendorMaster_Cleaned.csv'

df_final.to_csv(
    csv_file_name,
    encoding='utf_8_sig', # 「_sig」をつけることでBOM付きになり、ExcelやD365で文字化けしません
    index=False           # 行番号を出力しない
)
print(f"【成功】CSVファイルを保存しました: {csv_file_name}")

【成功】Excelファイルを保存しました: D365_VendorMaster_Cleaned.xlsx
【成功】CSVファイルを保存しました: D365_VendorMaster_Cleaned.csv


In [ ]:
import pandas as pd

# 1. 旧システムから抽出した、スペース混入や重複があるベンダーデータ
# ※ 'Vendor_A '（末尾半角）、'　Vendor_B'（先頭全角）などの表記揺れを再現
raw_vendor_data = {
    'Vendor_Code': ['V001', 'V002', 'V003', 'V004', 'V005'],
    'Vendor_Name': ['Vendor_A ', '　Vendor_B', 'Vendor_A', 'Vendor_C', 'Vendor_B'], # V003, V005はクレンジング後に重複となるデータ
    'Registered_Date': ['2025-01-10', '2025-02-15', '2026-05-01', '2026-05-20', '2026-06-01'] # 後から登録された方が新しい
}
df_vendor = pd.DataFrame(raw_vendor_data)

print("--- [Execution Result 1] Raw Vendor Data Before Cleaning ---")
display(df_vendor)

# 2. 文字列クレンジング（前後の半角・全角スペースの一括除去）
# strip() に ' 　'（半角スペースと全角スペース）を指定することで、両方を同時に除去します
df_vendor['Vendor_Name'] = df_vendor['Vendor_Name'].str.strip(' 　')

# 3. 重複データの検出（監査用ログの作成）
# keep=False で重複している行をすべて抽出して確認します
duplicate_flags = df_vendor.duplicated(subset=['Vendor_Name'], keep=False)
df_duplicate_logs = df_vendor[duplicate_flags].copy()

# 4. 重複データの排除（実務用クレンジング）
# keep='last' を指定することで、古いデータを削除し、一番新しい登録日のデータ（行）を正として残します
df_vendor_cleaned = df_vendor.drop_duplicates(subset=['Vendor_Name'], keep='last').reset_index(drop=True)

print("\n--- [Execution Result 2] Detected Duplicate Vendors (Audit Log) ---")
if not df_duplicate_logs.empty:
    print(f"🚨 警告: スペース除去後、同一と判定された重複ベンダーが {len(df_duplicate_logs)} 件検出されました。")
    display(df_duplicate_logs.sort_values(by='Vendor_Name'))
else:
    print("✅ 重複ベンダーは検出されませんでした。")

print("\n--- [Execution Result 3] Cleaned Vendor Master ---")
display(df_vendor_cleaned)

--- [Execution Result 1] Raw Vendor Data Before Cleaning ---


,Vendor_Code,Vendor_Name,Registered_Date
0,V001,Vendor_A,2025-01-10
1,V002,Vendor_B,2025-02-15
2,V003,Vendor_A,2026-05-01
3,V004,Vendor_C,2026-05-20
4,V005,Vendor_B,2026-06-01



--- [Execution Result 2] Detected Duplicate Vendors (Audit Log) ---
🚨 警告: スペース除去後、同一と判定された重複ベンダーが 4 件検出されました。


,Vendor_Code,Vendor_Name,Registered_Date
0,V001,Vendor_A,2025-01-10
2,V003,Vendor_A,2026-05-01
1,V002,Vendor_B,2025-02-15
4,V005,Vendor_B,2026-06-01



--- [Execution Result 3] Cleaned Vendor Master ---


,Vendor_Code,Vendor_Name,Registered_Date
0,V003,Vendor_A,2026-05-01
1,V004,Vendor_C,2026-05-20
2,V005,Vendor_B,2026-06-01


In [ ]:
import pandas as pd

# 1. 旧システム等から出力された、日付フォーマットがバラバラな取引データ
uneven_date_data = {
    'TransactionID': ['TX001', 'TX002', 'TX003', 'TX004', 'TX005'],
    'Raw_Date': ['2026/05/29', '2026.05.30', '20260531', 'INVALID_DATE', '2026-06-01'],  # TX004は不正データ
    'Amount': [45000, 120000, 85000, 30000, 50000]
}
df_tx = pd.DataFrame(uneven_date_data)

# 2. 日付の一括標準化処理
# errors='coerce' を指定することで、変換できない不正な文字列を強制的に「NaT（日付の欠損）」に変換します
df_tx['Standard_Date'] = pd.to_datetime(df_tx['Raw_Date'], errors='coerce')

# 3. 変換後の日付から「YYYY-MM-DD」の文字列形式にフォーマットを固定
# 補足: NaTの行はそのまま空欄（None）になります
df_tx['Standard_Date'] = df_tx['Standard_Date'].dt.strftime('%Y-%m-%d')

# 4. 日付の標準化に失敗（データ化け・不正日付）した行を抽出
# 文字列変換後は None または NaN を isna() で捕捉できます
df_date_errors = df_tx[df_tx['Standard_Date'].isna()].copy()

print("=============================================================================")
print("             TRANSACTION DATE FORMAT ALIGNMENT REPORT                        ")
print("=============================================================================")
display(df_tx)

print("\n--- 🔍 Date Format Integrity Check Result ---")
if not df_date_errors.empty:
    print(f"🚨 警告: 標準日付フォーマット（YYYY-MM-DD）への変換に失敗した明細が {len(df_date_errors)} 件検出されました。")
    print("このままインポートするとERPのデータベースエラーを引き起こすため、対象行を保留します。")
    display(df_date_errors[['TransactionID', 'Raw_Date', 'Amount']])
else:
    print("✅ すべての取引日付が正常に標準フォーマット（YYYY-MM-DD）へ統一されました。")

             TRANSACTION DATE FORMAT ALIGNMENT REPORT                        


,TransactionID,Raw_Date,Amount,Standard_Date
0,TX001,2026/05/29,45000,2026-05-29
1,TX002,2026.05.30,120000,NaN
2,TX003,20260531,85000,NaN
3,TX004,INVALID_DATE,30000,NaN
4,TX005,2026-06-01,50000,NaN



--- 🔍 Date Format Integrity Check Result ---
🚨 警告: 標準日付フォーマット（YYYY-MM-DD）への変換に失敗した明細が 4 件検出されました。
このままインポートするとERPのデータベースエラーを引き起こすため、対象行を保留します。


,TransactionID,Raw_Date,Amount
1,TX002,2026.05.30,120000
2,TX003,20260531,85000
3,TX004,INVALID_DATE,30000
4,TX005,2026-06-01,50000


In [ ]:
import pandas as pd

# 1. 元のデータ
uneven_date_data = {
    'TransactionID': ['TX001', 'TX002', 'TX003', 'TX004', 'TX005'],
    'Raw_Date': ['2026/05/29', '2026.05.30', '20260531', 'INVALID_DATE', '2026-06-01'],
    'Amount': [45000, 120000, 85000, 30000, 50000]
}
df_tx_fixed = pd.DataFrame(uneven_date_data)

# 2. 【重要修正】 format='mixed' を指定して、バラバラな形式を1行ずつ解析させます
# errors='coerce' と組み合わせることで、本当に不正な文字列だけを NaT にします
df_tx_fixed['Standard_Date'] = pd.to_datetime(df_tx_fixed['Raw_Date'], errors='coerce', format='mixed')

# 3. YYYY-MM-DD の形式に統一
df_tx_fixed['Standard_Date'] = df_tx_fixed['Standard_Date'].dt.strftime('%Y-%m-%d')

# 4. 変換エラー（NaT / None になった行）を正確に抽出
df_date_errors_fixed = df_tx_fixed[df_tx_fixed['Standard_Date'].isna()].copy()

print("=============================================================================")
print("             TRANSACTION DATE FORMAT ALIGNMENT REPORT (FIXED)                ")
print("=============================================================================")
display(df_tx_fixed)

print("\n--- 🔍 Date Format Integrity Check Result ---")
if not df_date_errors_fixed.empty:
    print(f"🚨 警告: 標準日付フォーマットへの変換に失敗した明細が {len(df_date_errors_fixed)} 件検出されました。")
    display(df_date_errors_fixed[['TransactionID', 'Raw_Date', 'Amount']])
else:
    print("✅ すべての取引日付が正常に標準フォーマットへ統一されました。")

In [ ]:
import pandas as pd

# 1. 旧システムから抽出した仕訳明細データ（100件、1,000件規模のデータを想定）
legacy_detail_data = {
    'LineID': ['L001', 'L002', 'L003', 'L004'],
    'Amount': [50000, -50000, 120000, -120000]  # 合計すると0になる仕訳
}
df_legacy_detail = pd.DataFrame(legacy_detail_data)

# 2. 新システム（D365）にインポートされた仕訳明細データ
# ※ L004 のマイナス120,000円がインポート時に「12,000円」にデータ化け（ゼロが1つ欠損）した想定
new_detail_data = {
    'LineID': ['L001', 'L002', 'L003', 'L004'],
    'Amount': [50000, -50000, 120000, -12000]  # エラーデータ
}
df_new_detail = pd.DataFrame(new_detail_data)

# 3. 旧システム側の統計量（件数、および金額絶対値の合計）を算出
legacy_count = len(df_legacy_detail)
legacy_hash_total = df_legacy_detail['Amount'].abs().sum()

# 4. 新システム側の統計量を算出
new_count = len(df_new_detail)
new_hash_total = df_new_detail['Amount'].abs().sum()

# 5. 検証結果の判定用データフレームの作成
verification_summary = {
    'Metric': ['Total Record Count', 'Amount Hash Total (Absolute)'],
    'Legacy_System': [legacy_count, legacy_hash_total],
    'New_System': [new_count, new_hash_total]
}
df_verif_summary = pd.DataFrame(verification_summary)

# 差分の計算（新 - 旧）
df_verif_summary['Variance'] = df_verif_summary['New_System'] - df_verif_summary['Legacy_System']

print("=============================================================================")
print("             TRANSACTION BATCH INTEGRITY VERIFICATION REPORT                 ")
print("=============================================================================")
display(df_verif_summary)

print("\n--- 🔍 Batch Integrity Check Result ---")
# 件数の差分とハッシュトータルの差分がともに 0 であるか判定
if (df_verif_summary.loc[0, 'Variance'] == 0) and (df_verif_summary.loc[1, 'Variance'] == 0):
    print("✅ 【バッチ検証成功】新旧システム間で件数およびハッシュトータルが完全一致しました。")
    print("データの欠損や金額のデータ化けはありません。次のステージへ移行可能です。")
else:
    print("❌ 【バッチ検証失敗】データ不整合が検知されました。インポート処理に異常があります。")

    # 具体的なエラー原因の冷徹な切り分け
    if df_verif_summary.loc[0, 'Variance'] != 0:
        print(f"   → 原因①: レコード件数に {df_verif_summary.loc[0, 'Variance']} 件のズレがあります（移行漏れ、または重複）。")
    if df_verif_summary.loc[1, 'Variance'] != 0:
        print(f"   → 原因②: 金額のハッシュトータルに {df_verif_summary.loc[1, 'Variance']:,} 円の差額があります（金額のデータ化け）。")

In [ ]:
import pandas as pd

# 1. 新旧の明細データフレームを LineID をキーにして内部結合（Inner Merge）
# ※件数が一致しているため、LineIDごとの金額の横並び比較を行います
df_line_compare = pd.merge(
    df_legacy_detail,
    df_new_detail,
    on='LineID',
    suffixes=('_Legacy', '_New')
)

# 2. 行単位の差額を計算（新システム金額 - 旧システム金額）
df_line_compare['Line_Variance'] = df_line_compare['Amount_New'] - df_line_compare['Amount_Legacy']

# 3. 差額が 0 ではない（金額が化けている）行のみをピンポイント抽出
df_corrupted_lines = df_line_compare[df_line_compare['Line_Variance'] != 0].copy()

print("=============================================================================")
print("             LINE-LEVEL DATA CORRUPTION DEBUG REPORT                         ")
print("=============================================================================")

if not df_corrupted_lines.empty:
    print(f"🚨 修正必要箇所: 金額がデータ化けしている明細が {len(df_corrupted_lines)} 件特定されました。\n")
    display(df_corrupted_lines[['LineID', 'Amount_Legacy', 'Amount_New', 'Line_Variance']])

    # 4. データ移行チーム（またはエンジニア）向けの具体的なデバッグ指示を出力
    print("📝 【エンジニア向けデバッグ指示書】")
    print("--------------------------------------------------------------------------------")
    for _, row in df_corrupted_lines.iterrows():
        print(f"💡 [LineID: {row['LineID']}]")
        print(f"   - 旧システム実績値: {row['Amount_Legacy']:,} 円")
        print(f"   - 新システム登録値: {row['Amount_New']:,} 円")
        print(f"   - 発生している差額: {row['Line_Variance']:,} 円")
        print(f"   → アクション: インポート元の文字コード、または桁数の丸め処理（切り捨てバグ）を調査してください。")
    print("--------------------------------------------------------------------------------")
else:
    print("✅ すべての行の金額が完全に一致しています。個別デバッグは不要です。")

In [ ]:
import pandas as pd

# 1. 旧システムから出力した最終残高データ
legacy_balance_data = {
    'Account': ['11110', '21110', '51120', '11120'],
    'Legacy_Balance': [1500000, -450000, 120000, 85000]  # 貸方はマイナス表記
}
df_legacy_bal = pd.DataFrame(legacy_balance_data)

# 2. 新システム（D365）にインポートされた開始残高データ
# ※ '21110'（買掛金）で移行漏れによる金額ズレ、'11120' がデータごと抜け落ちている想定
new_system_data = {
    'Account': ['11110', '21110', '51120'],
    'New_Balance': [1500000, -400000, 120000]
}
df_new_bal = pd.DataFrame(new_system_data)

# 3. 勘定科目（Account）をキーにして両方のデータを外部結合（Outer Merge）
# how='outer' を指定することで、どちらか片方にしか存在しない科目も漏らさず結合します
df_reconcile = pd.merge(df_legacy_bal, df_new_bal, on='Account', how='outer')

# 4. 欠損値（データが存在しない箇所）を 0 に置換
df_reconcile = df_reconcile.fillna(0)

# 5. 差額の計算（新システム残高 - 旧システム残高）
df_reconcile['Variance'] = df_reconcile['New_Balance'] - df_legacy_bal['Legacy_Balance']

# 6. 差額が 0 ではない（不整合がある）行だけを抽出
df_discrepancies = df_reconcile[df_reconcile['Variance'] != 0].copy()

print("--- [Execution Result 1] All Reconciliation Data ---")
display(df_reconcile)

print("\n--- [Execution Result 2] Balance Discrepancy Detection ---")
if not df_discrepancies.empty:
    print(f"🚨 警告: 新旧システム間で残高が一致しない科目が {len(df_discrepancies)} 件検出されました。")
    print("総勘定元帳の不一致は決算書の歪みに直結するため、データ移行を承認できません。")
    display(df_discrepancies[['Account', 'Legacy_Balance', 'New_Balance', 'Variance']])
else:
    print("✅ すべての勘定科目の開始残高が1円の狂いもなく完全に一致しました。")

In [ ]:
import pandas as pd

# 1. 旧システムから出力した最終残高データ
legacy_balance_data = {
    'Account': ['11110', '21110', '51120', '11120'],
    'Legacy_Balance': [1500000, -450000, 120000, 85000]
}
df_legacy_bal = pd.DataFrame(legacy_balance_data)

# 2. 新システム（D365）にインポートされた開始残高データ
new_system_data = {
    'Account': ['11110', '21110', '51120'],
    'New_Balance': [1500000, -400000, 120000]
}
df_new_bal = pd.DataFrame(new_system_data)

# 3. 勘定科目（Account）をキーにして外部結合
df_reconcile = pd.merge(df_legacy_bal, df_new_bal, on='Account', how='outer')

# 4. 欠損値を 0 に置換
df_reconcile = df_reconcile.fillna(0)

# 5. 差額の計算（修正：必ず結合後の同じデータフレーム内の列同士で計算します）
df_reconcile['Variance'] = df_reconcile['New_Balance'] - df_reconcile['Legacy_Balance']

# 6. 差額が 0 ではない行を抽出
df_discrepancies = df_reconcile[df_reconcile['Variance'] != 0].copy()

print("--- [Execution Result 1] All Reconciliation Data (Fixed) ---")
display(df_reconcile)

print("\n--- [Execution Result 2] Balance Discrepancy Detection ---")
if not df_discrepancies.empty:
    print(f"🚨 警告: 新旧システム間で残高が一致しない科目が {len(df_discrepancies)} 件検出されました。")
    print("総勘定元帳の不一致は決算書の歪みに直結するため、データ移行を承認できません。")
    display(df_discrepancies[['Account', 'Legacy_Balance', 'New_Balance', 'Variance']])
else:
    print("✅ すべての勘定科目の開始残高が1円の狂いもなく完全に一致しました。")

In [ ]:
import pandas as pd

# データ準備（外部結合およびfillnaまで完了した状態を再現）
legacy_balance_data = {'Account': ['11110', '21110', '51120', '11120'], 'Legacy_Balance': [1500000, -450000, 120000, 85000]}
new_system_data = {'Account': ['11110', '21110', '51120'], 'New_Balance': [1500000, -400000, 120000]}
df_reconcile = pd.merge(pd.DataFrame(legacy_balance_data), pd.DataFrame(new_system_data), on='Account', how='outer').fillna(0)

# 【実務修正】「旧システム残高」から「新システム残高」を引き算し、人間が左から右に読んで納得できる差分（不整合）を計算
df_reconcile['Discrepancy'] = df_reconcile['Legacy_Balance'] - df_reconcile['New_Balance']

# 差分が 0 ではない行を抽出
df_report = df_reconcile[df_reconcile['Discrepancy'] != 0].copy()

# 重役が直感的に「何が起きているか」を理解できるように【移行状態】のステータスを自動生成
def determine_migration_status(row):
    if row['Discrepancy'] > 0:
        return f"❌ 新システム側で {abs(row['Discrepancy']):,} 円の【登録不足（移行漏れ）】"
    else:
        return f"❌ 新システム側で {abs(row['Discrepancy']):,} 円の【過大登録】"

df_report['Migration_Status'] = df_report.apply(determine_migration_status, axis=1)

print("=============================================================================")
print("             DATA MIGRATION BALANCE RECONCILIATION REPORT                    ")
print("=============================================================================")
display(df_report[['Account', 'Legacy_Balance', 'New_Balance', 'Migration_Status']])

In [ ]:
import pandas as pd

# 1. 旧システムから抽出した仕訳データ（模擬データ）
old_ledger_data = {
    'Voucher': ['V-101', 'V-102', 'V-103', 'V-104', 'V-105'],
    'Old_Account': ['101', '201', '101', '999', '301'],  # '999' は新システムに未定義の想定
    'Amount': [150000, 45000, 300000, 85000, 120000]
}
df_old = pd.DataFrame(old_ledger_data)

# 2. 新旧の勘定科目マッピング定義（対応表）
account_mapping = {
    '101': '11110 (Cash and Cash Equivalents)',
    '201': '21110 (Accounts Payable)',
    '301': '51120 (Travel and Expense)'
}

# 3. マッピング処理の実行
# map() 関数を使用し、旧コードに対応する新コードを適用します
df_old['New_Account'] = df_old['Old_Account'].map(account_mapping)

# 4. マッピング不整合（エラーデータ）の抽出
# 新科目が空欄（NaN：Not a Number）になっている行を特定します
df_errors = df_old[df_old['New_Account'].isna()].copy()

print("--- [Execution Result 1] All Processed Data ---")
display(df_old)

print("\n--- [Execution Result 2] Mapping Error Detection ---")
if not df_errors.empty:
    print(f"🚨 警告: 新システムへのマッピングが未定義の仕訳が {len(df_errors)} 件検出されました。")
    print("データ移行を中断し、マッピング定義（account_mapping）を修正する必要があります。")
    display(df_errors[['Voucher', 'Old_Account', 'Amount']])
else:
    print("✅ すべての勘定科目が正常に新コードへ変換されました。")

In [ ]:
import pandas as pd
from google.colab import files
from datetime import datetime

# 1. 抽出済みのエラーデータ（df_errors）に対して、システム登録用のコメント列を動的に追加
df_errors_export = df_errors.copy()
df_errors_export['Error_Type'] = 'Missing Account Mapping'
df_errors_export['Action_Required'] = 'Define new D365 account code for Old_Account'

# 2. 動的なファイル名の作成（実行時の日付を使用）
current_date = datetime.now().strftime('%Y%m%d')
filename = f"COA_Mapping_Errors_{current_date}.csv"

# 3. Excelでの文字化けを防止するBOM付き(utf-8-sig)でCSV出力
df_errors_export.to_csv(filename, index=False, encoding='utf-8-sig')

print("--- [Execution Result 3] CSV Export Report ---")
print(f"✅ エラーデータ {len(df_errors_export)} 件を抽出しました。")
print(f"✅ ファイル '{filename}' を作成しました。ダウンロードを開始します。")

# 4. ローカルPCへ自動ダウンロード
files.download(filename)

In [ ]:
import pandas as pd

# 1. 既存のマッピング辞書（account_mapping）をコピーし、不足していた '999' を追加して更新
updated_mapping = account_mapping.copy()
updated_mapping['999'] = '11120 (Petty Cash)'  # 不足していたコードの定義を追加

# 2. 更新したマッピングを元のデータ（df_old）に再度適用（Remapping）
df_old_reprocessed = df_old[['Voucher', 'Old_Account', 'Amount']].copy()
df_old_reprocessed['New_Account'] = df_old_reprocessed['Old_Account'].map(updated_mapping)

# 3. 再処理後のマッピング不整合（エラーデータ）の再確認
df_new_errors = df_old_reprocessed[df_old_reprocessed['New_Account'].isna()]

print("--- [Execution Result 4] Reprocessed All Data ---")
display(df_old_reprocessed)

print("\n--- [Execution Result 5] Final Mapping Integrity Check ---")
# エラーデータフレームが空（empty）であるかどうかで、処理の成功を自動判定
if df_new_errors.empty:
    print("✅ 検証成功: すべての勘定科目がエラーなく正常にマッピングされました。")
    print("不整合は 0 件です。データ移行プロセスを正常に進めることができます。")
else:
    print(f"❌ 検証失敗: まだ未定義のコードが {len(df_new_errors)} 件残っています。再確認が必要です。")

In [ ]:
import pandas as pd

# 1. ［最新の勘定科目マスタ表］（Excel等から読み込んだ状態を再現）
# ※実務通り、担当者が手書きせず、このテーブルを正（ソース）として処理します
master_coa_table = {
    'Old_Code': ['101', '201', '301', '999'],
    'New_Code_And_Name': [
        '11110 (Cash and Cash Equivalents)',
        '21110 (Accounts Payable)',
        '51120 (Travel and Expense)',
        '11120 (Petty Cash)'  # マスタ側に正式追加されたコード
    ]
}
df_master_coa = pd.DataFrame(master_coa_table)

# 2. マスタ表の2列から、Pythonのマッピング辞書を自動生成（自動紐付け）
# zip関数を使い、Old_Codeをキー、New_Code_And_Nameを値にした辞書をシステムが自動作成します
automated_mapping_dict = dict(zip(df_master_coa['Old_Code'], df_master_coa['New_Code_And_Name']))

# 3. 最初の仕訳データ（df_old）をクリアな状態にして、自動生成した辞書でリマッピングを実行
df_remapped_pipeline = df_old[['Voucher', 'Old_Account', 'Amount']].copy()
df_remapped_pipeline['New_Account'] = df_remapped_pipeline['Old_Account'].map(automated_mapping_dict)

# 4. 最終整合性チェック（エラーの自動判定）
df_pipeline_errors = df_remapped_pipeline[df_remapped_pipeline['New_Account'].isna()]

print("--- [Execution Result 6] Automated Reprocessing Pipeline ---")
display(df_remapped_pipeline)

print("\n--- [Execution Result 7] Pipeline Validation Result ---")
if df_pipeline_errors.empty:
    print("✅ 【パイプライン検証成功】外部マスタとの自動連携による再マッピングが完了しました。")
    print(f"不整合データは {len(df_pipeline_errors)} 件です。データ移行の承認フラグを TRUE に設定します。")
else:
    print(f"❌ 【パイプライン検証失敗】未定義のコードが {len(df_pipeline_errors)} 件検出されました。")

In [ ]:
import pandas as pd

# 1. これまでの監査結果（財務インパクトの金額）を集約
# ※過去3日間の学習で算出したリアルな数値をセットします
audit_summary_data = {
    'Audit_Category': ['Ledger (Duplicate Entries)', 'Procurement (3-Way Matching)', 'Inventory (Cost Variance)'],
    'Risk_Amount_JPY': [60000, 20000, 3000000],  # 6万円、2万円、300万円
    'Primary_Module': ['General Ledger', 'Procurement & Sourcing', 'Cost Accounting'],
    'Priority': ['🔴 High', '🟡 Medium', '🔥 Critical']  # 金額と重要度に応じた格付け
}
df_global_audit = pd.DataFrame(audit_summary_data)

# 2. リスク金額順にソート（並べ替え）
df_global_audit = df_global_audit.sort_values(by='Risk_Amount_JPY', ascending=False).reset_index(drop=True)

print("================================================================================")
print("             D365 F&O CROSS-MODULE AUDIT EXECUTIVE SUMMARY                      ")
print("================================================================================")
display(df_global_audit)

# 3. Functional Consultant としての提言
print("--------------------------------------------------------------------------------")
print("私はD365の標準機能の導入だけでなく、出力されたデータを活用して")
print("企業のリスクを多角的に定量化する『データ駆動型のコンサルティング』が可能です。")
print("実際に、私がPythonで実装しGitHubに公開しているツールでは、以下のリスクを自動検知します：\n")

for index, row in df_global_audit.iterrows():
    print(f" [{row['Priority']}] モジュール: {row['Primary_Module']}")
    print(f"   - 監査対象: {row['Audit_Category']}")
    print(f"   - 財務影響額: {row['Risk_Amount_JPY']:,} 円")

    if 'Critical' in row['Priority']:
        print("   → 【戦略的提言】 標準原価差異（300万円）が激甚です。直ちに『Cost Rollup』を実行すべきです。")
    elif 'High' in row['Priority']:
        print("   → 【運用的提言】 仕訳の二重投稿（6万円）を検知。決算を狂わせるため『赤黒処理』を指示します。")
    elif 'Medium' in row['Priority']:
        print("   → 【統制的提言】 3-Way Matching不整合（2万円）を発見。『Invoice Hold』による支出防衛を提案します。")
    print("-" * 80)

In [ ]:
import pandas as pd

# 1. D365 F&O からエクスポートした想定の在庫・原価データ
inventory_cost_data = {
    'ItemNumber': ['ITEM001', 'ITEM002', 'ITEM003', 'ITEM004', 'ITEM005'],
    'ItemName': ['Microchip A', 'Steel Frame', 'Power Supply', 'Copper Wire', 'LCD Panel'],
    'Standard_Cost': [5000, 12000, 3500, 8000, 15000],  # D365のマスター設定値
    'Actual_Cost': [5100, 15000, 3450, 7200, 15000]     # 実際の購入・製造実績値
}
df_inventory = pd.DataFrame(inventory_cost_data)

# 2. 原価差異（Variance）と 差異率（Variance %）の計算
df_inventory['Cost_Variance'] = df_inventory['Actual_Cost'] - df_inventory['Standard_Cost']
df_inventory['Variance_Rate_Pct'] = (df_inventory['Cost_Variance'] / df_inventory['Standard_Cost']) * 100

# 3. 監査上の「許容限界しきい値（下落・高騰ともに5%）」を設定
THRESHOLD_PCT = 5.0

# 差異率の絶対値が5%を超えている「異常品目」を抽出
df_cost_alert = df_inventory[df_inventory['Variance_Rate_Pct'].abs() > THRESHOLD_PCT].copy()

print("--- D365 F&O Inventory Standard Cost Audit Report ---")
display(df_inventory)

# 4. Functional Consultant としての想定出力
print("\n--- 🔍 AI Standard Cost Variance Detection ---")
if not df_cost_alert.empty:
    print(f"🚨 警告: 許容基準（{THRESHOLD_PCT}%）を超える原価差異が {len(df_cost_alert)} 件検知されました。")
    display(df_cost_alert[['ItemNumber', 'ItemName', 'Standard_Cost', 'Actual_Cost', 'Variance_Rate_Pct']])

    print("\n💬 【コンサルタントからの原価改善提言】")
    for _, row in df_cost_alert.iterrows():
        if row['Variance_Rate_Pct'] > 0:
            print(f"💡 【要対応】 {row['ItemNumber']} ({row['ItemName']}): 原価が {row['Variance_Rate_Pct']:.1f}% 大幅に高騰しています。")
            print("   → アクション: P/L（損益計算書）の売上原価が圧迫されています。D365上で標準原価の再計算（Cost Rollup）を実行し、販売価格への転嫁または調達ルートの見直しを提案してください。")
        else:
            print(f"💡 【要対応】 {row['ItemNumber']} ({row['ItemName']}): 原価が {row['Variance_Rate_Pct']:.1f}% 大幅に下落しています。")
            print("   → アクション: 棚卸資産が過大評価されている可能性があります。D365の購買アグリーメント（Purchase Agreement）の見直し、または標準原価マスタの引き下げ改定を推奨します。")
else:
    print("✅ すべての品目の原価差異は許容範囲内（5%未満）であり、マスタは健全です。")

In [ ]:
import pandas as pd

# 1. 監査対象となった異常品目データ（df_cost_alert）を使用
# 実務に即して、各品目の「当期購買数量（Volume）」を一律 1,000 個と仮定して計算します
VOLUME = 1000
df_cost_alert['Financial_Impact'] = df_cost_alert['Cost_Variance'] * VOLUME

# 2. 高騰（損失）と下落（過大評価）の切り分け集計
# プラスの差異＝実際原価の方が高い（企業にとっては損）
total_cost_loss = df_cost_alert[df_cost_alert['Financial_Impact'] > 0]['Financial_Impact'].sum()

# マイナスの差異＝実際原価の方が安い（資産が過大評価されるリスク）
total_valuation_distortion = df_cost_alert[df_cost_alert['Financial_Impact'] < 0]['Financial_Impact'].sum()

print("--- ⚖️ D365 F&O Cost Variance Financial Impact Report ---")
print(f"想定取引数量: 各品目 {VOLUME:,} 個\n")

if total_cost_loss > 0:
    print(f"🚨 【売上原価の圧迫リスク】原価高騰による当期利益の押し下げ影響額: {total_cost_loss:,} 円")
if total_valuation_distortion != 0:
    print(f"⚠️ 【資産評価の歪みリスク】原価下落による棚卸資産の過大評価影響額: {abs(total_valuation_distortion):,} 円")

# 3. Functional Consultant としての経営層向けプレゼン用集約
print("\n--- 📝 Functional Consultant Executive Briefing ---")
print("--------------------------------------------------------------------------------")
print(f"📢 ITEM002 (Steel Frame) の高騰により、売上原価（COGS）が {total_cost_loss:,}円 増加し、今期利益を直撃します。")
print(f"   直ちにD365の『Cost Rollup（コスト再計算）』を実行し、標準原価の改定を財務へ上申してください。")
print(f"📢 ITEM004 (Copper Wire) の下落により、貸借対照表（B/S）上の棚卸資産が {abs(total_valuation_distortion):,}円 過大評価されるリスクがあります。")
print(f"   決算時の低価法（LCM）評価換えに備え、マスタ価格の引き下げを推奨します。")
print("--------------------------------------------------------------------------------")

In [ ]:
import pandas as pd

# 1. D365 F&O からエクスポートした想定の購買・受入・請求データ
purchase_matching_data = {
    'PO_Number': ['PO001', 'PO002', 'PO003', 'PO004'],
    'Vendor': ['Vendor_A', 'Vendor_B', 'Vendor_C', 'Vendor_D'],
    'PO_Amount': [500000, 1200000, 350000, 800000],        # 注文書の金額
    'Receipt_Amount': [500000, 1200000, 350000, 600000],   # 入庫（製品受領書）の金額
    'Invoice_Amount': [500000, 1220000, 350000, 600000]    # ベンダー請求書の金額
}
df_matching = pd.DataFrame(purchase_matching_data)

# 2. 3-Way Matching の検証ロジック
# 注文、受入、請求のすべての金額が一致しているかチェックするフラグを作成
df_matching['Is_Matched'] = (df_matching['PO_Amount'] == df_matching['Receipt_Amount']) & \
                             (df_matching['Receipt_Amount'] == df_matching['Invoice_Amount'])

# 3. 不整合（Mismatch）データのみを抽出
df_mismatch = df_matching[df_matching['Is_Matched'] == False].copy()

# 4. 請求書と注文書の「差額（過大請求額）」を計算
df_mismatch['Discrepancy'] = df_mismatch['Invoice_Amount'] - df_mismatch['PO_Amount']

print("--- D365 F&O Procurement 3-Way Matching Audit Report ---")
display(df_matching)

# 5. 3-Wayマッチングの想定結果出力
print("\n--- 🔍 AI 3-Way Matching Discrepancy Detection ---")
if not df_mismatch.empty:
    print(f"🚨 警告: 3-Way Matching（購買・受入・請求不整合）エラーが {len(df_mismatch)} 件検知されました。")
    display(df_mismatch[['PO_Number', 'Vendor', 'PO_Amount', 'Invoice_Amount', 'Discrepancy']])

    print("\n💬 【コンサルタントからの実務改善提言】")
    for _, row in df_mismatch.iterrows():
        if row['Discrepancy'] > 0:
            print(f"💡 【要調査】 {row['PO_Number']} ({row['Vendor']}): ベンダーからの過大請求が {row['Discrepancy']:,} 円発生しています。")
            print("   → アクション: 買掛金（A/P）の支払保留（Invoice Hold）を設定し、購買担当者経由でベンダーへクレジットノート（赤伝）の発行を要求してください。")
        else:
            print(f"💡 【要調査】 {row['PO_Number']} ({row['Vendor']}): 請求額が発注額を下回っています（一部未納、または値引きの可能性）。")
            print("   → アクション: 製品受領書（Receipt）のステータスを確認し、分割納品（Partial Delivery）の処理が正しくD365上で行われているか検証してください。")
else:
    print("✅ すべての購買トランザクションは3-Way Matchingを満たしており、正常です。")

In [ ]:
import pandas as pd

# 1. 不整合データ（df_mismatch）から過大請求（Discrepancy > 0）のみを抽出
df_overcharge = df_mismatch[df_mismatch['Discrepancy'] > 0].copy()

# 2. 過払いリスク総額の計算
total_overcharge_risk = df_overcharge['Discrepancy'].sum()

print("--- 💵 D365 F&O Procurement Overcharge Risk Report ---")

if total_overcharge_risk > 0:
    print(f"🚨 【重大な支出リスク】ベンダーからの過大請求によるリスク総額: {total_overcharge_risk:,} 円")
    print("\n【対象となる発注・請求明細】")
    display(df_overcharge[['PO_Number', 'Vendor', 'PO_Amount', 'Invoice_Amount', 'Discrepancy']])

    # 3. Functional Consultant としてのCFO向け進言
    print("\n💬 【コンサルタントからのガバナンス提言】")
    print(f"上記の発注に対してD365上で『Invoice Hold（支払保留）』を即座に実行し、")
    print(f"余分な支出 {total_overcharge_risk:,}円 を未然に防止する統制（Internal Control）を推奨します。")
else:
    print("✅ 現在、ベンダーからの過大請求リスクは検知されていません。")

In [ ]:
import pandas as pd

# 1. D365 F&O からエクスポートした想定の固定資産データ
# ※今期の償却率は一律 20% (0.20) とします
fixed_assets_data = {
    'AssetID': ['FA001', 'FA002', 'FA003', 'FA004', 'FA005'],
    'AssetName': ['Server-Tokyo', 'Delivery Truck', 'Office Laptop', 'HQ Projector', 'CNC Machine'],
    'AcquisitionCost': [1000000, 3000000, 200000, 150000, 5000000],
    'D365_Depreciation': [200000, 500000, 40000, 30000, 1000000]
}
df_fa = pd.DataFrame(fixed_assets_data)

# 2. AIによる理論上の正しい償却額（期待値）の計算
depreciation_rate = 0.20
df_fa['Expected_Depreciation'] = df_fa['AcquisitionCost'] * depreciation_rate

# 3. D365の実績値と理論値の「差分（ギャップ）」を計算
df_fa['Variance'] = df_fa['D365_Depreciation'] - df_fa['Expected_Depreciation']

# 4. 不整合（差分が0ではないもの）を抽出
df_mismatch = df_fa[df_fa['Variance'] != 0].copy()

print("--- D365 F&O Fixed Assets Depreciation Audit Report ---")
display(df_fa)

# 5. Functional Consultant としてのアドバイス出力
print("\n--- 🔍 AI Depreciation Variance Detection ---")
if not df_mismatch.empty:
    print(f"🚨 警告: 減価償却計算に不整合がある固定資産が {len(df_mismatch)} 件検知されました。")
    display(df_mismatch[['AssetID', 'AssetName', 'D365_Depreciation', 'Expected_Depreciation', 'Variance']])

    print("\n💬 【コンサルタントからの提言】")
    for _, row in df_mismatch.iterrows():
        print(f"💡 【要確認】 {row['AssetID']} ({row['AssetName']}): 差分 {row['Variance']:,} 円")
        if row['Variance'] < 0:
            print("   → 原因可能性: D365の償却開始日（Depreciation start date）の設定が遅れている、または減価償却プロファイルの設定ミスが疑われます。")
        else:
            print("   → 原因可能性: 過去の修正償却が二重に適用されている、または手動仕訳での過大計上の可能性があります。")
else:
    print("✅ すべての固定資産の減価償却費は、マスター通り正確に計算されています。")

In [ ]:
import pandas as pd

# 1. 差分の総額（財務インパクト）を計算
# 前回の df_fa の 'Variance' 列の合計を算出します
total_variance = df_fa['Variance'].sum()

# 監査上、修正を必須とする「重要性のしきい値（Materiality Threshold）」を設定
MATERIALITY_THRESHOLD = 50000

print("--- ⚖️ D365 F&O Depreciation Financial Impact Assessment ---")
print(f"総費用ギャップ（D365実績 - 理論値）: {total_variance:,.0f} 円")

# 2. しきい値判定と修正仕訳の自動生成
# 差分の絶対値（abs）がしきい値を超えているか判定します
if abs(total_variance) >= MATERIALITY_THRESHOLD:
    print(f"🚨 【決算修正が必要】差額が重要性のしきい値（{MATERIALITY_THRESHOLD:,}円）を超えています。")
    print("損益計算書（P/L）の修正が必要です。")

    print("\n📝 【D365 F&O 推奨調整仕訳（Proposed Journal Entry）】")
    print("--------------------------------------------------")
    if total_variance < 0:
        # 費用が足りない（過少計上）ので、追加で費用を計上する仕訳
        adjustment_amount = abs(total_variance)
        print(f" [借方 (Dr.)] 減価償却費 (Depreciation Expense)   : {adjustment_amount:,.0f} 円")
        print(f" [貸方 (Cr.)] 減価償却累計額 (Accumulated Dep.)  : {adjustment_amount:,.0f} 円")
        print(f"  ※摘要 (Text): 監査による FA002 減価償却不足分の追加計上")
    else:
        # 費用が多すぎる（過大計上）ので、費用を取り消す仕訳
        adjustment_amount = total_variance
        print(f" [借方 (Dr.)] 減価償却累計額 (Accumulated Dep.)  : {adjustment_amount:,.0f} 円")
        print(f" [貸方 (Cr.)] 減価償却費 (Depreciation Expense)   : {adjustment_amount:,.0f} 円")
        print(f"  ※摘要 (Text): 監査による 減価償却過大計上分の戻し入れ修正")
    print("--------------------------------------------------")
else:
    print(f"✅ 【修正不要】差額は許容範囲内（{MATERIALITY_THRESHOLD:,}円未満）です。")
    print("決算書への影響は軽微なため、当期の調整仕訳は不要です。来期のマスター修正のみ推奨します。")

In [ ]:
import pandas as pd

# 1. 昨日のエイジングデータ（df_ar）をベースとした拡張データ
# 分かりやすいように、今回は1つのセルで完結する形で定義します
ar_extended_data = {
    'InvoiceNo': ['INV001', 'INV002', 'INV003', 'INV004', 'INV005'],
    'Customer': ['Customer_A', 'Customer_B', 'Customer_C', 'Customer_D', 'Customer_E'],
    'Amount': [500000, 1200000, 350000, 800000, 1500000],
    'Days_Overdue': [0, 34, 4, 88, 14],
    'Risk_Level': ['✅ Low Risk', '🟡 Medium Risk', '✅ Low Risk', '🔴 High Risk', '✅ Low Risk']
}
df_collection = pd.DataFrame(ar_extended_data)

# 2. 回収優先度（Collection Priority）の動的判定
# 延滞日数が60日以上、または残高が100万円以上かつ延滞しているものを最優先とする
def determine_call_tier(row):
    if 'High' in row['Risk_Level'] or (row['Amount'] >= 1000000 and row['Days_Overdue'] > 0):
        return "🔥 Tier 1 (Immediate Call)"
    elif 'Medium' in row['Risk_Level']:
        return "⏳ Tier 2 (Follow-up Reminder)"
    else:
        return "🕊️ Tier 3 (Standard Monitoring)"

df_collection['Call_Tier'] = df_collection.apply(determine_call_tier, axis=1)

# 3. 担当者向けの「架電指示（Action Script）」の自動生成
def generate_script(tier):
    if "Tier 1" in tier:
        return "電話にて直接連絡。上長へのエスカレーションと出荷停止の警告を通知。"
    elif "Tier 2" in tier:
        return "督促メールを再送。入金予定日のコミットメントを求める。"
    else:
        return "定期入金確認。次回の請求サイクルまで通常監視。"

df_collection['Action_Instruction'] = df_collection['Call_Tier'].apply(generate_script)

# 4. 優先度の高い順にリストをソート（並べ替え）
df_call_list = df_collection.sort_values(by='Days_Overdue', ascending=False).reset_index(drop=True)

print("--- D365 F&O Prioritized Collection Action List ---")
display(df_call_list[['Call_Tier', 'Customer', 'Amount', 'Days_Overdue', 'Action_Instruction']])

In [ ]:
import pandas as pd
from google.colab import files

# 1. 出力用データの準備（前回の df_call_list を使用）
df_export = df_call_list.copy()

# 2. ファイル名に「今日の日付」を自動で組み込む（前回の型に関する学びの応用）
# 動的に今日の日付を取得し、文字列フォーマットにします
from datetime import date
today_str = date.today().strftime("%Y%m%d")
filename = f"D365_Urgent_Collection_Action_List_{today_str}.csv"

# 3. CSVファイルへの書き出し
# 日本語環境のExcelで開いても文字化けしないように 'utf-8-sig' を指定します
df_export.to_csv(filename, index=False, encoding='utf-8-sig')

print(f"--- 📄 Collection Action List Export Report ---")
print(f"✅ 全 {len(df_export)} 件の顧客タスクを優先度順に並び替えました。")
print(f"✅ ファイル '{filename}' を正常に作成しました。ダウンロードを開始します。")

# 4. ローカルPCへ自動ダウンロード
files.download(filename)

In [ ]:
import pandas as pd
from datetime import datetime

# 1. D365 F&O から取得した想定の売掛金データ
ar_data = {
    'InvoiceNo': ['INV001', 'INV002', 'INV003', 'INV004', 'INV005'],
    'Customer': ['Customer_A', 'Customer_B', 'Customer_C', 'Customer_D', 'Customer_E'],
    'Amount': [500000, 1200000, 350000, 800000, 1500000],
    'DueDate': ['2026-05-20', '2026-04-10', '2026-05-10', '2026-02-15', '2026-04-30']
}
df_ar = pd.DataFrame(ar_data)
df_ar['DueDate'] = pd.to_datetime(df_ar['DueDate'])

# 2. 実行日（本日）を基準に延滞日数を計算
# 前回の学びを活かし、本日（2026-05-14）を動的に設定して引き算します
current_date = pd.to_datetime('2026-05-14')

df_ar['Days_Overdue'] = (current_date - df_ar['DueDate']).dt.days

# 支払期日前（マイナス値）の場合は延滞日数を0にする処理
df_ar['Days_Overdue'] = df_ar['Days_Overdue'].apply(lambda x: x if x > 0 else 0)

# 3. エイジング・リスク判定ロジック
def judge_ar_risk(days):
    if days >= 61:
        return "🔴 High Risk (Severe Overdue)"
    elif days >= 31:
        return "🟡 Medium Risk (Warning)"
    else:
        return "✅ Low Risk (Active/Recent)"

df_ar['Risk_Level'] = df_ar['Days_Overdue'].apply(judge_ar_risk)

print(f"--- D365 F&O Accounts Receivable Aging Report (As of {current_date.date()}) ---")
display(df_ar)

# 4. Functional Consultant としての回収リスク総額の要約
high_risk_total = df_ar[df_ar['Risk_Level'].str.contains('High')]['Amount'].sum()
medium_risk_total = df_ar[df_ar['Risk_Level'].str.contains('Medium')]['Amount'].sum()

print("\n--- 📊 Financial Risk Summary ---")
if high_risk_total > 0 or medium_risk_total > 0:
    print(f"🚨 警告: 深刻な回収リスク（61日以上延滞）にさらされている残高総額: {high_risk_total:,} 円")
    print(f"⚠️ 注意: 要注意（31〜60日延滞）の残高総額: {medium_risk_total:,} 円")
    print("\n💬 【コンサルタントからの提言】")
    print("1. High Risk 顧客に対しては、与信管理（Credit Limit）の一時凍結および回収督促を開始してください。")
    print("2. 決算に備え、High Risk 残高に対する貸倒引当金（Bad Debt Allowance）の計上準備を推奨します。")
else:
    print("✅ すべての売掛金は健全な回収サイクルを維持しています。")

In [ ]:
import pandas as pd

# 1. 売掛金総額（Total AR）を計算
total_ar_amount = df_ar['Amount'].sum()

# 2. リスクレベルごとに残高（Amount）と件数（InvoiceNo）を集計
ar_summary = df_ar.groupby('Risk_Level').agg({
    'Amount': 'sum',
    'InvoiceNo': 'count'
}).reset_index()

# 列名の整理
ar_summary.columns = ['Risk_Level', 'Total_Amount', 'Invoice_Count']

# 3. 構成比率（Share %）の計算
ar_summary['Share (%)'] = (ar_summary['Total_Amount'] / total_ar_amount) * 100

# 4. レポートの並べ替え（リスクの高い順に並べる）
# 🔴 High -> 🟡 Medium -> ✅ Low の順になるようソートを調整
ar_summary['Sort_Order'] = ar_summary['Risk_Level'].apply(lambda x: 1 if 'High' in x else (2 if 'Medium' in x else 3))
ar_summary = ar_summary.sort_values(by='Sort_Order').drop(columns=['Sort_Order']).reset_index(drop=True)

print(f"--- 📊 D365 F&O Accounts Receivable Share Report (Total AR: {total_ar_amount:,} JPY) ---")

# 金額と比率の表示フォーマットを整えて出力
df_formatted = ar_summary.copy()
df_formatted['Total_Amount'] = df_formatted['Total_Amount'].apply(lambda x: f"{x:,}")
df_formatted['Share (%)'] = df_formatted['Share (%)'].apply(lambda x: f"{x:.1f}%")
display(df_formatted)

# 5. Functional Consultant としての最終評価
high_risk_share = ar_summary.loc[ar_summary['Risk_Level'].str.contains('High'), 'Share (%)'].values[0]
if high_risk_share > 15:
    print(f"\n📢 【財務健全性アラート】最重要リスク債権の割合が {high_risk_share:.1f}% に達しています。")
    print("   → キャッシュフロー悪化を防ぐため、次期の販売計画における『前受金（Advance Payment）』比率の引き上げを提案します。")

In [ ]:
import pandas as pd

# 1. D365 F&O からエクスポートした想定の仕訳データ
journal_entries = {
    'Voucher': ['V001', 'V002', 'V003', 'V004', 'V005'],
    'Date': ['2026-05-13', '2026-05-13', '2026-05-13', '2026-05-13', '2026-05-13'],
    'Account': ['Travel', 'Entertainment', 'Travel', 'Software', 'Entertainment'],
    'Amount': [15000, 45000, 15000, 250000, 45000],
    'CreatedBy': ['UserA', 'UserB', 'UserA', 'UserC', 'UserB']
}
df_ledger = pd.DataFrame(journal_entries)

# 2. 二重投稿（重複データ）の検出ロジック
# 'Date', 'Account', 'Amount', 'CreatedBy' がすべて一致するものを探します
# keep=False を指定することで、重複している行をすべて抽出します
duplicate_flags = df_ledger.duplicated(subset=['Date', 'Account', 'Amount', 'CreatedBy'], keep=False)
df_duplicates = df_ledger[duplicate_flags].copy()

print("--- D365 F&O General Ledger Audit Report ---")
display(df_ledger)

# 3. 監査結果の判定と出力
print("\n--- 🔍 AI Duplicate Entry Detection ---")
if not df_duplicates.empty:
    print(f"🚨 警告: 二重投稿の疑いがある仕訳が {len(df_duplicates)} 件（{len(df_duplicates)//2} セット）検知されました。")
    display(df_duplicates.sort_values(by=['Amount', 'CreatedBy']))

    # 4. Functional Consultant としてのアドバイス
    for actor in df_duplicates['CreatedBy'].unique():
        print(f"\n💡 【調査指示】 {actor} が投稿した重複仕訳について、以下の確認を行ってください：")
        print(f"   1. ベンダーからの請求書が二重に処理されていないか（誤謬チェック）")
        print(f"   2. システムのインターフェース処理が二重に走っていないか（技術チェック）")
else:
    print("✅ 全ての仕訳は重複なく、正常に処理されています。")

In [ ]:
import pandas as pd

# 1. 「余分な重複（2回目以降）」だけをピンポイントで抽出する
# keep='first' を指定すると、最初の1件を「正常」とみなし、2回目以降の重複だけを True にします
excess_duplicate_flags = df_ledger.duplicated(subset=['Date', 'Account', 'Amount', 'CreatedBy'], keep='first')
df_excess = df_ledger[excess_duplicate_flags].copy()

# 2. 余分に計上されてしまっている金額の合計を算出
total_financial_impact = df_excess['Amount'].sum()

print("--- 💸 D365 F&O Financial Impact Assessment ---")
if not df_excess.empty:
    print(f"🚨 【重大な財務影響】重複によって過大計上されているリスク総額: {total_financial_impact:,} 円")
    print("\n【内訳（削除・修正すべき余分な伝票リスト）】")
    display(df_excess[['Voucher', 'Account', 'Amount', 'CreatedBy']])

    # 3. Functional Consultant としての経営層向け進言
    print("\n💬 【コンサルタントからの提言】")
    print(f"上記の伝票({', '.join(df_excess['Voucher'])})を逆仕訳（赤黒処理）することで、")
    print(f"決算書における経費の過大計上リスク {total_financial_impact:,}円 を即座に解消できます。")
else:
    print("✅ 財務に影響を与える重複計上はありません。")

In [ ]:
import pandas as pd
import numpy as np

# 1. 予算データと実績推移（D365からエクスポートした想定）
# 4月〜8月までの5ヶ月分の支出データ
budget_data = {
    'Dept': ['Sales', 'IT', 'Marketing'],
    'Annual_Budget': [1000000, 2000000, 1500000],
    'Month_1': [80000, 150000, 100000],
    'Month_2': [85000, 200000, 110000],
    'Month_3': [78000, 250000, 120000],
    'Month_4': [82000, 280000, 115000],
    'Month_5': [90000, 300000, 130000]
}
df_budget = pd.DataFrame(budget_data)

# 2. 将来予測ロジック（現在の平均増加ペースから12ヶ月目の着地を予測）
months_passed = 5
months_remaining = 12 - months_passed

# 過去5ヶ月の合計支出
df_budget['Actual_To_Date'] = df_budget[['Month_1', 'Month_2', 'Month_3', 'Month_4', 'Month_5']].sum(axis=1)

# 直近の月次平均支出
df_budget['Monthly_Avg'] = df_budget['Actual_To_Date'] / months_passed

# 年度末(12ヶ月)の予測着地額 (現在までの実績 + 残り期間の予測)
df_budget['Projected_Year_End'] = df_budget['Actual_To_Date'] + (df_budget['Monthly_Avg'] * months_remaining)

# 3. 予算消化率と超過判定
df_budget['Burn_Rate_Pct'] = (df_budget['Projected_Year_End'] / df_budget['Annual_Budget']) * 100

print("--- D365 Budget Forecast Report (Month 5) ---")
display(df_budget[['Dept', 'Annual_Budget', 'Projected_Year_End', 'Burn_Rate_Pct']])

# 4. コンサルタントとしてのAIアラート
for index, row in df_budget.iterrows():
    if row['Burn_Rate_Pct'] > 100:
        print(f"\n🚨 【Alert】{row['Dept']} 部門で予算超過の恐れがあります (予測: {row['Burn_Rate_Pct']:.1f}%)")
        print(f"   💡 推奨案: 固定費の見直し、または予備費からの振替を検討してください。")
    elif row['Burn_Rate_Pct'] > 90:
        print(f"\n⚠️ 【Warning】{row['Dept']} 部門の予算消化が早まっています (予測: {row['Burn_Rate_Pct']:.1f}%)")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. データの準備（前回の df_budget を使用）
labels = df_budget['Dept']
budget_values = df_budget['Annual_Budget']
projected_values = df_budget['Projected_Year_End']

x = np.arange(len(labels))  # ラベルの位置
width = 0.35  # 棒の幅

# 2. グラフの描画
fig, ax = plt.subplots(figsize=(10, 6))
rects1 = ax.bar(x - width/2, budget_values, width, label='Annual Budget', color='lightgrey')
rects2 = ax.bar(x + width/2, projected_values, width, label='Projected Year-End', color='salmon')

# 3. タイトル、ラベル、凡例の設定
ax.set_title('D365 Budget vs Projected Year-End Breakdown', fontsize=14, fontweight='bold')
ax.set_ylabel('Amount (JPY)')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()

# 4. 数値ラベルを棒の上に表示（自動計算機能）
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height/1000:,.0f}K',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),  # 3ポイント上にオフセット
                    textcoords="offset points",
                    ha='center', va='bottom')

autolabel(rects1)
autolabel(rects2)

# 5. 超過している部分を強調する補助線
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

print("--- コンサルタントの視点 ---")
print("IT部門の赤い棒がグレーの予算枠を大きく突き抜けています。")
print("この視覚的エビデンスを元に、次期予算策定または現状のコスト削減案を提示します。")

In [ ]:
import pandas as pd

# 1. 経費精算ログ
expense_data = {
    'User': ['UserA', 'UserB', 'UserC', 'UserD'],
    'Category': ['Travel', 'Entertainment', 'Hardware', 'Entertainment'],
    'Amount': [50000, 150000, 300000, 5000],
    'Note': ['Osaka trip', 'Client dinner', 'Laptop purchase', 'Lunch with team'],
    'Status': ['On-Trip', 'In-Office', 'In-Office', 'In-Office']
}
df_expense = pd.DataFrame(expense_data)

# 2. AIによる監査ロジック（簡易推論）
def ai_audit(row):
    # ロジック1: 出張中ではないのに高額な交際費がある場合
    if row['Status'] == 'In-Office' and row['Category'] == 'Entertainment' and row['Amount'] > 100000:
        return "🔴 High Risk (Unusual Entertainment Expense)"
    # ロジック2: ハードウェア購入額が基準（20万）を超えている場合
    elif row['Category'] == 'Hardware' and row['Amount'] > 200000:
        return "🟡 Warning (High Value Asset Purchase)"
    # ロジック3: それ以外（正常）
    else:
        return "✅ Low (Within Policy)"

# 3. 解析の実行
df_expense['Audit_Result'] = df_expense.apply(ai_audit, axis=1)

print("--- 🤖 AI-Powered Expense Audit Report ---")
display(df_expense)

# 4. 異常値の要約（昨日のトリアージ機能を応用）
alerts = df_expense[df_expense['Audit_Result'].str.contains('High|Warning')]
if not alerts.empty:
    print(f"\n🚩 【監査アラート】精査が必要な経費が {len(alerts)} 件あります。承認者へ通知します。")

In [ ]:
import pandas as pd

# 1. 部署マスタデータの追加（模擬）
user_dept_map = {
    'User': ['UserA', 'UserB', 'UserC', 'UserD'],
    'Dept': ['Sales', 'Sales', 'IT', 'Sales']
}
df_dept = pd.DataFrame(user_dept_map)

# 2. 監査結果と部署データをマージ
df_audit_dept = pd.merge(df_expense, df_dept, on='User')

# 3. 「リスクあり」と判定された行のみを抽出して集計
# リスクがあるものだけを対象に、部署ごとの合計金額を算出します
risk_analysis = df_audit_dept[df_audit_dept['Audit_Result'].str.contains('High|Warning')]

# 部署ごとのリスク総額と件数を集計
dept_risk_summary = risk_analysis.groupby('Dept').agg({
    'Amount': 'sum',
    'User': 'count'
}).reset_index()

dept_risk_summary.columns = ['Department', 'Total_Risk_Amount', 'Incident_Count']

print("--- 🏢 Departmental Risk Exposure Report ---")
display(dept_risk_summary.sort_values(by='Total_Risk_Amount', ascending=False))

# 4. 管理者（Asia Finance Lead）へのアドバイス
for index, row in dept_risk_summary.iterrows():
    if row['Total_Risk_Amount'] >= 100000:
        print(f"📢 【要監査】 {row['Department']} 部門の未承認リスク額が {row['Total_Risk_Amount']:,}円 に達しています。")
        print(f"   → 当該部門のマネージャーへ、経費ガイドラインの再徹底を指示してください。")

In [ ]:
import pandas as pd

# 1. 昨日のバッチ処理結果（df_batch）から、的中しなかった（Is_Hit == 0）案件を抽出
# 昨日の結果変数 df_batch がある前提で進めます
unresolved_cases = df_batch[df_batch['Is_Hit'] == 0].copy()

# 2. AIによるナレッジ下書き作成ロジック
def create_knowledge_draft(inquiry):
    # 本来は生成AI(OpenAIなど)が文章を作りますが、ここではその「加工ロジック」を再現します
    return f"【未登録案件】 {inquiry} に関する調査。現在、解決策を募集中。"

# 3. 未解決案件に対して「ナレッジ候補」の列を追加
unresolved_cases['Knowledge_Draft'] = unresolved_cases['Inquiry'].apply(create_knowledge_draft)

print("--- 📝 AI Knowledge Update: New Drafts Created ---")
if not unresolved_cases.empty:
    print(f"✅ 新しくナレッジベースに追加すべき案件が {len(unresolved_cases)} 件見つかりました。")
    display(unresolved_cases[['Inquiry', 'Knowledge_Draft']])

    # 4. これを「新規ナレッジ登録用リスト」としてCSV出力する準備
    # (昨日の「75%」以外を拾い上げることで、組織の知恵を100%に近づけます)
else:
    print("✅ 全ての案件が既存のナレッジでカバーされています。")

In [ ]:
import pandas as pd

# 1. 以前作成した「AIトリアージ(緊急度判定)関数」を再利用
def ai_triage_simple(text):
    if "VPN" in text or "error" in text:
        return "🔴 High"
    elif "slow" in text:
        return "🟡 Medium"
    else:
        return "🟢 Low"

# 2. 未解決案件（unresolved_cases）に対して緊急度を付与
unresolved_cases = unresolved_cases.copy()
unresolved_cases['Priority'] = unresolved_cases['Inquiry'].apply(ai_triage_simple)

# 3. チームリーダー向けの「ナレッジ作成依頼リスト」として整理
# 優先度の高い順に並べ替えます
knowledge_priority_list = unresolved_cases.sort_values(by='Priority')

print("--- 👨‍💼 Team Leader Insight: Knowledge Creation Priority ---")
if not knowledge_priority_list.empty:
    print("🚩 以下の未登録案件について、優先度に基づいた解決策の作成をお願いします。")
    display(knowledge_priority_list[['Priority', 'Inquiry', 'Knowledge_Draft']])

    # 4. リーダーへのアドバイス
    for _, row in knowledge_priority_list.iterrows():
        if "High" in row['Priority']:
            print(f"🔥 【特急】 '{row['Inquiry']}' は未登録ですがリスクが高いです。即時調査を！")
else:
    print("✅ 全ての未登録案件は処理済み、または存在しません。")

In [ ]:
import pandas as pd

# 1. 現在の的中率（hit_rate）を評価
# ※ 前回のステップで算出した 75.0% を使用します
# シミュレーションとして、もし SLA_THRESHOLD が 80.0% だった場合のアラートも試せるようにします
current_hit_rate = hit_rate  # 実際の結果（75.0）
# SLA_THRESHOLD = 70.0        # 目標基準（70.0%）
SLA_THRESHOLD = 80.0        # 目標基準（80.0%）

print(f"--- 📈 AI Performance Monitoring (SLA: {SLA_THRESHOLD}%) ---")
print(f"Current Accuracy: {current_hit_rate:.1f}%")

# 2. 自動判定とアラート文の生成
if current_hit_rate < SLA_THRESHOLD:
    print(f"\n🚨 【ALERT】AIナレッジ的中率が基準（{SLA_THRESHOLD}%）を下回りました！")
    print("--------------------------------------------------")
    print("【分析結果】")
    print("最近、新しいタイプの問い合わせが増えており、既存の知恵では対応しきれていません。")
    print("\n【推奨アクション】")
    print("1. 過去1週間の『未解決案件（Is_Hit == 0）』を抽出し、一斉にナレッジ化してください。")
    print("2. 頻出キーワードをAIのナレッジベース（knowledge_base）に再登録してください。")
    print("--------------------------------------------------")
else:
    print(f"\n✅ 良好: AIのパフォーマンスは基準（{SLA_THRESHOLD}%）を維持しています。")
    print("現在の運用を継続しつつ、定期的なナレッジの微調整を行ってください。")

In [ ]:
import pandas as pd

# 1. 過去の解決事例（ナレッジベース）
knowledge_base = [
    {"issue": "VPN connection failed 0x8007", "solution": "証明書の更新が必要です。"},
    {"issue": "D365 Login error", "solution": "ブラウザのキャッシュをクリアしてください。"},
    {"issue": "Intune Policy Conflict", "solution": "重複するプロファイルの割り当てを解除してください。"}
]

# 2. 本日届いた新しい問い合わせ
new_inquiry = "I cannot connect to VPN, getting error 0x8007"

# 3. AIによる類似性検索ロジック（簡易版）
def find_best_solution(inquiry, kb):
    # お問い合わせ内容の単語が含まれているかチェック
    inquiry = inquiry.lower()
    for entry in kb:
        # 課題(issue)の中にキーワードが含まれているか
        if any(word in inquiry for word in entry["issue"].lower().split()):
            return entry["solution"]
    return "該当する過去事例が見つかりません。新規調査が必要です。"

# 4. 検索の実行
suggested_fix = find_best_solution(new_inquiry, knowledge_base)

print(f"--- 🤖 AI Knowledge Search Result ---")
print(f"📩 新規問い合わせ: {new_inquiry}")
print(f"💡 AIが提案する解決策: {suggested_fix}")

# 5. 集計レポート（昨日のようにDataFrame化）
report = pd.DataFrame([{"Inquiry": new_inquiry, "Suggested_Fix": suggested_fix}])
display(report)

In [ ]:
import pandas as pd

# 1. 複数の新規問い合わせリスト
new_inquiries = [
    "VPN is failing with 0x8007",        # 的中するはず
    "D365 login is very slow today",     # 的中するはず
    "How to change my profile picture?", # 過去事例にない（新規調査）
    "Conflict in Intune policy"          # 的中するはず
]

# 2. 一括解析の実行
results = []
for inquiry in new_inquiries:
    # 前回の find_best_solution 関数を使用
    suggested_fix = find_best_solution(inquiry, knowledge_base)

    # 「的中」の判定ロジック
    # 「新規調査が必要」という定型文以外が返ってきたら「的中(Hit)」とみなす
    is_hit = 1 if "新規調査が必要" not in suggested_fix else 0

    results.append({
        "Inquiry": inquiry,
        "Suggested_Fix": suggested_fix,
        "Is_Hit": is_hit
    })

df_batch = pd.DataFrame(results)

# 3. 的中率（Accuracy Rate）の算出
hit_rate = (df_batch['Is_Hit'].sum() / len(df_batch)) * 100

print("--- 🤖 AI Knowledge Matcher: Batch Analysis Report ---")
display(df_batch)

print(f"\n📊 AIナレッジ的中率: {hit_rate:.1f}%")

# 4. アドバイザーとしての考察
if hit_rate >= 70:
    print("✅ 良好: 大半の問い合わせが既存のナレッジで対応可能です。")
else:
    print("⚠️ 課題: 未知の問い合わせが多いです。ナレッジベースの拡充が必要です。")

In [ ]:
import pandas as pd

# 1. 届いた問い合わせ（チケット）のリスト
customer_messages = [
    "システムにログインできず、業務が完全に止まっています！至急連絡をください！",
    "新機能の使い方について教えてほしいです。急ぎではありません。",
    "昨日設定を変更してから、一部の端末でエラーが出ることがあります。調査をお願いします。",
    "間違えて契約を更新してしまいました。キャンセルしたいです。"
]

# 2. AIの判定ロジック（簡易自然言語解析）
def ai_triage(text):
    # 緊急キーワードのチェック
    if "止まって" in text or "至急" in text or "ログインできず" in text:
        return "🔴 High (Critical Issue)", "Angry/Upset"
    # 一般的な調査
    elif "エラー" in text or "調査" in text:
        return "🟡 Medium (Technical Inquiry)", "Neutral"
    # 一般的な質問
    else:
        return "🟢 Low (General Question)", "Happy/Calm"

# 3. データの解析
results = []
for msg in customer_messages:
    priority, sentiment = ai_triage(msg)
    results.append({
        "Customer_Message": msg,
        "Priority": priority,
        "Sentiment": sentiment
    })

df_triage = pd.DataFrame(results)

print("--- 🤖 AI Customer Support Triage Report ---")
display(df_triage)

# 4. エスカレーションが必要な「深刻な件」を特定
critical_cases = df_triage[df_triage['Priority'].str.contains("High")]
if not critical_cases.empty:
    print(f"\n🔥 【緊急対応】直ちに対応が必要なクリティカル案件が {len(critical_cases)} 件あります。")

In [ ]:
import pandas as pd
from google.colab import files

# 1. データの整理と並べ替え
# 優先度の高い順（High -> Medium -> Low）に並べ替えます
# ※今回はカテゴリ名に絵文字が入っているので、ロジカルに並べ替えるために
#   一時的な「ソート用ランク」を作ります
priority_map = {
    "🔴 High (Critical Issue)": 1,
    "🟡 Medium (Technical Inquiry)": 2,
    "🟢 Low (General Question)": 3
}
df_triage['Sort_Rank'] = df_triage['Priority'].map(priority_map)

# ランク順（1が最高）にソート
df_final_list = df_triage.sort_values(by='Sort_Rank').copy()

# 2. 調査・対応用の「アクション指示」をAIが付与
def suggest_action(priority):
    if "High" in priority:
        return "IMMEDIATE CALL: Secure a primary engineer and contact customer."
    elif "Medium" in priority:
        return "INVESTIGATION: Replicate the issue in the test environment."
    else:
        return "REPLY: Provide documentation and closing guidance."

df_final_list['Suggested_Action'] = df_final_list['Priority'].apply(suggest_action)

# 3. 不要なソート用ランク列を削除して、CSV出力
df_export = df_final_list.drop(columns=['Sort_Rank'])
filename = f"Daily_Support_Priority_List_{pd.Timestamp.now().strftime('%Y%m%d')}.csv"
df_export.to_csv(filename, index=False, encoding='utf-8-sig')

print(f"--- Support Priority List Generation ---")
print(f"✅ 全 {len(df_export)} 件のチケットを優先度順に整理しました。")
display(df_export)

# 4. ダウンロード
files.download(filename)

In [ ]:
import pandas as pd

# 1. 解析対象の「難解なエラーログ」
raw_logs = [
    "Error 0x80070005: Access denied while writing to C:\\Windows\\Temp.",
    "Timeout waiting for response from Microsoft Entra ID. Latency > 15s.",
    "Policy conflict detected: Setting 'PasswordLength' defined in 2 separate profiles."
]

# 2. AIのナレッジベース（学習済みデータ）
ai_knowledge = {
    "0x80070005": "権限不足です。インストーラーを『管理者として実行』するか、フォルダの所有権を確認してください。",
    "Timeout": "ネットワーク遅延、またはサービス障害の可能性があります。サービス正常性ダッシュボードを確認してください。",
    "conflict": "複数のポリシーが競合しています。重複するプロファイルの割り当てを解除してください。"
}

# 3. AIによる解析エンジンのシミュレーション
def ai_logic(log):
    # AIがログの意味を「理解」し、キーワードに基づいて解決策を導き出す
    for key, advice in ai_knowledge.items():
        if key.lower() in log.lower():
            return advice
    return "未知のエラーです。シニアエンジニアへのエスカレーションを推奨します。"

# 4. ログ解析の実行
analysis_results = []
for log in raw_logs:
    solution = ai_logic(log)
    analysis_results.append({"Raw_Log": log, "AI_Suggested_Solution": solution})

df_ai_support = pd.DataFrame(analysis_results)

print("--- 🤖 AI-Powered Support Assistant: Insight Report ---")
for i, row in df_ai_support.iterrows():
    print(f"\n[Log {i+1}]: {row['Raw_Log']}")
    print(f"💡 AIの診断: {row['AI_Suggested_Solution']}")

In [ ]:
# 1. 前回の解析結果（df_ai_support）を使用

print("--- 📧 AI-Generated Draft Responses for Customers ---")

for index, row in df_ai_support.iterrows():
    # AIが診断結果を元に、メールのドラフトを作成する
    email_draft = f"""
【Draft Case ID: #20260507-{index+1}】
Dear Customer,

Thank you for contacting Microsoft Support.
Regarding the log you provided: "{row['Raw_Log']}"

[Our Diagnosis]
{row['AI_Suggested_Solution']}

We recommend performing the above steps to resolve the issue.
If you need further assistance, please let us know.

Best Regards,
Microsoft Support Advisor AI
--------------------------------------------------
    """
    print(email_draft)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. 各カテゴリのサマリーデータ（これまでの学習結果を集約）
# A. OSコンプライアンス（正常/古い）
os_summary = {'Status': ['Compliant', 'Outdated'], 'Count': [85, 15]}
df_os = pd.DataFrame(os_summary)

# B. ネットワーク品質（拠点ごとの平均遅延）
net_summary = {'Location': ['Tokyo', 'Osaka', 'London'], 'Avg_Latency': [2.1, 12.5, 0.8]}
df_net = pd.DataFrame(net_summary)

# C. ユーザーリスク（昨日のスコアリング結果）
risk_summary = {'Category': ['High Risk', 'Warning', 'Safe'], 'User_Count': [1, 5, 94]}
df_risk = pd.DataFrame(risk_summary)

# 2. ダッシュボードの描画（3つのグラフを1画面に配置）
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

# グラフ1: OS分布（円グラフ）
ax1.pie(df_os['Count'], labels=df_os['Status'], autopct='%1.1f%%', colors=['skyblue', 'salmon'])
ax1.set_title('OS Compliance Rate')

# グラフ2: ネットワーク遅延（棒グラフ）
ax2.bar(df_net['Location'], df_net['Avg_Latency'], color='plum')
ax2.axhline(y=5, color='red', linestyle='--', label='SLA')
ax2.set_title('Network Latency by Location (sec)')
ax2.legend()

# グラフ3: ユーザーリスク分布（円グラフ）
ax3.pie(df_risk['User_Count'], labels=df_risk['Category'], autopct='%1.1f%%', colors=['red', 'orange', 'lightgreen'])
ax3.set_title('User Risk Distribution')

plt.tight_layout()
plt.show()

# 3. Advisorとしての推奨アクション（Recommendations）自動生成
print("--- 📋 Microsoft Support Advisor: Executive Summary ---")
compliance_rate = df_os.loc[df_os['Status']=='Compliant', 'Count'].values[0]

if compliance_rate < 90:
    print(f"🚩 【Action】OS Compliance ({compliance_rate}%) が目標を下回っています。一斉アップデートを推奨します。")

slow_loc = df_net[df_net['Avg_Latency'] > 5]['Location'].tolist()
if slow_loc:
    print(f"🚩 【Action】{slow_loc} 拠点で遅延が深刻です。VPNゲートウェイの増設を検討してください。")

high_risk = df_risk.loc[df_risk['Category']=='High Risk', 'User_Count'].values[0]
if high_risk > 0:
    print(f"🚩 【Action】{high_risk} 名の「最重要調査対象」がいます。アカウント保護を強化してください。")

In [ ]:
import pandas as pd
from google.colab import files

# 1. 報告用サマリー文の作成（f-stringを活用）
report_date = pd.Timestamp.now().strftime('%Y-%m-%d')
compliance_val = df_os.loc[df_os['Status']=='Compliant', 'Count'].values[0]
slow_loc_list = df_net[df_net['Avg_Latency'] > 5]['Location'].tolist()
high_risk_num = df_risk.loc[df_risk['Category']=='High Risk', 'User_Count'].values[0]

report_content = f"""
=====================================================
    INTUNE ORGANIZATION HEALTH REPORT
=====================================================
Report Date: {report_date}
Status: ACTION REQUIRED

[1. EXECUTIVE SUMMARY]
Overall OS Compliance is {compliance_val}%.
Infrastructure latency detected in specific regions.
Immediate attention required for high-risk user accounts.

[2. KEY PERFORMANCE INDICATORS]
- OS Compliance: {compliance_val}% (Target: 90%)
- Regional Latency: {df_net.set_index('Location')['Avg_Latency'].to_dict()}
- Security Risk: {high_risk_num} Critical User(s) detected

[3. RECOMMENDED ACTIONS]
1. OS UPDATE: Initiate forced updates for outdated devices.
2. NETWORK: Investigate VPN Gateway capacity in {slow_loc_list}.
3. SECURITY: Reset credentials and enable MFA for High-Risk users.

-----------------------------------------------------
Generated by: Microsoft Support Advisor Toolkit (Python)
=====================================================
"""

# 2. テキストファイルとして保存
file_name = f"Intune_Health_Report_{report_date}.txt"
with open(file_name, "w", encoding="utf-8") as f:
    f.write(report_content)

print(f"--- 最終報告書 生成完了 ---")
print(report_content)

# 3. ファイルのダウンロード
files.download(file_name)

In [ ]:
import pandas as pd

# 1. 統合されたサインインログ（模擬データ）
log_data = {
    'User': ['UserA', 'UserB', 'UserA', 'UserC', 'UserB', 'UserB', 'UserD'],
    'Result': ['Success', 'Blocked', 'Success', 'Success', 'Blocked', 'Blocked', 'Success'],
    'OS_Status': ['New', 'Old', 'New', 'New', 'Old', 'Old', 'New'],
    'Hour': [10, 23, 11, 14, 23, 0, 15] # 23時や0時は深夜アクセス
}
df_logs = pd.DataFrame(log_data)

# 2. ユーザーごとのリスク要素を集計
# A. ブロック回数
user_risk = df_logs[df_logs['Result'] == 'Blocked'].groupby('User').size().reset_index(name='Block_Points')

# B. 古いOSの使用回数
old_os_risk = df_logs[df_logs['OS_Status'] == 'Old'].groupby('User').size().reset_index(name='OS_Points')

# C. 深夜アクセスの回数
night_risk = df_logs[df_logs['Hour'].isin([23, 0, 1, 2, 3, 4, 5])].groupby('User').size().reset_index(name='Night_Points')

# 3. 全てのデータをマージ（統合）して合計スコアを算出
# 全ユーザーをベースにするため、外部結合(outer)を使います
df_score = pd.merge(user_risk, old_os_risk, on='User', how='outer')
df_score = pd.merge(df_score, night_risk, on='User', how='outer').fillna(0)

# 合計リスクスコアの計算
df_score['Total_Risk_Score'] = df_score['Block_Points'] + df_score['OS_Points'] + df_score['Night_Points']

# 4. スコア順に並べ替え
df_score = df_score.sort_values(by='Total_Risk_Score', ascending=False)

print("--- User Risk Analysis Report ---")
display(df_score)

# 5. エスカレーション判定
high_risk_user = df_score.iloc[0]
if high_risk_user['Total_Risk_Score'] >= 5:
    print(f"\n🔥 最優先調査対象: {high_risk_user['User']} (Score: {high_risk_user['Total_Risk_Score']})")
    print("🚩 理由: 複数のセキュリティリスクが重なっています。アカウントのロックまたは直接の指導を推奨します。")

In [ ]:
import pandas as pd
from google.colab import files

# 1. 閾値（スコア5点）を超えるユーザーを抽出
high_risk_list = df_score[df_score['Total_Risk_Score'] >= 5].copy()

# 2. 調査用のアクション（指示）を追加
# 「なぜ危ないか」のサマリーを動的に作成
high_risk_list['Investigation_Priority'] = 'Critical'
high_risk_list['Action_Required'] = 'Verify Identity and Request OS Update'

# 3. CSVファイルへの書き出し
filename = f"High_Risk_User_Audit_{pd.Timestamp.now().strftime('%Y%m%d')}.csv"
high_risk_list.to_csv(filename, index=False, encoding='utf-8-sig')

print(f"--- High-Risk Alert Report ---")
if not high_risk_list.empty:
    print(f"✅ 優先調査が必要なユーザーが {len(high_risk_list)} 名特定されました。")
    display(high_risk_list)

    # 4. ダウンロード
    files.download(filename)
else:
    print("✅ 現在、緊急の調査が必要な高リスクユーザーはいません。")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. 24時間のサインインログ（模擬データ）
# 10時台に成功が多く、23時台に不審なブロックが多い想定
data = {
    'Timestamp': [
        '2026-05-04 09:10', '2026-05-04 09:45', '2026-05-04 10:05', '2026-05-04 10:20',
        '2026-05-04 10:30', '2026-05-04 13:00', '2026-05-04 15:00', '2026-05-04 23:10',
        '2026-05-04 23:20', '2026-05-04 23:45'
    ],
    'Result': ['Success', 'Success', 'Success', 'Success', 'Success', 'Success', 'Success', 'Blocked', 'Blocked', 'Blocked']
}
df_trend = pd.DataFrame(data)
df_trend['Timestamp'] = pd.to_datetime(df_trend['Timestamp'])

# 2. 1時間ごとに集計
# Success と Blocked ごとに件数をカウントします
df_trend.set_index('Timestamp', inplace=True)
hourly_summary = df_trend.groupby([pd.Grouper(freq='h'), 'Result']).size().unstack(fill_value=0)

# 3. 折れ線グラフによる可視化
plt.figure(figsize=(12, 6))

# Success の推移
plt.plot(hourly_summary.index, hourly_summary.get('Success', 0), marker='o', label='Success', color='skyblue', linewidth=2)
# Blocked の推移
plt.plot(hourly_summary.index, hourly_summary.get('Blocked', 0), marker='x', label='Blocked', color='tomato', linewidth=2, linestyle='--')

plt.title('Sign-in Trend Analysis (Last 24 Hours)', fontsize=14, fontweight='bold')
plt.xlabel('Time')
plt.ylabel('Number of Attempts')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()

# X軸の時間表示を整える
plt.gcf().autofmt_xdate()

plt.show()

# 4. 異常の自動判定
print("--- 傾向分析レポート ---")
if 'Blocked' in hourly_summary.columns and hourly_summary['Blocked'].max() >= 3:
    peak_time = hourly_summary['Blocked'].idxmax()
    print(f"🚨 【注意】 {peak_time} 頃にブロックが集中しています。不正アクセスの試行がないか確認してください。")
else:
    print("✅ 特筆すべきブロックの集中は見られません。")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. 複合ログデータ（時刻と理由を組み合わせる）
complex_data = {
    'Timestamp': [
        '2026-05-04 09:00', '2026-05-04 09:15', '2026-05-04 10:00',
        '2026-05-04 23:05', '2026-05-04 23:30', '2026-05-04 23:45'
    ],
    'Reason': [
        'OS Outdated', 'OS Outdated', 'Device Not Managed',
        'Location Not Allowed', 'Location Not Allowed', 'Location Not Allowed'
    ]
}
df_complex = pd.DataFrame(complex_data)
df_complex['Timestamp'] = pd.to_datetime(df_complex['Timestamp'])

# 2. 時間帯（Hour）を抽出
df_complex['Hour'] = df_complex['Timestamp'].dt.hour

# 3. ピボットテーブルで「時間帯」×「理由」のクロス集計
# 行を「時間帯」、列を「理由」にして件数を数えます
cross_summary = df_complex.pivot_table(
    index='Hour',
    columns='Reason',
    aggfunc='size',
    fill_value=0
)

# 4. ヒートマップによる可視化（直感的にどこが濃いか判別）
plt.figure(figsize=(10, 5))
sns.heatmap(cross_summary, annot=True, cmap='YlOrRd', cbar_kws={'label': 'Count'})

plt.title('Intune Security Audit: Block Reasons by Hour', fontsize=14, fontweight='bold')
plt.xlabel('Block Reason')
plt.ylabel('Hour of Day (24h)')
plt.show()

print("--- 複合分析インサイト ---")
# 23時台の「Location Not Allowed」を自動特定
if 23 in cross_summary.index and 'Location Not Allowed' in cross_summary.columns:
    if cross_summary.loc[23, 'Location Not Allowed'] >= 2:
        print("🚩 警告：深夜23時台に『許可されていない地域』からのアクセスが集中しています。")
        print("   → これは単なるユーザーの操作ミスではなく、海外からの不正アクセスの試行である可能性が極めて高いです。")

In [ ]:
import pandas as pd

# 1. ログイン試行ログ
access_logs = {
    'User': ['UserA', 'UserB', 'UserC', 'UserD', 'UserE'],
    'Device_Status': ['Compliant', 'Non-Compliant', 'Compliant', 'Unregistered', 'Compliant'],
    'Location': ['Japan', 'Japan', 'USA', 'Japan', 'Brazil'],
    'Result': ['Success', 'Blocked', 'Success', 'Blocked', 'Blocked'],
    'Reason': [None, 'OS Outdated', None, 'Device Not Managed', 'Location Not Allowed']
}
df_access = pd.DataFrame(access_logs)

# 2. ブロックされた（Blocked）ログに絞る
blocked_analysis = df_access[df_access['Result'] == 'Blocked'].copy()

print("--- Conditional Access: Sign-in Analysis ---")
# 全体の結果を表示
display(df_access['Result'].value_counts())

if not blocked_analysis.empty:
    print(f"\n🚨 【Security Alert】合計 {len(blocked_analysis)} 件のアクセスブロックを検知しました。")

    # 3. トラブルシューティング案内（正しい列名 'Reason' を使用）
    print("\n--- Troubleshooting Guidance for Support Advisors ---")
    for index, row in blocked_analysis.iterrows():
        # ここを 'Reason' に修正しました
        reason = row['Reason']
        print(f"👤 {row['User']}: 原因は [{reason}] です。")

        if reason == 'Device Not Managed':
            print("   → 対処法: 端末を Intune にポータルサイトアプリから登録（Enroll）するよう案内してください。")
        elif reason == 'OS Outdated':
            print("   → 対処法: OSを最新版にアップデートした後に再試行するよう案内してください。")
        elif reason == 'Location Not Allowed':
            print("   → 対処法: 許可されていない地域からのアクセスです。会社の接続ポリシーを確認してください。")

In [ ]:
import matplotlib.pyplot as plt

# 1. ブロック理由ごとの集計（前回作成した blocked_analysis を使用）
# value_counts() で理由ごとの件数を数えます
reason_summary = blocked_analysis['Reason'].value_counts()

# 2. 円グラフの作成
plt.figure(figsize=(8, 8))

# カラーマップの設定（セキュリティ警告を意識した配色）
colors = ['#ff9999', '#ffcc99', '#99ff99'] # 赤・オレンジ・緑系の淡い色

# グラフの描画
plt.pie(
    reason_summary,
    labels=reason_summary.index,
    autopct='%1.1f%%',
    startangle=140,
    colors=colors,
    explode=[0.05] * len(reason_summary) # 全ての要素を少し浮かせて見やすくする
)

# 3. タイトルと体裁の設定
plt.title('Analysis of Conditional Access Block Reasons', fontsize=15, fontweight='bold')
plt.axis('equal') # 綺麗な正円にする

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
from google.colab import files
from datetime import date

# 1. ログデータに日付情報を追加（シミュレーション用）
# 全てのログが「今日」発生したと仮定します
#df_access['Date'] = '2026-05-03'
today = date.today()
formatted_date = today.strftime("%Y-%m-%d")
df_access['Date'] = formatted_date

# 2. 「今日」かつ「Blocked」のデータのみを抽出
#target_date = '2026-05-03'
target_date = df_access['Date']
today_blocks = df_access[
    (df_access['Date'] == target_date) &
    (df_access['Result'] == 'Blocked')
].copy()

# 3. 調査用レポートとして列を整理
# ユーザー名、場所、理由、日付の順に並べ替えます
report_data = today_blocks[['Date', 'User', 'Location', 'Reason']]

# 4. CSVファイルとして書き出し
filename = f"Security_Block_Report_{target_date}.csv"
report_data.to_csv(filename, index=False, encoding='utf-8-sig')

print(f"--- 当日セキュリティレポート作成 (Daily Security Export) ---")
if not report_data.empty:
    print(f"✅ 本日 ({target_date}) のブロック履歴を {len(report_data)} 件抽出しました。")
    print(f"✅ ファイル '{filename}' を作成しました。")
    display(report_data)

    # 5. ローカルPCへダウンロード
    files.download(filename)
else:
    print(f"✅ 本日 ({target_date}) はブロックされたアクセスはありません。")

In [ ]:
import pandas as pd

# 1. Intune 管理画面上のステータス（表向きのデータ）
intune_status = {
    'DeviceName': ['PC-01', 'PC-02', 'PC-03', 'PC-04', 'PC-05'],
    'App': ['Office365', 'Office365', 'Office365', 'Office365', 'Office365'],
    'Intune_Result': ['Success', 'Success', 'Success', 'Failed', 'Success']
}
df_intune = pd.DataFrame(intune_status)

# 2. デバイスからのテレメトリログ（実際の起動・存在確認データ）
# ※ Success と言いながら、実際には「No_Process（動いていない）」ものを探します
actual_telemetry = {
    'DeviceName': ['PC-01', 'PC-02', 'PC-03', 'PC-04', 'PC-05'],
    'App_Process': ['Active', 'Active', 'No_Process', 'No_Process', 'Active']
}
df_telemetry = pd.DataFrame(actual_telemetry)

# 3. 2つのデータを結合して「裏切り」を探す
df_check = pd.merge(df_intune, df_telemetry, on='DeviceName')

# 4. サイレント・フェイラーの判定ロジック
# 条件: Intuneの結果は Success なのに、実際のプロセスが No_Process
df_check['Silent_Failure'] = (df_check['Intune_Result'] == 'Success') & (df_check['App_Process'] == 'No_Process')

print("--- Intune Software Distribution Integrity Check ---")

# サイレント・フェイラーの抽出
silent_failures = df_check[df_check['Silent_Failure'] == True]

if not silent_failures.empty:
    print(f"🚨 【要注意】サイレント・フェイラーが {len(silent_failures)} 件検知されました。")
    print("表示上は成功していますが、実際にはアプリケーションが正常に配置されていません。")
    display(silent_failures[['DeviceName', 'App', 'Intune_Result', 'App_Process']])
else:
    print("✅ 全ての成功報告は、実際の稼働データと一致しています。")

# 5. 全体の配布成功率（実数ベース）
real_success_rate = (len(df_check[df_check['App_Process'] == 'Active']) / len(df_check)) * 100
print(f"\n📊 実際の有効インストール率: {real_success_rate:.1f}%")

In [ ]:
# 1. 詳細なインストーラーログ（Exit Code を含む）
# ※ 3010 は「再起動が必要だが成功」、0 は「成功」
# ※ 1603 は「致命的エラー」ですが、設定ミスで Success 扱いになっている想定
detailed_logs = {
    'DeviceName': ['PC-01', 'PC-02', 'PC-03', 'PC-04', 'PC-05'],
    'ExitCode': [0, 3010, 1603, 1603, 0],
    'Detailed_Message': ['Installed', 'Pending Reboot', 'Fatal error during installation', 'Fatal error', 'Installed']
}
df_exit_codes = pd.DataFrame(detailed_logs)

# 2. サイレント・フェイラーを起こした PC-03 の詳細を突き合わせる
analysis_result = pd.merge(silent_failures, df_exit_codes, on='DeviceName')

print("--- Root Cause Analysis: Silent Failure Details ---")
display(analysis_result[['DeviceName', 'Intune_Result', 'App_Process', 'ExitCode', 'Detailed_Message']])

# 3. エンジニアへのアドバイス
for _, row in analysis_result.iterrows():
    if row['ExitCode'] == 1603:
        print(f"\n💡 【判明】 {row['DeviceName']} の原因は MSI Error 1603 です。")
        print("   Intuneのアプリ登録設定で、1603 を『成功(Success)』として誤登録していないか確認してください。")

In [ ]:
import pandas as pd

# 1. Intune 監査ログ（模擬データ）
audit_log_data = {
    'Timestamp': [
        '2026-04-25 09:00', '2026-04-26 10:30', '2026-04-27 14:00',
        '2026-04-27 15:30', '2026-04-28 11:00', '2026-04-29 16:00'
    ],
    'Actor': ['Admin_A', 'Admin_B', 'Admin_A', 'Admin_C', 'Admin_B', 'Admin_A'],
    'Operation': ['Create', 'Patch', 'Patch', 'Delete', 'Patch', 'Patch'],
    'TargetPolicy': ['Wi-Fi-Config', 'Password-Policy', 'Password-Policy', 'Camera-Block', 'App-Deploy', 'Firewall-Config'],
    'Result': ['Success', 'Success', 'Success', 'Success', 'Success', 'Success']
}
df_audit = pd.DataFrame(audit_log_data)
df_audit['Timestamp'] = pd.to_datetime(df_audit['Timestamp'])

# 2. 「Patch（設定変更）」に絞った分析
# 設定変更はトラブルの原因になりやすいため、重点的に調査します
patch_history = df_audit[df_audit['Operation'] == 'Patch'].copy()

# 3. 管理者（Actor）ごとの操作頻度
# 誰がどの設定を触ったのかを可視化するための集集計
admin_activity = df_audit.groupby('Actor')['Operation'].count().reset_index(name='Total_Actions')

print("--- Intune Audit Log Analysis (Activity Overview) ---")
display(admin_activity)

print("\n--- Critical Changes (Patch Operations) History ---")
# 時系列順（新しい順）に並べ替えて表示
display(patch_history.sort_values(by='Timestamp', ascending=False))

# 4. 特定の日時以降の変更を特定
# 例：トラブルが始まったとされる 4/27 以降の変更
incident_start = pd.to_datetime('2026-04-27')
recent_changes = df_audit[df_audit['Timestamp'] >= incident_start]

if not recent_changes.empty:
    print(f"\n🚩 インシデント発生期間（{incident_start.date()}以降）の変更が {len(recent_changes)} 件見つかりました。")
    display(recent_changes)

In [ ]:
import pandas as pd

# 1. 監査ログデータの定義（ここを統合しました）
audit_log_data = {
    'Timestamp': [
        '2026-04-27 14:00', '2026-04-27 14:30', # ここで 30分間隔の競合を発生させます
        '2026-04-25 09:00', '2026-04-27 15:30', '2026-04-28 11:00', '2026-04-29 16:00'
    ],
    'Actor': ['Admin_A', 'Admin_B', 'Admin_A', 'Admin_C', 'Admin_B', 'Admin_A'],
    'Operation': ['Patch', 'Patch', 'Create', 'Delete', 'Patch', 'Patch'],
    'TargetPolicy': ['Password-Policy', 'Password-Policy', 'Wi-Fi-Config', 'Camera-Block', 'App-Deploy', 'Firewall-Config'],
    'Result': ['Success', 'Success', 'Success', 'Success', 'Success', 'Success']
}
df_audit = pd.DataFrame(audit_log_data)
df_audit['Timestamp'] = pd.to_datetime(df_audit['Timestamp'])

# 2. デバイスとポリシーの紐付けデータ
device_policy_map = {
    'DeviceName': ['PC-01', 'PC-02', 'PC-03', 'PC-04', 'PC-05', 'PC-06', 'PC-07'],
    'AssignedPolicy': ['Wi-Fi-Config', 'Password-Policy', 'Password-Policy', 'Password-Policy', 'Camera-Block', 'Password-Policy', 'Wi-Fi-Config']
}
df_map = pd.DataFrame(device_policy_map)

# --- ここから分析ロジック ---

# 3. 【ミッション1】影響範囲（Impact Analysis）
target_policy = 'Password-Policy'
affected_devices = df_map[df_map['AssignedPolicy'] == target_policy]

print(f"--- インシデント影響範囲分析 (Target: {target_policy}) ---")
print(f"🚩 影響を受けている可能性のあるデバイスは合計 {len(affected_devices)} 台です。")
display(affected_devices)

# 4. 【ミッション2】管理者の同時操作（競合）検知
# タイムスタンプ順に並べ替え
df_audit_sorted = df_audit.sort_values(by=['TargetPolicy', 'Timestamp'])

# 差分計算
df_audit_sorted['Time_Diff'] = df_audit_sorted.groupby('TargetPolicy')['Timestamp'].diff()
df_audit_sorted['Prev_Actor'] = df_audit_sorted.groupby('TargetPolicy')['Actor'].shift()

# 1時間(60分)以内に、別の人が同じポリシーを触った場合を抽出
conflict_risks = df_audit_sorted[
    (df_audit_sorted['Time_Diff'] < pd.Timedelta(hours=1)) &
    (df_audit_sorted['Actor'] != df_audit_sorted['Prev_Actor'])
].copy()

print("\n--- 管理者操作競合（Admin Conflict）検知 ---")
if not conflict_risks.empty:
    print(f"⚠️ 警告: 短時間に複数の管理者による重複操作が検知されました。設定が上書きされている可能性があります。")
    # 分単位で表示を見やすくします
    conflict_risks['Diff_Minutes'] = conflict_risks['Time_Diff'].dt.total_seconds() / 60
    display(conflict_risks[['Timestamp', 'TargetPolicy', 'Actor', 'Prev_Actor', 'Diff_Minutes']])
else:
    print("✅ 管理者間の操作競合は見つかりませんでした。")

In [ ]:
import pandas as pd
from google.colab import files

# 1. メンテナンス対象データの整理（影響デバイス 4台を使用）
# ※ 前回の affected_devices をコピーして加工します
maintenance_list = affected_devices.copy()

# 2. 作業者向けの付加情報の追加
maintenance_list['Investigation_Reason'] = f'Conflict detected for {target_policy}'
maintenance_list['Action_Required'] = 'Verify manual policy sync and check event logs'

# 3. CSVファイルへの書き出し
filename = f"Urgent_Maintenance_List_{target_policy}.csv"
maintenance_list.to_csv(filename, index=False, encoding='utf-8-sig')

print(f"--- 緊急メンテナンスリスト 生成レポート ---")
print(f"✅ 対象デバイス数: {len(maintenance_list)} 台")
print(f"✅ ファイル '{filename}' を正常に作成しました。")

# 4. ローカルPCへダウンロード
files.download(filename)

In [ ]:
# 1. 昨日の不整合データ（PC-04が4/27にエラーになったという情報）
# ※ 昨日の結果から変数として持ってきた想定です
incident_device = 'PC-04'
incident_date = pd.to_datetime('2026-04-27')

# 2. 監査ログから、インシデント発生日「当日」に行われた「設定変更(Patch)」を抽出
# これが「トラブルの引き金」になった可能性が高い変更です
suspect_changes = df_audit[
    (df_audit['Timestamp'].dt.date == incident_date.date()) &
    (df_audit['Operation'] == 'Patch')
]

print(f"--- インシデント調査：{incident_device} の原因分析 ---")

if not suspect_changes.empty:
    print(f"🔍 発見: {incident_date.date()} に以下の設定変更が行われていました。")
    print("これが不整合を引き起こした直接の原因である可能性があります。")
    display(suspect_changes)

    # 3. 調査用のアクションメッセージ生成
    for _, row in suspect_changes.iterrows():
        print(f"\n💡 【調査依頼】 {row['Actor']} が行った {row['TargetPolicy']} の変更内容を確認してください。")
else:
    print("✅ インシデント当日に該当する設定変更は見つかりませんでした。")

In [ ]:
# 1. 昨日の不整合データ（PC-04が4/27にエラーになったという情報）
# ※ 昨日の結果から変数として持ってきた想定です
incident_device = 'PC-04'
incident_date = pd.to_datetime('2026-04-27')

# 2. 監査ログから、インシデント発生日「当日」に行われた「設定変更(Patch)」を抽出
# これが「トラブルの引き金」になった可能性が高い変更です
suspect_changes = df_audit[
    (df_audit['Timestamp'].dt.date == incident_date.date()) &
    (df_audit['Operation'] == 'Patch')
]

print(f"--- インシデント調査：{incident_device} の原因分析 ---")

if not suspect_changes.empty:
    print(f"🔍 発見: {incident_date.date()} に以下の設定変更が行われていました。")
    print("これが不整合を引き起こした直接の原因である可能性があります。")
    display(suspect_changes)

    # 3. 調査用のアクションメッセージ生成
    for _, row in suspect_changes.iterrows():
        print(f"\n💡 【調査依頼】 {row['Actor']} が行った {row['TargetPolicy']} の変更内容を確認してください。")
else:
    print("✅ インシデント当日に該当する設定変更は見つかりませんでした。")

In [ ]:
import pandas as pd

# 1. ポリシー適用ログ（模擬データ）
policy_data = {
    'DeviceName': ['PC-01', 'PC-02', 'PC-03', 'PC-04', 'PC-05', 'PC-06', 'PC-07'],
    'SettingName': ['Password-Policy', 'Password-Policy', 'Wi-Fi-Config', 'Password-Policy', 'Camera-Block', 'Password-Policy', 'Wi-Fi-Config'],
    'Status': ['Success', 'Conflict', 'Success', 'Error', 'Success', 'Conflict', 'Success'],
    'LastUpdate': ['2026-04-28', '2026-04-29', '2026-04-29', '2026-04-27', '2026-04-29', '2026-04-29', '2026-04-28']
}
df_policy = pd.DataFrame(policy_data)

# 2. 状態別の集計
status_summary = df_policy['Status'].value_counts().reset_index()
status_summary.columns = ['Status', 'Count']

# 3. 特に深刻な「Conflict（競合）」と「Error（エラー）」を抽出
# Conflictは、複数のポリシーが同じ設定を上書きしようとしている「設定ミス」の可能性が高い
priority_issues = df_policy[df_policy['Status'].isin(['Conflict', 'Error'])]

print("--- Intune Policy Deployment Summary ---")
display(status_summary)

if not priority_issues.empty:
    print(f"\n🚨 【Critical】設定不整合（Conflict/Error）が {len(priority_issues)} 件あります。")
    # 設定名ごとにどれくらい起きているかを表示
    issue_by_setting = priority_issues.groupby('SettingName').size().reset_index(name='IssueCount')
    display(issue_by_setting)

    print("\n--- 調査対象端末リスト ---")
    display(priority_issues[['DeviceName', 'SettingName', 'Status']])
else:
    print("\n✅ すべてのポリシーが正常に配信されています。")

In [ ]:
import pandas as pd
from datetime import datetime

# 1. 現在日付の設定（実行環境の日付を使用）
# ※ データの LastUpdate と比較するため
current_date = pd.to_datetime('2026-04-29') # シナリオに合わせて現在を4/29と仮定

# 2. 不整合データ（Conflict/Error）に対して経過日数を計算
# 前回の priority_issues を使用します
priority_issues = priority_issues.copy()
priority_issues['LastUpdate'] = pd.to_datetime(priority_issues['LastUpdate'])
priority_issues['経過日数'] = (current_date - priority_issues['LastUpdate']).dt.days

# 3. 3日以上放置されている問題を抽出
stale_threshold = 3
stale_issues = priority_issues[priority_issues['経過日数'] >= stale_threshold]

print(f"--- Intune Stale Policy Issues Report (Threshold: {stale_threshold} days) ---")

if not stale_issues.empty:
    print(f"🚨 警告: {stale_threshold}日以上放置されている深刻な不整合が {len(stale_issues)} 件あります。")
    # 放置日数が多い順に表示
    display(stale_issues.sort_values(by='経過日数', ascending=False)[['DeviceName', 'SettingName', 'Status', 'LastUpdate', '経過日数']])
else:
    print(f"✅ 全ての不整合は直近 {stale_threshold}日以内に発生または更新されています。")

In [ ]:
import pandas as pd

# 1. 基準日の設定（4/27のデータが「3日以上前」になるよう、5/1に設定してみます）
current_date = pd.to_datetime('2026-05-01')

# 2. データの再確認と経過日数の計算
# ※ 前回の priority_issues を安全にコピーして計算します
analysis_df = priority_issues.copy()
analysis_df['LastUpdate'] = pd.to_datetime(analysis_df['LastUpdate'])
analysis_df['経過日数'] = (current_date - analysis_df['LastUpdate']).dt.days

# 3. 判定（あえて全件表示して、計算が正しいか確認します）
print(f"--- 診断モード：すべての不整合と経過日数 (基準日: {current_date.date()}) ---")
display(analysis_df[['DeviceName', 'SettingName', 'Status', 'LastUpdate', '経過日数']])

# 4. 長期放置（3日以上）の抽出
stale_threshold = 3
stale_issues = analysis_df[analysis_df['経過日数'] >= stale_threshold]

print(f"\n--- 最終レポート (Threshold: {stale_threshold} days) ---")
if not stale_issues.empty:
    print(f"🚨 警告: {stale_threshold}日以上放置されている深刻な不整合が {len(stale_issues)} 件あります。")
    display(stale_issues)
else:
    print(f"✅ 設定した閾値（{stale_threshold}日）を超える放置案件はありません。")

In [ ]:
import matplotlib.pyplot as plt

# 1. データの集計（元の df_policy を使用）
status_counts = df_policy['Status'].value_counts()

# 2. 配色の設定（ビジネスレポートの標準に合わせる）
# Success = Green, Conflict = Orange, Error = Red
color_map = {
    'Success': '#66b3ff', # Light Blue系
    'Conflict': '#ffcc99', # Orange系
    'Error': '#ff9999'    # Red系
}
colors = [color_map.get(x, '#808080') for x in status_counts.index]

# 3. 円グラフの作成
plt.figure(figsize=(8, 8))
plt.pie(
    status_counts,
    labels=status_counts.index,
    autopct='%1.1f%%',
    startangle=140,
    colors=colors,
    explode=[0.05 if x != 'Success' else 0 for x in status_counts.index] # 異常値を少し浮かせる
)

plt.title('Intune Policy Compliance Overview', fontsize=15, fontweight='bold')

# 4. コンプライアンス率の計算と表示
compliance_rate = (status_counts.get('Success', 0) / len(df_policy)) * 100
plt.annotate(f'Compliance Rate: {compliance_rate:.1f}%', xy=(0, 0), xytext=(-1.1, -1.1),
             fontsize=12, fontweight='bold', color='darkblue')

plt.axis('equal')
plt.show()

print(f"--- 最終コンプライアンス・サマリー ---")
print(f"✅ 全体の成功率: {compliance_rate:.1f}%")
print(f"⚠️ 調査が必要な不整合: {len(priority_issues)} 件")

In [ ]:
import pandas as pd

# 1. 接続要求ログ（Request Logs）
req_data = {
    'SessionID': ['S1', 'S2', 'S3', 'S4', 'S5'],
    'User': ['UserA', 'UserB', 'UserC', 'UserD', 'UserE'],
    'ReqTime': pd.to_datetime([
        '2026-04-20 09:00:01',
        '2026-04-20 09:05:10',
        '2026-04-20 09:10:00',
        '2026-04-20 09:15:30',
        '2026-04-20 09:20:00'
    ])
}
df_req = pd.DataFrame(req_data)

# 2. 接続結果ログ（Result Logs）
# ※ S3（UserC）の結果が届いていない（タイムアウト）想定
res_data = {
    'SessionID': ['S1', 'S2', 'S4', 'S5'],
    'Status': ['Success', 'Success', 'Failed', 'Success'],
    'ResTime': pd.to_datetime([
        '2026-04-20 09:00:03', # 2秒で成功
        '2026-04-20 09:05:12', # 2秒で成功
        '2026-04-20 09:15:45', # 15秒で失敗
        '2026-04-20 09:20:01'  # 1秒で成功
    ])
}
df_res = pd.DataFrame(res_data)

# 3. SessionIDをキーにして2つのログを結合（Left Join）
df_net = pd.merge(df_req, df_res, on='SessionID', how='left')

# 4. パフォーマンス計算
# 接続にかかった時間を秒単位で算出
df_net['Duration_Sec'] = (df_net['ResTime'] - df_net['ReqTime']).dt.total_seconds()

# 5. 異常検知
# A. タイムアウト（結果ログが存在しない）
timeouts = df_net[df_net['Status'].isnull()]

# B. 遅延（接続に10秒以上かかったもの）
slow_connections = df_net[df_net['Duration_Sec'] > 10]

print("--- Network Connectivity Analysis Report ---")

if not timeouts.empty:
    print(f"\n🚨 【Timeout】応答がないセッションが {len(timeouts)} 件あります:")
    display(timeouts[['SessionID', 'User', 'ReqTime']])

if not slow_connections.empty:
    print(f"\n⚠️ 【Latency】接続が遅いセッション（10秒超）が {len(slow_connections)} 件あります:")
    display(slow_connections[['SessionID', 'User', 'Duration_Sec', 'Status']])

# 6. 全体の統計
print("\n--- Connectivity Statistics ---")
print(f"成功率: {len(df_net[df_net['Status']=='Success']) / len(df_req) * 100:.1f}%")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. 接続先情報を含んだ拡張データ
extended_data = {
    'SessionID': ['S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8'],
    'Location': ['Tokyo', 'Osaka', 'Tokyo', 'Osaka', 'Tokyo', 'London', 'London', 'London'],
    'Status': ['Success', 'Success', 'Timeout', 'Failed', 'Success', 'Success', 'Success', 'Success'],
    'Duration_Sec': [2.1, 2.5, None, 15.0, 1.8, 0.5, 0.8, 0.6]
}
df_loc = pd.DataFrame(extended_data)

# 2. 拠点ごとの統計（平均時間と失敗件数）
# 失敗・タイムアウトを「異常」としてカウント
df_loc['Is_Error'] = df_loc['Status'].isin(['Timeout', 'Failed'])

loc_stats = df_loc.groupby('Location').agg({
    'Duration_Sec': 'mean',
    'Is_Error': 'sum',
    'SessionID': 'count'
}).reset_index()

loc_stats.columns = ['Location', 'Avg_Duration', 'Error_Count', 'Total_Sessions']

# 3. 失敗率（Error Rate）の計算
loc_stats['Error_Rate (%)'] = (loc_stats['Error_Count'] / loc_stats['Total_Sessions']) * 100

print("--- Location-Based Connectivity Audit ---")
display(loc_stats.sort_values(by='Error_Rate (%)', ascending=False))

# 4. 可視化：拠点別の平均接続時間
plt.figure(figsize=(10, 5))
plt.bar(loc_stats['Location'], loc_stats['Avg_Duration'], color='plum')
plt.axhline(y=5, color='red', linestyle='--', label='Warning Threshold (5s)')
plt.title('Average Connection Latency by Location')
plt.ylabel('Seconds')
plt.legend()
plt.show()

In [ ]:
import pandas as pd
from google.colab import files

# 1. 調査対象の基準（しきい値）設定
# 失敗率が30%を超える拠点を「緊急調査対象」とする
ERROR_RATE_THRESHOLD = 30.0

# 2. 異常拠点の抽出
urgent_investigation = loc_stats[loc_stats['Error_Rate (%)'] > ERROR_RATE_THRESHOLD].copy()

# 3. 担当者が分かりやすいように列名や補足情報を整理
urgent_investigation['Investigation_Priority'] = 'High'
urgent_investigation['Recommended_Action'] = 'Check VPN Gateway and local AP logs'

# 4. CSVファイルとして書き出し
# インフラチームがExcel等で開くことを想定し、utf-8-sig を使用
filename = "Urgent_Network_Investigation_Request.csv"
urgent_investigation.to_csv(filename, index=False, encoding='utf-8-sig')

print(f"--- ネットワーク調査依頼書 生成レポート ---")
if not urgent_investigation.empty:
    print(f"✅ 緊急調査対象として {len(urgent_investigation)} 拠点を特定しました。")
    print(f"✅ ファイル '{filename}' を作成し、ダウンロードを開始します。")
    display(urgent_investigation)

    # 5. ローカルPCへダウンロード
    files.download(filename)
else:
    print("✅ 全拠点の品質が基準内です。調査依頼の必要はありません。")

In [ ]:
import pandas as pd

# 1. Intuneのアプリ配布ログ（模擬データ）
log_data = {
    'DeviceName': ['PC-01', 'PC-02', 'PC-03', 'PC-04', 'PC-05', 'PC-06'],
    'App': ['Office365', 'Office365', 'Teams', 'Office365', 'VPN_Client', 'Office365'],
    'ErrorCode': ['0x80070005', '0x80070643', '0x80070005', '0x80070005', '0x80040154', '0x80070005'],
    'Message': ['Access Denied', 'Update Error', 'Access Denied', 'Access Denied', 'Class not reg', 'Access Denied']
}
df_logs = pd.DataFrame(log_data)

# 2. エラーコードごとの集計
error_summary = df_logs['ErrorCode'].value_counts().reset_index()
error_summary.columns = ['ErrorCode', 'Count']

# 3. 最も頻発しているエラーの特定
top_error = error_summary.iloc[0]

print("--- Intune App Deployment Error Analysis ---")
display(error_summary)
print(f"\n🚩 最優先調査対象: エラーコード '{top_error['ErrorCode']}' が {top_error['Count']} 件発生しています。")
print("   → アクセス権限(Access Denied)の設定を見直す必要があります。")

In [ ]:
# 1. 前回の「High Risk」判定ロジックを活用
# ここでは「暗号化がオフ」のユーザーを抽出
target_users = [
    {'Name': 'UserB', 'Device': 'iPad-UserB', 'Issue': 'Encryption Disabled'},
    {'Name': 'UserF', 'Device': 'PC-UserF', 'Issue': 'Encryption Disabled'}
]

print("\n--- 自動生成されたユーザー通知（通知文ドラフト） ---")

for user in target_users:
    email_draft = f"""
To: {user['Name']}
Subject: 【重要】デバイスのセキュリティ設定を確認してください

{user['Name']} 様

Intune 統合管理システムにより、ご利用のデバイス({user['Device']})において
セキュリティポリシー違反が検知されました。

■検知された問題: {user['Issue']}
■必要なアクション: 設定画面より「デバイスの暗号化」を有効にしてください。

この設定が完了するまで、社内リソースへのアクセスが制限される場合があります。
ご協力をお願いいたします。

Microsoft Support Advisor (Automated Message)
--------------------------------------------------
    """
    print(email_draft)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. データの準備（エラー集計結果をソート）
error_summary = df_logs['ErrorCode'].value_counts().reset_index()
error_summary.columns = ['ErrorCode', 'Count']
error_summary = error_summary.sort_values(by='Count', ascending=False)

# 2. 累積比率の計算
error_summary['Cumulative_Percentage'] = error_summary['Count'].cumsum() / error_summary['Count'].sum() * 100

# 3. グラフの作成
fig, ax1 = plt.subplots(figsize=(10, 6))

# 棒グラフ（エラー件数）
ax1.bar(error_summary['ErrorCode'], error_summary['Count'], color='skyblue', label='Error Count')
ax1.set_xlabel('Error Code')
ax1.set_ylabel('Frequency (Count)')

# 折れ線グラフ（累積比率）を表示するための2軸目を作成
ax2 = ax1.twinx()
ax2.plot(error_summary['ErrorCode'], error_summary['Cumulative_Percentage'], color='red', marker='D', ms=7, label='Cumulative %')
ax2.set_ylabel('Cumulative Percentage (%)')
ax2.set_ylim(0, 110)

# 4. タイトルと補助線の追加
plt.title('Intune Deployment Error - Pareto Analysis', fontsize=14)
ax2.axhline(y=80, color='orange', linestyle='--', label='80% Threshold') # 80%ライン

# 凡例をまとめる
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

# 1. アプリとエラーコードの組み合わせで集計
# どのアプリで、どのエラーが何件起きているかをマトリックス化します
app_error_matrix = df_logs.groupby(['App', 'ErrorCode']).size().unstack(fill_value=0)

# 2. アプリごとの総エラー数を計算し、多い順に並べ替え
app_error_matrix['Total'] = app_error_matrix.sum(axis=1)
app_error_matrix = app_error_matrix.sort_values(by='Total', ascending=False)

print("--- Intune App-Specific Error Breakdown ---")
display(app_error_matrix)

# 3. 特定の最頻発エラー(0x80070005)が集中しているアプリを特定
target_error = '0x80070005'
if target_error in app_error_matrix.columns:
    top_app = app_error_matrix[target_error].idxmax()
    error_count = app_error_matrix[target_error].max()

    print(f"\n🔍 深掘り分析結果:")
    print(f"最頻発エラー '{target_error}' は、主に '{top_app}' ({error_count}件) で発生しています。")
    print(f"→ アクション: '{top_app}' の配布ポリシーとアクセス権限を集中的に調査してください。")

In [ ]:
import pandas as pd
from google.colab import files

# 1. 報告用データの準備
# 前回の集計結果（app_error_matrix）を使用し、Total順に並んだ状態を維持します
report_df = app_error_matrix.copy()

# 2. 統計情報の追加（エラー率などの計算も可能ですが、今回はシンプルに合計を強調）
report_df.index.name = 'Target_App'

# 3. CSVファイルとして書き出し
# utf-8-sig は、日本語環境のExcelで文字化けさせないための「お約束」です
filename = "Intune_App_Error_Investigation_Report.csv"
report_df.to_csv(filename, encoding='utf-8-sig')

print(f"--- 調査報告書 生成レポート ---")
print(f"✅ ファイル '{filename}' を正常に作成しました。")
print(f"✅ 最優先調査アプリ: {report_df.index[0]}")

# 4. ローカルPCへダウンロード
files.download(filename)

In [ ]:
import pandas as pd
from datetime import datetime, timedelta

# 1. データの日付を 2026 年（現在に近い日付）に更新
device_data = {
    'DeviceName': ['PC-UserA', 'iPad-UserB', 'PC-UserC', 'iPhone-UserD', 'Mac-UserE'],
    'OS': ['Windows 11', 'iOS', 'Windows 11', 'iOS', 'macOS'],
    'OS_Version': [22621, 14.2, 22621, 17.1, 13.5],
    'Encryption': ['Enabled', 'Disabled', 'Enabled', 'Enabled', 'Enabled'],
    'LastLogin': [
        '2026-04-10', # 比較的最近
        '2026-04-15', # 比較的最近
        '2026-03-01', # 1ヶ月以上前（これが抽出されるべき）
        '2026-04-20', # 非常に最近
        '2026-04-12'  # 比較的最近
    ]
}
df_devices = pd.DataFrame(device_data)
df_devices['LastLogin'] = pd.to_datetime(df_devices['LastLogin'])

# 2. セキュリティ・コンプライアンス判定
# A. 暗号化がオフの端末
encryption_alert = df_devices[df_devices['Encryption'] == 'Disabled']

# B. OSバージョンが古い端末 (iOS 15未満を対象とする)
outdated_os = df_devices[(df_devices['OS'] == 'iOS') & (df_devices['OS_Version'] < 15.0)]

# C. 30日以上ログインがない非アクティブ端末
thirty_days_ago = datetime.now() - timedelta(days=30)
inactive_devices = df_devices[df_devices['LastLogin'] < thirty_days_ago]

print("--- Intune Device Compliance Report ---")

if not encryption_alert.empty:
    print(f"\n🚨 【Security】暗号化がオフの端末を {len(encryption_alert)} 件検知しました:")
    display(encryption_alert[['DeviceName', 'OS', 'Encryption']])

if not outdated_os.empty:
    print(f"\n⚠️ 【Update Required】OSバージョンが古い端末を {len(outdated_os)} 件検知しました:")
    display(outdated_os[['DeviceName', 'OS', 'OS_Version']])

print(f"基準日（30日前）: {thirty_days_ago.date()}")
if not inactive_devices.empty:
    print(f"\nℹ️ 【Inventory】30日以上ログインがない端末を {len(inactive_devices)} 件検知しました:")
    display(inactive_devices[['DeviceName', 'LastLogin']])
else:
    print("\n✅ 30日以上放置されている端末はありません。")

In [ ]:
import matplotlib.pyplot as plt

# 1. OSごとの分布を集計
os_counts = df_devices['OS'].value_counts()

# 2. 円グラフの作成
plt.figure(figsize=(8, 8))
plt.pie(os_counts, labels=os_counts.index, autopct='%1.1f%%', startangle=140, colors=['skyblue', 'salmon', 'lightgreen'])
plt.title('Organizational Device OS Distribution', fontsize=14)
plt.axis('equal')
plt.show()

# 3. 特定OS（iOS）の深掘り調査
print("\n--- iOS Device Detailed Security Audit ---")
ios_devices = df_devices[df_devices['OS'] == 'iOS'].copy()

# セキュリティチェック（暗号化とOSバージョンの複合条件）
# 暗号化無効 または バージョン15未満 を「高リスク」とする
ios_devices['Risk_Level'] = ios_devices.apply(
    lambda x: '🔴 High Risk' if (x['Encryption'] == 'Disabled' or x['OS_Version'] < 15.0) else '✅ Compliant',
    axis=1
)

display(ios_devices[['DeviceName', 'OS_Version', 'Encryption', 'Risk_Level']])

In [ ]:
import pandas as pd

# 1. 旧システムのマスタ（移行元）
old_master = {
    '旧コード': ['100', '200', '300', '400'],
    '名称': ['現金', '普通預金', '売掛金', '支払利息']
}
df_old = pd.DataFrame(old_master)

# 2. 新システムのマスタ（移行先 / D365想定）
# ※ 3000番の名前が「未回収金」になっていたり、4000番が欠落していたりする想定
new_master = {
    '新コード': ['1000', '2000', '3000'],
    '名称': ['現金', '普通預金', '未回収金'] # '売掛金'ではなくなっている
}
df_new = pd.DataFrame(new_master)

# 3. 名称をキーにして外部結合（Outer Join）を行う
# indicator=True を使うと、どちらの表に存在するデータか一瞬でわかります
df_diff = pd.merge(df_old, df_new, on='名称', how='outer', indicator=True)

# 4. 差異（Diff）の分析
# 4-1. 新システムに移行漏れしている名称（旧にはあるが新にはない）
missing_in_new = df_diff[df_diff['_merge'] == 'left_only']

# 4-2. 名称が一致しない、またはコード変換が不明なもの
# 300番(売掛金)は、新システムでは「未回収金」になっているため、名称一致では紐付かない
# これを「名寄せが必要な候補」として抽出します

print("--- 新旧マスタ比較レポート ---")

if len(missing_in_new) > 0:
    print(f"⚠️ 移行漏れ・名称不一致の疑い: {len(missing_in_new)} 件")
    display(missing_in_new[['名称', '旧コード', '_merge']])

# 5. 【名寄せ・クレンジング】
# 類推やマッピング表を使って、新しい名称（未回収金）を「売掛金」に読み替える処理
# ここでは「名称の置換」を実演します
df_new['名称'] = df_new['名称'].replace('未回収金', '売掛金')

print("\n--- クレンジング実行後の再突合 ---")
df_cleaned = pd.merge(df_old, df_new, on='名称', how='inner')
display(df_cleaned)

In [ ]:
import pandas as pd

# 1. 採番データ（再定義）
seq_data = {
    '伝票番号': ['V001', 'V002', 'V004', 'V004', 'V005'],
    '金額': [100, 200, 400, 400, 500]
}
df_seq = pd.DataFrame(seq_data)

# 2. 数字部分を抽出
# ※ \d+ で数字を抜き出し、整数型に変換
df_seq['num'] = df_seq['伝票番号'].str.extract(r'(\d+)').astype(int)

# 3. 理想の連番リストを作成
# 最小値(1)から最大値(5)までの全ての数字を含んだ集合を作る
full_range = set(range(df_seq['num'].min(), df_seq['num'].max() + 1))

# 4. 実際のデータにある数字の集合
actual_nums = set(df_seq['num'])

# 5. 欠番の特定（引き算）
missing_nums = sorted(list(full_range - actual_nums))

print("\n--- 採番整合性レポート ---")

# 重複チェックの表示
duplicates = df_seq[df_seq.duplicated(subset=['伝票番号'], keep=False)]
if len(duplicates) > 0:
    print(f"🚫 重複エラー: 同じ伝票番号が {len(duplicates)} 件存在します。")
    display(duplicates[['伝票番号', '金額']])

# 欠番チェックの表示（ここを強化）
if missing_nums:
    print(f"⚠️ 欠番警告: 以下の番号がデータ内に存在しません。")
    for n in missing_nums:
        # V + 3桁ゼロ埋め形式に戻して表示
        print(f"  - V{n:03}")
else:
    print("✅ 番号の飛び（欠番）はありません。")

In [ ]:
import pandas as pd
import re

# 1. 不適切な文字を含んだ品目データ
# ※ 改行(\n)、タブ(\t)、アスタリスク(*)、アンパサンド(&)などが混入している想定
item_data = {
    '品目コード': ['ITM001', 'ITM002', 'ITM003', 'ITM004'],
    '品目名': [
        'Office Paper A4',
        'Keyboard\nWireless', # 改行が含まれている
        'Mouse &* Pad',       # & と * が含まれている
        'Laptop\tPro'         # タブが含まれている
    ]
}
df_item = pd.DataFrame(item_data)

# 2. クリーニング用の関数定義
def clean_special_characters(text):
    if not isinstance(text, str):
        return text

    # a. 改行、タブ、戻り（\n, \t, \r）を半角スペースに置換
    text = re.sub(r'[\n\t\r]', ' ', text)

    # b. D365で禁止・注意が必要な記号（* と &）をアンダースコアに置換
    # 業務ルールに合わせてここを書き換えます
    text = re.sub(r'[*&]', '_', text)

    # c. 連続したスペースを1つにまとめる（トリミング）
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# 3. クリーニングの実行
df_item['クリーニング後品目名'] = df_item['品目名'].apply(clean_special_characters)

# 4. 変更があった行だけを抽出して確認
df_changes = df_item[df_item['品目名'] != df_item['クリーニング後品目名']]

print("--- 特殊文字クリーニングレポート ---")
if len(df_changes) > 0:
    print(f"⚠️ 修正: 不適切な文字が含まれていたデータを {len(df_changes)} 件修正しました。")
    display(df_changes[['品目コード', '品目名', 'クリーニング後品目名']])
else:
    print("✅ クリーニングが必要な文字は見つかりませんでした。")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. 1ヶ月分の大量ログ（模擬データ）
# 4/15に異常なスパイク（山）がある想定
data = {
    '作成日時': pd.to_datetime(['2024-04-01', '2024-04-05', '2024-04-10', '2024-04-15']*50 + ['2024-04-15']*200),
    '伝票番号': range(400)
}
df_volume = pd.DataFrame(data)

# 2. 日次でカウントを集計
daily_trend = df_volume.set_index('作成日時').resample('D').count()

# 3. グラフ化して視覚的に特定
plt.figure(figsize=(10, 5))
plt.plot(daily_trend.index, daily_trend['伝票番号'], marker='o', color='teal')
plt.title('Daily Transaction Volume (Detection of Abnormal Spikes)')
plt.grid(True)
plt.show()

print("--- ボリューム分析結果 ---")
peak_day = daily_trend['伝票番号'].idxmax()
print(f"🚩 異常検知: {peak_day.date()} に最大件数 {daily_trend['伝票番号'].max()} 件が集中しています。")

In [ ]:
# 1. サポートログのデータ（模擬）
log_data = {
    'ErrorCode': ['ERR001', 'ERR002', 'ERR001', 'ERR003', 'ERR001', 'ERR002', 'ERR004'],
    'Message': [
        'Posting failed: Account locked',
        'Tax calculation error',
        'Posting failed: Account locked',
        'Currency conversion timeout',
        'Posting failed: Account locked',
        'Tax calculation error',
        'Unknown system error'
    ],
    'Module': ['Finance', 'Tax', 'Finance', 'Global', 'Finance', 'Tax', 'System']
}
df_logs = pd.DataFrame(log_data)

# 2. エラーメッセージの頻度（出現回数）をカウント
error_counts = df_logs['Message'].value_counts().reset_index()
error_counts.columns = ['ErrorMessage', 'Count']

print("\n--- エラーパターン解析レポート ---")
display(error_counts)

# 3. 最も頻発しているエラーを特定
top_error = error_counts.iloc[0]
print(f"🔥 最優先調査対象: '{top_error['ErrorMessage']}' が {top_error['Count']} 件発生しています。")

In [ ]:
import pandas as pd

# 1. 散在しているエラーログ
log_data = {
    'LogID': ['L1', 'L2', 'L3', 'L4', 'L5', 'L6'],
    'Message': [
        'Connection to server timed out',
        'User Taro lacks Permission to post',
        'Validation failed: Date is out of range',
        'Network Timeout occurred during sync',
        'Permission denied for account 1101',
        'Syntax error in XML data'
    ]
}
df_logs = pd.DataFrame(log_data)

# 2. カテゴリー判定のロジック
def categorize_error(msg):
    msg = msg.lower() # 検索しやすくするために小文字に統一
    if 'timeout' in msg:
        return 'Network/Timeout'
    elif 'permission' in msg:
        return 'Security/Permission'
    elif 'validation' in msg:
        return 'Data/Validation'
    else:
        return 'Other/Unknown'

# 3. 新しい列「Category」を作成して適用
df_logs['Category'] = df_logs['Message'].apply(categorize_error)

# 4. カテゴリーごとの集計
summary = df_logs['Category'].value_counts().reset_index()
summary.columns = ['Category', 'Count']

print("--- エラーメッセージ自動分類レポート ---")
display(df_logs)

print("\n--- カテゴリー別集計結果 ---")
display(summary)

In [ ]:
import pandas as pd

# 1. 散在しているエラーログ
log_data = {
    'LogID': ['L1', 'L2', 'L3', 'L4', 'L5', 'L6'],
    'Message': [
        'Connection to server timed out',
        'User Taro lacks Permission to post',
        'Validation failed: Date is out of range',
        'Network Timeout occurred during sync',
        'Permission denied for account 1101',
        'Syntax error in XML data'
    ]
}
df_logs = pd.DataFrame(log_data)

# 2. カテゴリー判定のロジック
def categorize_error(msg):
    msg = msg.lower()
    # 'timeout' も 'timed out' も両方カバーするために 'time' で判定
    if 'time' in msg:
        return 'Network/Timeout'
    elif 'permission' in msg or 'denied' in msg or 'lacks' in msg:
        return 'Security/Permission'
    elif 'validation' in msg:
        return 'Data/Validation'
    else:
        return 'Other/Unknown'

# 3. 新しい列「Category」を作成して適用
df_logs['Category'] = df_logs['Message'].apply(categorize_error)

# 4. カテゴリーごとの集計
summary = df_logs['Category'].value_counts().reset_index()
summary.columns = ['Category', 'Count']

print("--- エラーメッセージ自動分類レポート ---")
display(df_logs)

print("\n--- カテゴリー別集計結果 ---")
display(summary)

In [ ]:
import matplotlib.pyplot as plt

# 1. データの準備（集計結果を使用）
# summary は前回のコードで作成した df_logs['Category'].value_counts() です
labels = summary['Category']
counts = summary['Count']

# 2. 円グラフの作成
plt.figure(figsize=(8, 8))
# autopct='%1.1f%%' でパーセンテージを表示します
# startangle=140 で見やすい角度から開始します
plt.pie(counts, labels=labels, autopct='%1.1f%%', startangle=140,
        colors=['skyblue', 'lightgreen', 'salmon', 'gold'])

# 3. タイトルとレイアウトの設定
plt.title('Error Category Distribution', fontsize=15)
plt.axis('equal')  # 円を正円にします

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

# 1. ログインログデータ
login_logs = {
    '時刻': pd.to_datetime(['2024-04-15 10:00', '2024-04-15 10:01', '2024-04-15 10:02',
                          '2024-04-15 10:02', '2024-04-15 10:03', '2024-04-15 10:03']),
    'IPアドレス': ['192.168.1.1', '192.168.1.1', '10.0.0.5', '192.168.1.1', '192.168.1.1', '10.0.0.5'],
    '結果': ['Fail', 'Fail', 'Success', 'Fail', 'Fail', 'Success']
}
df_login = pd.DataFrame(login_logs)

# 2. 失敗（Fail）したログのみを抽出
failed_attempts = df_login[df_login['結果'] == 'Fail']

# 3. IPアドレスごとに失敗回数をカウント
attack_summary = failed_attempts.groupby('IPアドレス').size().reset_index(name='失敗回数')

# 4. 3回以上失敗しているIPを「攻撃の疑い」として抽出
suspicious_ips = attack_summary[attack_summary['失敗回数'] >= 3]

print("--- 不正ログイン試行レポート ---")
if len(suspicious_ips) > 0:
    print(f"🚨 警告: 攻撃の疑いがあるIPアドレスが {len(suspicious_ips)} 件見つかりました。")
    display(suspicious_ips)
else:
    print("✅ 不審なログイン試行はありません。")

In [ ]:
# 1. 処理時間ログ（秒単位）
perf_data = {
    '処理名': ['InvoicePost', 'PaymentRun', 'InvoicePost', 'ReportGen', 'InvoicePost', 'PaymentRun'],
    '所要時間_秒': [2.5, 120.0, 3.1, 45.0, 250.0, 115.0] # 3回目のInvoicePostが異常に長い
}
df_perf = pd.DataFrame(perf_data)

# 2. 処理名ごとの「平均」と「最大」の所要時間を算出
performance_summary = df_perf.groupby('処理名')['所要時間_秒'].agg(['mean', 'max', 'count']).reset_index()

# 3. 平均の2倍以上の時間がかかっている個別の異常値を特定
# ここでは「200秒以上」を閾値（しきいち）として警告します
threshold = 200
slow_queries = df_perf[df_perf['所要時間_秒'] > threshold]

print("\n--- パフォーマンス統計レポート ---")
display(performance_summary)

if len(slow_queries) > 0:
    print(f"🐢 遅延検知: {threshold}秒以上かかっている処理が {len(slow_queries)} 件あります。改善を検討してください。")
    display(slow_queries)

In [ ]:
import pandas as pd
import numpy as np

# 1. 処理実行ログ（バッチ処理の履歴を想定）
# Batch_Aは安定、Batch_Bは非常に不安定なデータ
performance_logs = {
    'バッチ名': ['Batch_A']*5 + ['Batch_B']*5,
    '所要時間_秒': [10, 11, 10, 12, 10,  # Batch_A: 安定
                 5, 150, 10, 200, 12]  # Batch_B: 不安定（スパイクが発生）
}
df_perf = pd.DataFrame(performance_logs)

# 2. 統計量の算出（平均、最大、標準偏差）
# std() が標準偏差を計算するメソッドです
stats = df_perf.groupby('バッチ名')['所要時間_秒'].agg(['mean', 'max', 'std']).reset_index()

# 3. 安定性の評価
# 標準偏差が平均の 50% を超えている場合を「不安定」と定義してみます
stats['安定性評価'] = stats.apply(
    lambda x: '⚠️ 不安定 (Unstable)' if x['std'] > (x['mean'] * 0.5) else '✅ 安定 (Stable)',
    axis=1
)

print("--- バッチ処理 安定性解析レポート ---")
display(stats)

# 4. 不安定なバッチの理由を表示
unstable_batches = stats[stats['安定性評価'].str.contains('不安定')]
if len(unstable_batches) > 0:
    print(f"\n📢 エスカレーションが必要な候補: {unstable_batches['バッチ名'].tolist()}")
    print("理由: 標準偏差(std)が高く、処理時間に極端なばらつきがあります。リソース競合の調査を推奨します。")

In [ ]:
import matplotlib.pyplot as plt

# 1. グラフの作成
plt.figure(figsize=(8, 6))

# boxplot関数に、バッチごとのデータを渡します
# バッチAのデータとバッチBのデータを分けてリスト化します
data_a = df_perf[df_perf['バッチ名'] == 'Batch_A']['所要時間_秒']
data_b = df_perf[df_perf['バッチ名'] == 'Batch_B']['所要時間_秒']

# plt.boxplot([data_a, data_b], labels=['Batch_A', 'Batch_B'])
plt.boxplot([data_a, data_b], tick_labels=['Batch_A', 'Batch_B'])

# 2. ラベルとタイトルの設定
plt.title('Performance Variability (Box Plot)', fontsize=14)
plt.ylabel('Duration (Seconds)')
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show()

In [ ]:
import pandas as pd
from google.colab import files

# 1. 調査したいカテゴリーを指定（例：Network/Timeout）
target_category = 'Network/Timeout'

# 2. 指定したカテゴリーのみを抽出
investigation_df = df_logs[df_logs['Category'] == target_category].copy()

# 3. 調査に不要な列があれば削除し、必要な情報を整える
# 今回はシンプルにそのまま出力します
output_data = investigation_df[['LogID', 'Message']]

# 4. CSVファイルとして書き出し
# ファイル名にカテゴリー名を含めると親切です
file_name = f"Investigation_Report_{target_category.replace('/', '_')}.csv"
output_data.to_csv(file_name, index=False, encoding='utf-8-sig')

print(f"--- 調査資料作成レポート ---")
print(f"✅ カテゴリー '{target_category}' のデータを {len(output_data)} 件抽出しました。")
print(f"✅ ファイル '{file_name}' を作成しました。")

# 5. ローカルPCへダウンロード
files.download(file_name)

In [ ]:
import pandas as pd
from google.colab import files

# ※ 前回のセルで作成したクリーニング済みの `df_item` を使用します

# 1. 出力する列を整理（インポートに必要な列だけに絞るのが実務的です）
# ここでは品目コードと、クリーニング後の品目名のみを抽出します
df_for_import = df_item[['品目コード', 'クリーニング後品目名']].copy()

# 2. 列名をD365のエンティティ（ItemNameなど）に合わせてリネーム（任意）
df_for_import.columns = ['ItemNumber', 'ItemName']

# 3. CSVファイルとして書き出し
# utf-8-sig を使うのが日本語環境での「お作法」です
csv_file = 'D365_Item_Import_Ready.csv'
df_for_import.to_csv(csv_file, index=False, encoding='utf-8-sig', quoting=1)

print(f"✅ インポート用ファイル '{csv_file}' を作成しました。")

# 4. ローカルPCへダウンロード
files.download(csv_file)

In [ ]:
import pandas as pd
import numpy as np

# 1. 項目が欠落しているマスタデータ（模擬データ）
# ※ 200は科目名が欠落、300はBSなのに税区分が欠落している想定
master_data = {
    '勘定科目コード': [100, 200, 300, 400],
    '科目名': ['現金', np.nan, '売掛金', '売上高'],
    '勘定タイプ': ['BS', 'BS', 'BS', 'PL'],
    '税区分': ['非課税', '非課税', np.nan, '課税']
}
df_master = pd.DataFrame(master_data)

# 2. 全般的な欠損値（空欄）のチェック
# isnull() を使うことで、どこに空欄があるか一瞬でわかります
missing_any = df_master[df_master.isnull().any(axis=1)]

# 3. 業務ルールに基づいた特定の不整合チェック
# ルール: 「勘定タイプが BS」かつ「税区分が空(null)」のものを探す
rule_violation = df_master[
    (df_master['勘定タイプ'] == 'BS') & (df_master['税区分'].isnull())
]

print("--- マスタ項目バリデーションレポート ---")

# レポート出力1: 単純な空欄チェック
if len(missing_any) > 0:
    print(f"⚠️ 全般警告: 項目が未入力の行が {len(missing_any)} 件あります。")
    display(missing_any)

# レポート出力2: 業務ルール違反チェック
if len(rule_violation) > 0:
    print(f"\n🚫 業務ルール違反: BS科目で税区分が未設定の重大なエラーが {len(rule_violation)} 件あります。")
    display(rule_violation[['勘定科目コード', '科目名', '勘定タイプ']])
else:
    print("\n✅ すべてのBS科目に税区分が設定されています。")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. サーバーの負荷ログ（模擬データ）
# 負荷（同時接続数）が増えるほど、処理時間が伸びているようなデータを想定
load_data = {
    '時刻': ['10:00', '11:00', '12:00', '13:00', '14:00', '15:00', '16:00'],
    '同時接続数': [10, 50, 100, 150, 200, 250, 300],
    '平均応答時間_ms': [100, 120, 250, 400, 800, 1200, 1800] # 急激に伸びている
}
df_load = pd.DataFrame(load_data)

# 2. 相関係数（Correlation）の算出
# 1に近いほど「強い正の相関」があり、一方が増えるともう一方も増えることを意味します
correlation = df_load['同時接続数'].corr(df_load['平均応答時間_ms'])

# 3. 散布図（Scatter Plot）による可視化
plt.figure(figsize=(8, 6))
plt.scatter(df_load['同時接続数'], df_load['平均応答時間_ms'], color='purple', s=100)

# 近似線（トレンドライン）の追加
import numpy as np
z = np.polyfit(df_load['同時接続数'], df_load['平均応答時間_ms'], 1)
p = np.poly1d(z)
plt.plot(df_load['同時接続数'], p(df_load['同時接続数']), "r--", alpha=0.8)

plt.title(f'Correlation Analysis (r = {correlation:.2f})', fontsize=14)
plt.xlabel('Number of Concurrent Users')
plt.ylabel('Average Response Time (ms)')
plt.grid(True, linestyle=':', alpha=0.6)

plt.show()

print(f"--- 分析結果 ---")
print(f"📈 相関係数: {correlation:.2f}")
if correlation > 0.8:
    print("🚩 強い相関を検知：同時接続数の増加がレスポンス低下の直接的な原因である可能性が高いです。")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. 過去データの準備（前回のデータを使用）
x_past = df_load['同時接続数']
y_past = df_load['平均応答時間_ms']

# 2. 線形回帰モデルの作成 (y = ax + b)
# np.polyfit で傾き(a)と切片(b)を算出します
a, b = np.polyfit(x_past, y_past, 1)

# 3. 来月の予測（同時接続数 500 の時）
future_users = 500
predicted_response_time = a * future_users + b

print(f"--- 未来予測レポート ---")
print(f"📈 予測モデル: 応答時間 = {a:.2f} × 接続数 + {b:.2f}")
print(f"🔮 予測結果: 同時接続数が {future_users} 名に達した時、")
print(f"   平均応答時間は 【{predicted_response_time:.1f} ms】 まで悪化すると予測されます。")

# 4. 可視化
plt.figure(figsize=(10, 6))
# 過去データの散布図
plt.scatter(x_past, y_past, color='purple', label='Past Data')
# 回帰線（トレンドライン）
x_range = np.array([min(x_past), future_users])
plt.plot(x_range, a * x_range + b, 'r--', alpha=0.6, label='Prediction Line')
# 予測ポイント
plt.scatter(future_users, predicted_response_time, color='gold', s=200, marker='*', label='Predicted Point (500 users)')

plt.title('Performance Prediction for Next Month', fontsize=14)
plt.xlabel('Number of Concurrent Users')
plt.ylabel('Average Response Time (ms)')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)

plt.show()

In [ ]:
import datetime

# 1. SLAの設定（2秒 = 2000ms を超えたら警告）
SLA_THRESHOLD = 2000
target_date = (datetime.datetime.now() + datetime.timedelta(days=30)).strftime('%Y-%m-%d')

# 2. 自動判定とメッセージ生成
def generate_alert_email(predicted_value, threshold, user_count):
    if predicted_value > threshold:
        # 警告文のテンプレート（f-stringを使用）
        subject = f"【Alert】System Performance Prediction Risk - {target_date}"
        body = f"""
Dear System Administration Team,

This is an automated performance risk alert based on our latest regression analysis.

[Summary]
Our predictive model indicates that the system response time will exceed the agreed SLA threshold ({threshold} ms) within the next 30 days.

[Prediction Details]
- Estimated Date: {target_date}
- Predicted User Load: {user_count} concurrent users
- Predicted Response Time: {predicted_value:.1f} ms (SLA Violation: +{predicted_value - threshold:.1f} ms)

[Recommendation]
Immediate investigation into server resource scaling (CPU/Memory) or database query optimization is required to prevent a potential service outage.

This report was automatically generated by the Performance Forecasting Tool.
--------------------------------------------------
GitHub Repo: [Your GitHub URL]
        """
        return subject, body
    else:
        return None, "System performance is within expected SLA parameters."

# 3. 実行
subject, email_content = generate_alert_email(predicted_response_time, SLA_THRESHOLD, future_users)

print("--- 自動生成された警告メール本文 ---")
if subject:
    print(f"Subject: {subject}")
    print(email_content)
else:
    print(email_content)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. 大量の仕訳金額データ（模擬データ）
# 意図的に「8」で始まる数字を多く混ぜて、不自然なデータを作ります
amounts = [120, 1500, 180, 250, 310, 4500, 560, 670, 780, 900] * 20
suspicious_data = [8800, 8120, 8500, 8000, 8990] * 15 # 「8」で始まる不自然なデータ
df_audit = pd.DataFrame({'Amount': amounts + suspicious_data})

# 2. 先頭の数字（First Digit）を抽出
df_audit['First_Digit'] = df_audit['Amount'].astype(str).str[0].astype(int)

# 3. 実際の出現頻度を計算
actual_counts = df_audit['First_Digit'].value_counts(normalize=True).sort_index()

# 4. ベンフォードの法則の理論値（理想的な分布）
benford_theoretical = [0.301, 0.176, 0.125, 0.097, 0.079, 0.067, 0.058, 0.051, 0.046]

# 5. 可視化して比較
plt.figure(figsize=(10, 6))
plt.bar(range(1, 10), actual_counts, alpha=0.7, label='Actual Data', color='skyblue')
plt.step(range(1, 10), benford_theoretical, where='mid', color='red', label='Benford Line', linewidth=2)

plt.title("Fraud Detection: Benford's Law Analysis", fontsize=14)
plt.xlabel('First Digit')
plt.ylabel('Frequency')
plt.xticks(range(1, 10))
plt.legend()
plt.show()

# 6. 異常の特定
if actual_counts[8] > 0.1: # 8の出現率が異常に高い場合
    print(f"🚩 異常検知: 数字 '8' で始まる金額が {actual_counts[8]*100:.1f}% も存在します。")
    print("   これは自然な分布を逸脱しており、意図的な操作や特定のパターンのエラーが疑われます。")

In [ ]:
import pandas as pd

# 1. 仕訳データ（模擬データ）
# ※ V003 と V005 が、あまりにも「キリが良すぎる」不自然な金額の想定
journal_data = {
    '伝票番号': ['V001', 'V002', 'V003', 'V004', 'V005'],
    '内容': ['PC購入', '事務用品', 'コンサルティング料', 'タクシー代', '広告宣伝費'],
    '金額': [152430, 8420, 1000000, 1250, 500000], # 100万と50万
    '担当者': ['UserA', 'UserB', 'UserC', 'UserA', 'UserC']
}
df_audit = pd.DataFrame(journal_data)

# 2. キリの良い数字判定ロジック
# 金額を 1,000 で割った余りが 0 かどうかを判定します
# ここでは「10,000円単位」でピッタリのものを抽出対象とします
threshold = 10000
df_audit['Is_Round'] = df_audit['金額'] % threshold == 0

# 3. 異常データの抽出
round_transactions = df_audit[df_audit['Is_Round'] == True]

# 4. 担当者ごとの傾向分析（特定の人が多用していないか）
round_summary = round_transactions.groupby('担当者').size().reset_index(name='キリの良い数字の件数')

print(f"--- 監査レポート：キリの良い数字 ({threshold:,}円単位) の検知 ---")
if len(round_transactions) > 0:
    print(f"⚠️ 警告: 不自然にキリの良い金額が {len(round_transactions)} 件見つかりました。")
    display(round_transactions[['伝票番号', '内容', '金額', '担当者']])

    print("\n--- 担当者別・不自然な計上の集計 ---")
    display(round_summary)
else:
    print("✅ キリの良い数字の偏りは見つかりませんでした。")

In [ ]:
import pandas as pd

# 1. 大規模な仕訳データ（模擬データ）
audit_data = {
    '伝票番号': ['V001', 'V002', 'V003', 'V004', 'V005', 'V006'],
    '内容': ['オフィス賃借', '通信費', 'コンサル料', '交通費', '機材購入', '雑費'],
    '金額': [823450, 12400, 800000, 4560, 500000, 8000], # 80万と50万が特に怪しい
    '担当者': ['UserA', 'UserB', 'UserC', 'UserA', 'UserC', 'UserB']
}
df_final = pd.DataFrame(audit_data)

# 2. フラグ1：ベンフォードの法則に基づく「不自然な先頭桁」
# 前回の分析で異常だった「8」をターゲットにします
df_final['First_Digit'] = df_final['金額'].astype(str).str[0].astype(int)
df_final['Flag_Benford'] = df_final['First_Digit'] == 8

# 3. フラグ2：キリの良い数字（10,000円単位）
df_final['Flag_Round'] = df_final['金額'] % 10000 == 0

# 4. リスクスコアの算出（両方のフラグが立てば 2）
df_final['Risk_Score'] = df_final[['Flag_Benford', 'Flag_Round']].sum(axis=1)

# 5. 最重要調査対象（High Risk）の抽出
high_risk_targets = df_final[df_final['Risk_Score'] >= 2]

print("--- 統合監査レポート：最重要調査対象 (High Risk) ---")
if len(high_risk_targets) > 0:
    print(f"🚩 警告: 複数の不正兆候が重なる『最重要調査対象』が {len(high_risk_targets)} 件あります。")
    display(high_risk_targets[['伝票番号', '内容', '金額', '担当者', 'Risk_Score']])
else:
    print("✅ 複数のリスクが重なる重大な異常は見つかりませんでした。")

# 全体の分布確認
print("\n--- 全データの全容とリスクスコア ---")
display(df_final.sort_values(by='Risk_Score', ascending=False))

In [ ]:
import pandas as pd
import seaborn as sns # 色の管理に便利なライブラリ

# 1. データの準備（前回の df_final を使用します）
# 表示用に重要な列だけを選択し、スコア順に並べ替えます
df_heatmap = df_final[['伝票番号', '内容', '金額', '担当者', 'Risk_Score']].sort_values(by='Risk_Score', ascending=False)

# 2. ヒートマップ（条件付き書式）の適用
# 'Risk_Score' 列の値に応じて、赤色のグラデーションをつけます
styled_df = df_heatmap.style.background_gradient(cmap='Reds', subset=['Risk_Score'])

print("--- 統合リスク・ヒートマップ (Integrated Risk Heatmap) ---")
print("※ 赤色が濃い行ほど、複数の不正兆候が重なっている最重要調査対象です。")

# Google Colab で綺麗に表示するための出力
styled_df

In [ ]:
import pandas as pd

# 1. 人事マスタ（正規の社員リスト）
employee_master = {
    '社員名': ['UserA', 'UserB', 'UserC'],
    '部署': ['経理部', '営業部', 'IT部']
}
df_emp = pd.DataFrame(employee_master)

# 2. 仕訳データ
# 金額のリスト [10000, 20000, 50000, 30000] を追記して修正しました
journal_data = {
    '伝票番号': ['V001', 'V002', 'V003', 'V004'],
    '作成者': ['UserA', 'UserB', 'GhostUser', 'UserC'],
    '金額': [10000, 20000, 50000, 30000]
}
df_journal = pd.DataFrame(journal_data)

# 3. 外部結合（Left Join）でマスタをぶつける
df_cross_check = pd.merge(df_journal, df_emp, left_on='作成者', right_on='社員名', how='left', indicator=True)

# 4. 「幽霊社員（left_only）」による仕訳を抽出
ghost_entries = df_cross_check[df_cross_check['_merge'] == 'left_only']

print("--- クロス・バリデーション：幽霊社員検知レポート ---")
if len(ghost_entries) > 0:
    print(f"🚨 重大な不整合検知: 人事マスタに未登録の『幽霊作成者』による仕訳が {len(ghost_entries)} 件あります。")
    display(ghost_entries[['伝票番号', '作成者', '金額']])
else:
    print("✅ すべての仕訳は正規の社員によって作成されています。")

In [ ]:
import pandas as pd

# 1. 統合されたテストデータ（V003が全項目で怪しい、V002は幽霊社員のみ）
merged_data = {
    '伝票番号': ['V001', 'V002', 'V003', 'V004'],
    '作成者': ['UserA', 'GhostUser', 'GhostUser', 'UserC'],
    '金額': [123450, 20000, 800000, 456700], # V003は8(統計)かつ0000(業務)
    'マスタ登録': [True, False, False, True] # 人事マスタに存在するか
}
df_final = pd.DataFrame(merged_data)

# 2. リスクフラグの算出
# A. 統計リスク（先頭が8）
df_final['F_Stat'] = df_final['金額'].astype(str).str.startswith('8')

# B. 業務リスク（キリが良い）
df_final['F_Biz'] = df_final['金額'] % 10000 == 0

# C. 組織リスク（人事マスタ不在）
df_final['F_Org'] = ~df_final['マスタ登録']

# 3. 統合スコアリング（組織リスクは重大なため 2点加算とする）
df_final['Risk_Score'] = (
    df_final['F_Stat'].astype(int) +
    df_final['F_Biz'].astype(int) +
    (df_final['F_Org'].astype(int) * 2) # 重み付け
)

# 4. 深刻度の判定
def judge_severity(score):
    if score >= 3: return '🔴 Critical (Fraud Suspected)'
    elif score >= 2: return '🟡 Warning (Review Required)'
    else: return '🟢 Normal'

df_final['Severity'] = df_final['Risk_Score'].apply(judge_severity)

print("--- 統合リスク・インテリジェンス・レポート ---")
# スコア順に並べ替えて表示
display(df_final.sort_values(by='Risk_Score', ascending=False))

# 最優先調査対象の抽出
critical_list = df_final[df_final['Risk_Score'] >= 3]
if not critical_list.empty:
    print(f"\n🔥 最優先調査対象が {len(critical_list)} 件あります。即座にエスカレーションが必要です！")

In [ ]:
import pandas as pd

# 1. 模擬仕訳データ（大量のログを想定）
raw_data = {
    '部門': ['営業部', 'IT部', '営業部', '人事部', 'IT部', '営業部', '人事部', 'IT部'],
    '勘定科目': ['旅費', '機材費', '交際費', '採用費', 'ソフト費', '旅費', '研修費', '機材費'],
    '金額': [50000, 200000, 30000, 150000, 80000, 45000, 60000, 210000],
    'ステータス': ['Success', 'Success', 'Failed', 'Success', 'Success', 'Success', 'Success', 'Failed']
}
df_logs = pd.DataFrame(raw_data)

# 2. ピボットテーブルの作成
# 「部門」を行、「勘定科目」を列にして、金額を合計（sum）します
pivot_summary = df_logs.pivot_table(
    index='部門',
    columns='勘定科目',
    values='金額',
    aggfunc='sum',
    fill_value=0 # データがない場所を0で埋める
)

# 3. エラー（Failed）が発生している部門と科目を特定
error_logs = df_logs[df_logs['ステータス'] == 'Failed']

print("--- 部門別・科目別 支出集計表 (Pivot Table) ---")
display(pivot_summary)

print("\n--- 自動生成された異常（Failed）調査依頼メッセージ ---")
if not error_logs.empty:
    for index, row in error_logs.iterrows():
        print(f"🚩 【調査依頼】 {row['部門']} の {row['勘定科目']}（{row['金額']:,}円）が投稿エラーとなっています。ログを確認してください。")
else:
    print("✅ すべての処理は正常に完了しています。")

In [ ]:
import matplotlib.pyplot as plt

# --- 1. 表示用のデータを英語に変換（マッピング） ---
# インデックス（部門）の変換
dept_map = {'IT部': 'IT', '人事部': 'HR', '営業部': 'Sales'}
pivot_summary.index = pivot_summary.index.map(dept_map)

# カラム（科目）の変換
account_map = {
    '旅費': 'Travel', '機材費': 'Hardware', '交際費': 'Ent.',
    '採用費': 'Recruit', 'ソフト費': 'Software', '研修費': 'Training'
}
pivot_summary.columns = pivot_summary.columns.map(account_map)

# --- 2. グラフの作成 ---
ax = pivot_summary.plot(kind='bar', stacked=True, figsize=(10, 6), colormap='viridis')

# 3. タイトルとラベルの設定（英語）
plt.title('Expenditure Breakdown by Department', fontsize=15)
plt.xlabel('Department')
plt.ylabel('Amount (JPY)')
plt.xticks(rotation=0)
plt.legend(title='Account', bbox_to_anchor=(1.05, 1), loc='upper left')

# 4. 合計値の表示
totals = pivot_summary.sum(axis=1)
for i, total in enumerate(totals):
    ax.text(i, total + 5000, f'{total:,.0f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

# 1. 形式が不適切なマスタデータ（模擬データ）
# ※ 5001 (5桁でNG), 600A (文字混入でNG), 0700 (0開始でNG)
master_raw = {
    '勘定科目コード': ['1000', '2000', '50001', '600A', '0700'],
    '科目名': ['現金', '普通預金', '固定資産', '支払利息', '雑費']
}
df_master = pd.DataFrame(master_raw)

# 2. 形式チェックロジック
# 2-1. 桁数チェック (4桁以外を抽出)
df_master['桁数'] = df_master['勘定科目コード'].str.len()
invalid_length = df_master[df_master['桁数'] != 4]

# 2-2. 数字のみチェック (数字以外が含まれるものを抽出)
# isdigit() を使って判定
invalid_type = df_master[~df_master['勘定科目コード'].str.isdigit()]

# 2-3. 先頭文字チェック (0から始まるものを抽出)
invalid_start = df_master[df_master['勘定科目コード'].str.startswith('0')]

# 3. レポート出力
print("--- データ形式バリデーションレポート ---")

if len(invalid_length) > 0:
    print(f"⚠️ 桁数エラー: 4桁ではないコードが {len(invalid_length)} 件あります。")
    display(invalid_length)

if len(invalid_type) > 0:
    print(f"🚫 型エラー: 数字以外の文字が含まれるコードが {len(invalid_type)} 件あります。")
    display(invalid_type)

if len(invalid_start) > 0:
    print(f"🛑 運用ルール違反: '0'から始まる無効なコードが {len(invalid_start)} 件あります。")
    display(invalid_start)

In [ ]:
import pandas as pd
import unicodedata

# 1. データの準備
df_old = pd.DataFrame({
    '旧コード': ['100', '200', '300'],
    '名称': ['現金　', '普通預金', '売掛金']
})

df_new = pd.DataFrame({
    '新コード': ['1000', '2000', '3000'],
    '名称': ['現金', '普通預金', 'ｳﾘｶｹｷﾝ'] # 半角カナ
})

# 2. 【重要】辞書のキーを正規化（NFKC）後の「全角カナ」で定義する
# NFKCを通ると半角の 'ｳﾘｶｹｷﾝ' は 全角の 'ウリカケキン' になります
name_mapping = {
    'ウリカケキン': '売掛金',
    'ゲンキン': '現金'
}

# 3. クレンジング関数
def advanced_normalize(text):
    if not isinstance(text, str):
        return text

    # a. ここで「全角」に統一される（半角カナ -> 全角カナ）
    clean_text = unicodedata.normalize('NFKC', text).strip()

    # b. 統一された状態（全角カナ）で辞書を引く
    return name_mapping.get(clean_text, clean_text)

# 4. 適用
df_old['名称_final'] = df_old['名称'].apply(advanced_normalize)
df_new['名称_final'] = df_new['名称'].apply(advanced_normalize)

# 5. 突合
df_matched = pd.merge(df_old, df_new, on='名称_final', how='inner')

print("--- 高度な名寄せ（NFKC考慮版）レポート ---")
display(df_matched[['名称_final', '旧コード', '新コード']])

In [ ]:
import pandas as pd

# 1. 形式がバラバラな日付データ
date_data = {
    '伝票番号': ['V1', 'V2', 'V3', 'V4'],
    '取引日': ['2024/04/15', '2024.04.16', '20240417', 'invalid-date']
}
df_date = pd.DataFrame(date_data)

# 2. pd.to_datetime を使って一括変換
# errors='coerce' を使うことで、解析不能な文字列を NaT (Not a Time) に変換し、エラー停止を防ぎます
df_date['正規化日付'] = pd.to_datetime(df_date['取引日'], errors='coerce')

# 3. D365形式 (YYYY-MM-DD) の文字列に変換
df_date['D365形式'] = df_date['正規化日付'].dt.strftime('%Y-%m-%d')

print("--- 日付正規化レポート ---")
display(df_date)

# NaT (変換失敗) がある場合の警告
errors = df_date[df_date['正規化日付'].isnull()]
if len(errors) > 0:
    print(f"⚠️ 警告: 日付として認識できないデータが {len(errors)} 件あります。")

In [ ]:
import pandas as pd
import numpy as np

# 1. 様々なエラーを含んだテストデータ
master_raw = {
    '勘定科目コード': ['1000', '2000', '50001', '600A', '0700', '1000'], # 1000が重複
    '科目名': ['現金', '普通預金', '固定資産', np.nan, '雑費', '現金']
}
df = pd.DataFrame(master_raw)

# 2. 統合バリデーションの実行
# 2-1. 桁数エラー (4桁以外)
df['Err_桁数'] = df['勘定科目コード'].str.len() != 4

# 2-2. 型エラー (数字以外)
df['Err_型'] = ~df['勘定科目コード'].str.isdigit()

# 2-3. 空欄エラー (科目名がNaN)
df['Err_空欄'] = df['科目名'].isnull()

# 2-4. 重複エラー (コードが重複)
df['Err_重複'] = df.duplicated(subset=['勘定科目コード'], keep=False)

# 3. エラーの集計
# True(1) と False(0) を足して、その行にいくつエラーがあるか算出
error_columns = ['Err_桁数', 'Err_型', 'Err_空欄', 'Err_重複']
df['エラー件数'] = df[error_columns].sum(axis=1)

# 4. エラーがある行だけを抽出した「統合レポート」
error_report = df[df['エラー件数'] > 0].copy()

print("--- 統合データバリデーション・レポート ---")
if len(error_report) > 0:
    print(f"⚠️ 修正が必要な行が {len(error_report)} 件あります。")
    # 見やすくするために、エラーがない列は非表示にしたり並べ替えたりします
    display(error_report)
else:
    print("✅ 完璧です！すべてのデータが正常です。")

In [ ]:
import pandas as pd

# ※ 前回のセルで作成した `error_report` が存在することが前提です

# 1. Excelファイルとして書き出し
# ファイル名を 'Validation_Report.xlsx' とします
file_name = 'Master_Data_Validation_Report.xlsx'
error_report.to_excel(file_name, index=False)

print(f"✅ Excelファイル '{file_name}' を作成しました。")

# 2. Google ColabからローカルPCへダウンロードするためのコード
from google.colab import files
files.download(file_name)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. サンプルログデータ（4月初旬の10日間を想定）
trend_data = {
    '作成日時': [
        '2024-04-01 10:00:00', '2024-04-01 15:00:00',
        '2024-04-02 11:00:00',
        '2024-04-05 10:00:00', '2024-04-05 12:00:00', '2024-04-05 14:00:00', # 4/5に山
        '2024-04-06 09:00:00',
        '2024-04-09 13:00:00', '2024-04-10 10:00:00', '2024-04-10 16:00:00'
    ],
    '伝票番号': ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10']
}
df_trend = pd.DataFrame(trend_data)
df_trend['作成日時'] = pd.to_datetime(df_trend['作成日時'])

# 2. 日次での集計 (D = Daily)
# set_index してから resample することで「日次」の件数を算出
daily_counts = df_trend.set_index('作成日時').resample('D').count()

# 3. 折れ線グラフの作成
plt.figure(figsize=(12, 6))
plt.plot(daily_counts.index, daily_counts['伝票番号'], marker='o', linestyle='-', color='teal', linewidth=2)

# 4. タイトルとラベル（英語）
plt.title('Daily Transaction Count Trend (April 2024)', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Number of Transactions')
plt.grid(True, linestyle='--', alpha=0.7) # グリッド線を追加して見やすく

# 日付の表示形式を調整
plt.xticks(daily_counts.index, [d.strftime('%Y-%m-%d') for d in daily_counts.index], rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

# 1. 勘定科目マスタ（正規のリスト）
# 本来は 100, 200, 300 しか存在しない
master_data = {
    '勘定科目コード': [100, 200, 300],
    '科目名': ['現金', '普通預金', '売掛金']
}
df_master = pd.DataFrame(master_data)

# 2. 実際に発生した仕訳データ
# ※ 伝票 V003 はマスタに存在しない「999」という不正なコードを使っている想定
journal_data = {
    '伝票番号': ['V001', 'V002', 'V003', 'V004'],
    '勘定科目コード': [100, 200, 999, 100],
    '金額': [5000, 8000, 15000, 3000]
}
df_journal = pd.DataFrame(journal_data)

# 3. 外部結合でマスタをぶつける
# how='left' を使うことで仕訳をすべて残しつつ、マスタ情報をくっつける
df_check = pd.merge(df_journal, df_master, on='勘定科目コード', how='left')

# 4. マスタに存在しない（科目名が NaN になっている）行を抽出
# これをエスカレーションエンジニアは「Orphaned Record (親なしレコード)」と呼びます
orphans = df_check[df_check['科目名'].isnull()]

print("--- マスタ整合性チェックレポート ---")
if len(orphans) > 0:
    print(f"⚠️ 致命的なエラー: マスタに未登録の勘定科目を使用している仕訳が {len(orphans)} 件あります。")
    display(orphans[['伝票番号', '勘定科目コード', '金額']])
else:
    print("✅ すべての仕訳は有効なマスタを参照しています。")

In [ ]:
import pandas as pd

# 1. 矛盾を含んだマスタデータ（模擬データ）
# ※ コード 100 に「現金」と「現預金」が混在している
master_data_raw = {
    '勘定科目コード': [100, 100, 200, 300, 300],
    '科目名': ['現金', '現預金', '普通預金', '売掛金', '売掛金'] # 300は重複だが名前は同じ
}
df_master = pd.DataFrame(master_data_raw)

# 2. まず「コードと名前」のユニークな組み合わせを作る
df_unique = df_master.drop_duplicates()

# 3. コードごとに「名前がいくつ登録されているか」をカウント
# nunique() はユニークな値の個数を数える関数です
name_counts = df_unique.groupby('勘定科目コード')['科目名'].nunique().reset_index()
name_counts.columns = ['勘定科目コード', '名前の種類数']

# 4. 種類数が 2 以上のものが「矛盾（不整合）」
inconsistent_codes = name_counts[name_counts['名前の種類数'] > 1]

# 5. 該当するコードの具体的な中身を表示
inconsistent_master = df_unique[df_unique['勘定科目コード'].isin(inconsistent_codes['勘定科目コード'])]

print("--- マスタ名称不整合（表記ゆれ）チェックレポート ---")
if len(inconsistent_master) > 0:
    print(f"⚠️ 警告: 同じコードに複数の名称が割り当てられている不整合が {len(inconsistent_codes)} 件あります。")
    display(inconsistent_master.sort_values('勘定科目コード'))
else:
    print("✅ マスタ内の名称はすべて統一されています。")

In [ ]:
import pandas as pd

# 1. 作成ログデータ
# UserC が短時間（17:05台）に連続して作成している想定
log_data = {
    '伝票番号': ['V201', 'V202', 'V203', 'V204', 'V205', 'V206', 'V207'],
    '作成日時': [
        '2024-04-10 17:00:00',
        '2024-04-10 17:01:00',
        '2024-04-10 17:05:00', # UserC バースト開始
        '2024-04-10 17:05:10', # UserC
        '2024-04-10 17:05:20', # UserC
        '2024-04-10 17:05:30', # UserC
        '2024-04-10 17:05:40'  # UserC
    ],
    '作成者': ['UserA', 'UserB', 'UserC', 'UserC', 'UserC', 'UserC', 'UserC']
}
df_logs = pd.DataFrame(log_data)
df_logs['作成日時'] = pd.to_datetime(df_logs['作成日時'])

# 2. ユーザーごとに1分間のウィンドウで件数をカウントする
# ※ '1min' という指定で1分間の幅を見ます
df_logs = df_logs.sort_values('作成日時') # 時間順に並べ替え
df_logs = df_logs.set_index('作成日時')    # 時間をインデックス（索引）にする

# ユーザーごとにグループ化し、1分間（Rolling 1min）の件数を算出
burst_check = df_logs.groupby('作成者')['伝票番号'].rolling('1min').count()
burst_check = burst_check.reset_index()
burst_check.columns = ['作成者', '作成日時', '1分間内の件数']

# 3. 異常判定（1分間に5件以上を異常とする）
alert_threshold = 5
alerts = burst_check[burst_check['1分間内の件数'] >= alert_threshold]

print("--- 短時間大量操作（バースト）検知レポート ---")
if len(alerts) > 0:
    print(f"⚠️ 警告: 短時間に大量の操作を行ったユーザーが {len(alerts['作成者'].unique())} 名見つかりました。")
    display(alerts)
else:
    print("✅ 全てのユーザーの操作頻度は正常範囲内です。")

import matplotlib.pyplot as plt

# 4. ユーザーごとの「最大同時操作数」を抽出
# 各作成者の中で、最も高かった「1分間内の件数」を取得します
user_max_burst = burst_check.groupby('作成者')['1分間内の件数'].max().reset_index()

# 5. データの準備（英語表記に変換してグローバル対応）
# 日本語が含まれる場合は英語にマップします
user_labels = user_max_burst['作成者']
max_values = user_max_burst['1分間内の件数']

# 6. グラフの作成
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(user_labels, max_values, color=['skyblue', 'lightgreen', 'tomato'])

# 7. タイトルとラベルの設定（英語）
ax.set_title('Maximum Transactions per Minute by User', fontsize=14)
ax.set_xlabel('User Name')
ax.set_ylabel('Max Transactions (per min)')
ax.set_ylim(0, max(max_values) + 2) # 上部に少し余白を作る

# 8. 数値ラベルを棒の上に表示
ax.bar_label(bars, padding=3, fmt='%.0f')

# 異常値（しきい値5）に水平線を引く（エスカレーションエンジニアらしい演出）
ax.axhline(y=5, color='red', linestyle='--', label='Alert Threshold (5)')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

# 1. 予算データ (Budget)
budget_data = {
    '部門': ['営業部', 'マーケティング部', '開発部', '人事部', '総務部'],
    '予算額': [5000000, 3000000, 8000000, 1500000, 1000000]
}
df_budget = pd.DataFrame(budget_data)

# 2. 実績データ (Actual)
# ※ 開発部が予算超過、人事部が極端に未消化の想定
actual_data = {
    '部門': ['営業部', 'マーケティング部', '開発部', '人事部', '総務部'],
    '実績額': [4800000, 3100000, 9500000, 600000, 1050000]
}
df_actual = pd.DataFrame(actual_data)

# 3. 予実比較の統合
df_vansa = pd.merge(df_budget, df_actual, on='部門')

# 4. 差異と消化率の計算
df_vansa['差異'] = df_vansa['予算額'] - df_vansa['実績額']
df_vansa['消化率(%)'] = (df_vansa['実績額'] / df_vansa['予算額'] * 100).round(1)

# 5. 異常値（Alert）の判定ロジック
# 消化率が110%以上、または50%以下を異常とする
def judge_alert(percent):
    if percent >= 110:
        return "⚠️ 超過(Over)"
    elif percent <= 50:
        return "ℹ️ 未消化(Under)"
    else:
        return "✅ 正常"

df_vansa['ステータス'] = df_vansa['消化率(%)'].apply(judge_alert)

print("--- 予算対実績 差異分析レポート ---")
display(df_vansa)

# 6. アラート対象のみ抽出
alerts = df_vansa[df_vansa['ステータス'] != "✅ 正常"]
print("\n--- 要確認部門リスト ---")
display(alerts)

import matplotlib.pyplot as plt
import numpy as np

# 日本語フォントの設定（Colabで日本語を表示するための簡易設定）
plt.rcParams['font.family'] = 'sans-serif'

# 7. データの準備（英語にマッピング）
# 日本語の部門名を英語に変換して表示用にする
dept_map = {
    '営業部': 'Sales',
    'マーケティング部': 'Marketing',
    '開発部': 'R&D',
    '人事部': 'HR',
    '総務部': 'Admin'
}
labels = df_vansa['部門'].map(dept_map)
budget = df_vansa['予算額']
actual = df_vansa['実績額']

x = np.arange(len(labels))
width = 0.35

# 8. グラフの作成
fig, ax = plt.subplots(figsize=(10, 6))
rects1 = ax.bar(x - width/2, budget, width, label='Budget', color='skyblue')
rects2 = ax.bar(x + width/2, actual, width, label='Actual', color='salmon')

# 8. ラベルやタイトルの設定（英語）
ax.set_ylabel('Amount (JPY)')
ax.set_title('Budget vs Actual Comparison by Department')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()

# 9. 棒の上に数値を表示する（オプション）
ax.bar_label(rects1, padding=3, fmt='{:,.0f}')
ax.bar_label(rects2, padding=3, fmt='{:,.0f}')

fig.tight_layout()
plt.show()

In [ ]:
import pandas as pd

# 1. 大量の仕訳データ（模擬データ）
# ※ V003 と V005 が、日付・科目・金額がすべて同じ「重複」の疑いがあるデータ
journal_data = {
    '伝票番号': ['V001', 'V002', 'V003', 'V004', 'V005'],
    '取引日': ['2024-04-01', '2024-04-02', '2024-04-03', '2024-04-04', '2024-04-03'],
    '勘定科目': ['交際費', '消耗品費', '支払手数料', '通信費', '支払手数料'],
    '金額': [5000, 2000, 15000, 3000, 15000],
    '作成者': ['UserA', 'UserB', 'UserA', 'UserC', 'UserA']
}
df_journal = pd.DataFrame(journal_data)

# 2. 重複チェックロジック
# 「取引日」「勘定科目」「金額」の3つが一致するものを探す
# keep=False を指定すると、重複している行すべてを抽出できる
duplicates = df_journal[df_journal.duplicated(subset=['取引日', '勘定科目', '金額'], keep=False)]

print("--- 重複仕訳（二重計上）の疑いがあるデータ ---")
if len(duplicates) > 0:
    print(f"⚠️ 警告: 重複の可能性がある仕訳が {len(duplicates)} 件見つかりました。")
    display(duplicates.sort_values(by=['取引日', '金額']))
else:
    print("✅ 重複した仕訳は見つかりませんでした。")

# 3. (応用) 重複の「理由」を推測するヒント
# 同じ作成者が短時間に操作した可能性などを分析

In [ ]:
import pandas as pd

# 1. 作成日時を含む仕訳データ
audit_data = {
    '伝票番号': ['V101', 'V102', 'V103', 'V104', 'V105'],
    '作成日時': [
        '2024-04-01 10:00:00', # 月曜 10時 (正常)
        '2024-04-06 14:00:00', # 土曜 14時 (休日)
        '2024-04-03 23:30:00', # 水曜 23時半 (深夜)
        '2024-04-04 09:15:00', # 木曜 09時 (正常)
        '2024-04-07 02:00:00'  # 日曜 02時 (休日かつ深夜)
    ],
    '金額': [50000, 120000, 3000, 45000, 800000],
    '作成者': ['UserA', 'UserB', 'UserC', 'UserA', 'UserB']
}
df_audit = pd.DataFrame(audit_data)

# 2. 文字列を日付型（datetime）に変換
df_audit['作成日時'] = pd.to_datetime(df_audit['作成日時'])

# 3. 曜日（0=月, 6=日）と時間（0-23）を抽出
df_audit['曜日'] = df_audit['作成日時'].dt.weekday
df_audit['時間'] = df_audit['作成日時'].dt.hour

# 4. 異常判定ロジック
# 条件1: 曜日が土日 (5 または 6)
# 条件2: 時間が 22時以降 または 5時未満
is_weekend = df_audit['曜日'] >= 5
is_late_night = (df_audit['時間'] >= 22) | (df_audit['時間'] < 5)

# いずれかの条件に当てはまるものを抽出
suspicious_logs = df_audit[is_weekend | is_late_night].copy()

# 表示用に曜日の名前を追加
weekday_names = {0: 'Mon', 1: 'Tue', 2: 'Wed', 3: 'Thu', 4: 'Fri', 5: 'Sat', 6: 'Sun'}
suspicious_logs['曜日名'] = suspicious_logs['曜日'].map(weekday_names)

print("--- 不自然な時間帯・曜日の仕訳ログ ---")
if len(suspicious_logs) > 0:
    print(f"⚠️ 警告: 内部統制上の確認が必要な仕訳が {len(suspicious_logs)} 件あります。")
    display(suspicious_logs[['伝票番号', '作成日時', '曜日名', '時間', '作成者', '金額']])
else:
    print("✅ すべての仕訳は標準的な営業時間内に作成されています。")

In [ ]:
import pandas as pd

# 1. 各法人の個別試算表（TB）データ
# 1. 親会社のデータ
parent_data = {
    '勘定科目': ['売上高', '売上高', '売上原価', '一般管理費'],
    '金額': [-4000000, -1000000, 3000000, 1000000], # 子会社分を-100万にする
    '相手先': ['外部企業', '子会社A', '外部企業', '外部企業']
}
df_parent = pd.DataFrame(parent_data)

# 子会社A
sub_data = {
    '勘定科目': ['売上高', '売上原価', '一般管理費', '買掛金'],
    '金額': [-2000000, 1000000, 500000, -1500000],
    '相手先': ['外部企業', '親会社', '外部企業', '親会社']
}

df_parent = pd.DataFrame(parent_data)
df_sub = pd.DataFrame(sub_data)

# 2. 単純合算（連結の第一歩）
df_combined = pd.concat([df_parent, df_sub]).groupby('勘定科目')['金額'].sum().reset_index()
df_combined.rename(columns={'金額': '合算金額'}, inplace=True)

# 3. 内部取引消去ロジック
# 親会社の「売上（対子会社）」と子会社の「売上原価（対親会社）」を特定して消去
# elimination_amount = 1000000 # 相殺額
# ① データを手入力せず、親会社の「対子会社売上」から自動計算する
intercompany_sales = df_parent[
    (df_parent['勘定科目'] == '売上高') & (df_parent['相手先'] == '子会社A')
]['金額'].sum()

# 売上はマイナス表記なので、絶対値（abs）にして消去額とする
elimination_amount = abs(intercompany_sales)

print(f"自動抽出された相殺額: {elimination_amount:,}円")

# 消去仕訳のデータフレーム作成
elim_data = [
    {'勘定科目': '売上高', '消去額': elimination_amount},     # 売上のマイナス（貸方）を消すためにプラス
    {'勘定科目': '売上原価', '消去額': -elimination_amount} # 原価のプラス（借方）を消すためにマイナス
]
df_elim = pd.DataFrame(elim_data)

# 4. 最終連結財務諸表の作成
df_final = pd.merge(df_combined, df_elim, on='勘定科目', how='left').fillna(0)
df_final['連結金額'] = df_final['合算金額'] + df_final['消去額']

print("--- 連結決算ワークシート ---")
display(df_final[['勘定科目', '合算金額', '消去額', '連結金額']])

In [ ]:
import pandas as pd

# 1. 取引データ（請求時と支払時のデータ）
# ※ 伝票 FX003 は、入力された損益額が間違っている想定（計算チェック用）
exchange_data = {
    '取引ID': ['FX001', 'FX002', 'FX003'],
    '外貨額_USD': [100.0, 250.0, 150.0],
    '請求時レート': [150.0, 151.0, 149.0],
    '支払時レート': [155.0, 148.0, 152.0],
    '入力済為替損益': [-500, 750, -100] # FX003は本来 -450のはず
}
df_fx = pd.DataFrame(exchange_data)

# 2. 為替差損益の計算ロジック
# 計算式: 外貨額 * (請求時レート - 支払時レート)
# ※ 正数なら「益」、負数なら「損」
df_fx['理論上の為替損益'] = (
    df_fx['外貨額_USD'] * (df_fx['請求時レート'] - df_fx['支払時レート'])
).astype(int)

# 3. 誤差の検知
df_fx['計算誤差'] = df_fx['入力済為替損益'] - df_fx['理論上の為替損益']
error_fx = df_fx[df_fx['計算誤差'] != 0]

print("--- 為替差損益（実現損益）検証レポート ---")
if len(error_fx) > 0:
    print(f"⚠️ 警告: 為替差損益の計算不整合が {len(error_fx)} 件あります。")
    display(error_fx[['取引ID', '外貨額_USD', '理論上の為替損益', '入力済為替損益', '計算誤差']])
else:
    print("✅ すべての為替差損益が正しく計上されています。")

In [ ]:
import pandas as pd

# 1. 判定用の関数を定義
def judge_gain_loss(amount):
    if amount > 0:
        return "為替差益"
    elif amount < 0:
        return "為替差損"
    else:
        return "差損益なし"

# 2. 取引データ（請求時と支払時のデータ）
# ※ 伝票 FX003 は、入力された損益額が間違っている想定（計算チェック用）
exchange_data = {
    '取引ID': ['FX001', 'FX002', 'FX003'],
    '外貨額_USD': [100.0, 250.0, 150.0],
    '請求時レート': [150.0, 151.0, 149.0],
    '支払時レート': [155.0, 148.0, 152.0],
    '入力済為替損益': [-500, 750, -100] # FX003は本来 -450のはず
}
df_fx = pd.DataFrame(exchange_data)

# 2. 為替差損益の計算ロジック
# 計算式: 外貨額 * (請求時レート - 支払時レート)
# ※ 正数なら「益」、負数なら「損」
df_fx['理論上の為替損益'] = (
    df_fx['外貨額_USD'] * (df_fx['請求時レート'] - df_fx['支払時レート'])
).astype(int)

# 3. 誤差の検知
df_fx['計算誤差'] = df_fx['入力済為替損益'] - df_fx['理論上の為替損益']
error_fx = df_fx[df_fx['計算誤差'] != 0]

print("--- 為替差損益（実現損益）検証レポート ---")
if len(error_fx) > 0:
    print(f"⚠️ 警告: 為替差損益の計算不整合が {len(error_fx)} 件あります。")
    display(error_fx[['取引ID', '外貨額_USD', '理論上の為替損益', '入力済為替損益', '計算誤差']])
else:
    print("✅ すべての為替差損益が正しく計上されています。")

# 4. apply関数を使って「理論上の為替損益」列から判定結果を生成
df_fx['判定ラベル'] = df_fx['理論上の為替損益'].apply(judge_gain_loss)

# 5. 結果の表示
print("--- 為替差損益 判定レポート ---")
display(df_fx[['取引ID', '外貨額_USD', '理論上の為替損益', '判定ラベル']])

# 6. (応用) 為替差損（マイナス）のデータだけを抽出して合計を出す
total_loss = df_fx[df_fx['判定ラベル'] == "為替差損"]['理論上の為替損益'].sum()
print(f"\n今回の総為替差損額: {abs(total_loss):,}円")

In [ ]:
import pandas as pd

# 1. 判定用の関数を定義
def judge_gain_loss(amount):
    if amount > 0:
        return "為替差益"
    elif amount < 0:
        return "為替差損"
    else:
        return "差損益なし"

# 2. 取引データ（請求時と支払時のデータ）
# ※ 伝票 FX003 は、入力された損益額が間違っている想定（計算チェック用）
exchange_data = {
    '取引ID': ['FX001', 'FX002', 'FX003'],
    '外貨額_USD': [100.0, 250.0, 150.0],
    '請求時レート': [150.0, 151.0, 149.0],
    '支払時レート': [155.0, 148.0, 152.0],
    '入力済為替損益': [-500, 750, -100] # FX003は本来 -450のはず
}
df_fx = pd.DataFrame(exchange_data)

# 2. 為替差損益の計算ロジック
# 計算式: 外貨額 * (請求時レート - 支払時レート)
# ※ 正数なら「益」、負数なら「損」
df_fx['理論上の為替損益'] = (
    df_fx['外貨額_USD'] * (df_fx['請求時レート'] - df_fx['支払時レート'])
).astype(int)

# 3. 誤差の検知
df_fx['計算誤差'] = df_fx['入力済為替損益'] - df_fx['理論上の為替損益']
error_fx = df_fx[df_fx['計算誤差'] != 0]

print("--- 為替差損益（実現損益）検証レポート ---")
if len(error_fx) > 0:
    print(f"⚠️ 警告: 為替差損益の計算不整合が {len(error_fx)} 件あります。")
    display(error_fx[['取引ID', '外貨額_USD', '理論上の為替損益', '入力済為替損益', '計算誤差']])
else:
    print("✅ すべての為替差損益が正しく計上されています。")

# 4. apply関数を使って「理論上の為替損益」列から判定結果を生成
df_fx['判定ラベル'] = df_fx['理論上の為替損益'].apply(judge_gain_loss)

# 5. 結果の表示
print(f"--- 最終サマリー ---")
print(f"\n今回の総為替差損額: {abs(total_loss):,}円")

# 6.全データにラベルが付いた状態で一覧表示
display(df_fx[['取引ID', '外貨額_USD', '理論上の為替損益', '判定ラベル', '計算誤差']])

In [ ]:
import pandas as pd

# 1. 銀行明細データ（4/2に入金）
bank_data = {
    '銀行日': pd.to_datetime(['2024-04-02', '2024-04-10']),
    '金額': [50000, 30000],
    '銀行ID': ['B001', 'B002']
}
df_bank = pd.DataFrame(bank_data)

# 2. D365元帳データ（自社では4/1に計上：1日早い）
ledger_data = {
    '元帳日': pd.to_datetime(['2024-04-01', '2024-04-15']),
    '金額': [50000, 30000],
    '元帳ID': ['L001', 'L002']
}
df_ledger = pd.DataFrame(ledger_data)

# 3. 1日のズレを許容して照合するロジック
results = []

for b_idx, b_row in df_bank.iterrows():
    # 金額が同じものを抽出
    potential_matches = df_ledger[df_ledger['金額'] == b_row['金額']]

    matched = False
    for l_idx, l_row in potential_matches.iterrows():
        # 日付の差分を計算（絶対値で1日以内か）
        date_diff = abs((b_row['銀行日'] - l_row['元帳日']).days)

        if date_diff <= 1:
            results.append({
                '銀行ID': b_row['銀行ID'],
                '元帳ID': l_row['元帳ID'],
                '金額': b_row['金額'],
                '日付差': f"{date_diff}日",
                'ステータス': '照合成功'
            })
            matched = True
            break # 1つ見つかったら次の銀行データへ

    if not matched:
        results.append({'銀行ID': b_row['銀行ID'], '金額': b_row['金額'], 'ステータス': '未照合'})

df_final = pd.DataFrame(results)

print("--- 銀行勘定調整（1日のズレ許容）レポート ---")
display(df_final)

In [ ]:
import pandas as pd
from decimal import Decimal, ROUND_HALF_UP

def calculate_tax(amount, rate, method=ROUND_HALF_UP):
    """
    正確な端数処理（四捨五入）を行う税額計算関数
    """
    tax = Decimal(str(amount)) * Decimal(str(rate))
    return int(tax.to_integral_value(rounding=method))

# 1. 検証対象の請求書データ（入力済みデータ）
# ※ 3行目は税額が間違っている（1,000円のはずが1,100円になっている）想定
invoice_data = {
    '請求書No': ['INV001', 'INV002', 'INV003', 'INV004'],
    '税抜金額': [10000, 20000, 10000, 5555],
    '適用税率': [0.10, 0.08, 0.10, 0.10],
    '入力済税額': [1000, 1600, 1100, 556]
}
df_inv = pd.DataFrame(invoice_data)

# 2. Pythonロジックで「正しい税額」を再計算
df_inv['理論上の税額'] = df_inv.apply(
    lambda x: calculate_tax(x['税抜金額'], x['適用税率']), axis=1
)

# 3. 誤差（不整合）をチェック
df_inv['誤差'] = df_inv['入力済税額'] - df_inv['理論上の税額']
discrepancy_df = df_inv[df_inv['誤差'] != 0]

print("--- 請求書税額検証レポート ---")
if len(discrepancy_df) > 0:
    print(f"⚠️ 警告: 税額の不整合が {len(discrepancy_df)} 件見つかりました。")
    display(discrepancy_df)
else:
    print("✅ すべての税額計算が適正です。")

In [ ]:
import pandas as pd
from datetime import datetime

# 1. 改正基準日の設定（この日以降は 8%）
REVISION_DATE = datetime(2024, 10, 1)
OLD_RATE = 0.10
NEW_RATE = 0.08

# 2. 検証対象の取引データ（4行目は「10/1以降なのに10%」で計算されているミスを想定）
tx_data = {
    '伝票No': ['V001', 'V002', 'V003', 'V004'],
    '取引日': ['2024-09-25', '2024-09-30', '2024-10-01', '2024-10-05'],
    '税抜金額': [10000, 20000, 30000, 40000],
    '適用された税率': [0.10, 0.10, 0.08, 0.10]  # V004が設定ミス
}
df_tx = pd.DataFrame(tx_data)
df_tx['取引日'] = pd.to_datetime(df_tx['取引日'])

# 3. 理論上（法律上）あるべき税率を判定する関数
def get_expected_rate(row_date):
    return NEW_RATE if row_date >= REVISION_DATE else OLD_RATE

# 4. 検証ロジックの実行
df_tx['本来の税率'] = df_tx['取引日'].apply(get_expected_rate)
df_tx['税率不整合'] = df_tx['適用された税率'] != df_tx['本来の税率']

# 5. 不整合データの抽出
error_tx = df_tx[df_tx['税率不整合'] == True]

print(f"--- 税率改正（基準日: {REVISION_DATE.date()}）対応検証結果 ---")
if len(error_tx) > 0:
    print(f"⚠️ 警告: 適用日の判定ミスと思われるデータが {len(error_tx)} 件あります。")
    display(error_tx[['伝票No', '取引日', '適用された税率', '本来の税率']])
else:
    print("✅ すべての取引で正しい税率が適用されています。")

In [ ]:
import pandas as pd

# 1. 商品マスターの設定（品目ごとの正しい税率）
item_master = {
    '品目': ['りんご', 'お茶', '洗剤', 'ノート'],
    '正しい税率': [0.08, 0.08, 0.10, 0.10]
}
df_master = pd.DataFrame(item_master)

# 2. 実際に届いた販売伝票データ
# ※「お茶」が 0.10 で計算されているミスを想定
sales_data = {
    '伝票No': ['S001', 'S002', 'S003', 'S004'],
    '品目': ['りんご', 'お茶', '洗剤', 'ノート'],
    '適用税率': [0.08, 0.10, 0.10, 0.10]  # S002がミス
}
df_sales = pd.DataFrame(sales_data)

# 3. マスターデータを結合（Merge）して、本来の税率を紐付ける
df_check = pd.merge(df_sales, df_master, on='品目', how='left')

# 4. 不整合（Mismatch）の検知
# ここでも「mismatch」という名前を自分で決めて使います
mismatch_data = df_check[df_check['適用税率'] != df_check['正しい税率']]

print("--- 軽減税率・標準税率 適用チェック ---")
if len(mismatch_data) > 0:
    print(f"⚠️ 警告: 品目マスターと異なる税率が適用されている伝票が {len(mismatch_data)} 件あります。")
    display(mismatch_data[['伝票No', '品目', '適用税率', '正しい税率']])
else:
    print("✅ すべての品目に対して正しい税率が適用されています。")

In [ ]:
import pandas as pd

# 1. 銀行からの明細データ（外部データ）
bank_statement_data = {
    '取引日': ['2024-04-01', '2024-04-05', '2024-04-10', '2024-04-15'],
    '金額': [50000, 120000, 'abcd', 45000],
    '銀行参照番号': ['BK001', 'BK002', 'BK003', 'BK004']
}
df_bank = pd.DataFrame(bank_statement_data)

# 2. D365上の元帳データ（内部データ）
# 「金額に数字ではない文字」が入っている場合は警告を出す処理
# ※4/10の3000円が未入力、逆にD365には20000円の未達仕訳がある想定
d365_ledger_data = {
    '仕訳日': ['2024-04-01', '2024-04-05', '2024-04-20'],
    '金額': [50000, 120000, 20000],
    '伝票番号': ['V-001', 'V-002', 'V-003']
}
df_ledger = pd.DataFrame(d365_ledger_data)

# --- 照合ロジック (Inner Joinを利用) ---
# 「日付」と「金額」が一致するものを抽出
matched_df = pd.merge(
    df_bank,
    df_ledger,
    left_on=['取引日', '金額'],
    right_on=['仕訳日', '金額'],
    how='inner'
)

# --- エラーハンドリング処理 ---
# 金額を数値に変換。変換できない文字は強制的に「NaN（欠損値）」にする
df_bank['金額'] = pd.to_numeric(df_bank['金額'], errors='coerce')

# NaN（数字じゃなかった行）を探して警告を出す
invalid_rows = df_bank[df_bank['金額'].isnull()]

print("--- データ不正チェック ---")
if len(invalid_rows) > 0:
    print(f"⚠️ 警告: 数字ではないデータが {len(invalid_rows)} 件見つかりました。")
    display(invalid_rows)
else:
    print("✅ 全ての金額データが正常な数値です。")

# --- 未照合（不一致）の抽出 ---
# 銀行にはあるが、D365にないもの
unmatched_bank = df_bank[~df_bank['銀行参照番号'].isin(matched_df['銀行参照番号'])]

print("✅ 照合成功（日付と金額が一致）:")
display(matched_df[['取引日', '金額', '銀行参照番号', '伝票番号']])
display(unmatched_bank)

In [ ]:
import pandas as pd

def calculate_straight_line_depreciation(acquisition_cost, salvage_value, useful_life_years):
    """
    定額法の減価償却スケジュールを計算する関数
    """
    # 年間の償却額 = (取得価額 - 残存価額) / 耐用年数
    annual_depreciation = (acquisition_cost - salvage_value) / useful_life_years

    schedule = []
    current_book_value = acquisition_cost

    for year in range(1, useful_life_years + 1):
        # 最終年の調整（残存価額を下回らないようにする）
        if year == useful_life_years:
            depreciation_amount = current_book_value - salvage_value
        else:
            depreciation_amount = annual_depreciation

        current_book_value -= depreciation_amount

        schedule.append({
            "年": year,
            "期首帳簿価額": round(current_book_value + depreciation_amount),
            "償却額": round(depreciation_amount),
            "期末帳簿価額": round(current_book_value)
        })

    return pd.DataFrame(schedule)

# --- テスト用の設定値 ---
cost = 1200000    # 取得価額（120万円）
salvage = 10000   # 残存価額（1万円）
life = 5          # 耐用年数（5年）

# 実行
depreciation_df = calculate_straight_line_depreciation(cost, salvage, life)

print(f"取得価額: {cost:,}円 / 耐用年数: {life}年 の減価償却スケジュール")
display(depreciation_df)

# Excelとして保存したい場合（任意）
# depreciation_df.to_excel("depreciation_schedule.xlsx", index=False)

In [ ]:
import pandas as pd

# 1. 為替レートマスタ（日付ごとの1ドルの円相場）
exchange_rates = {
    '日付': ['2024-04-01', '2024-04-02', '2024-04-03'],
    'レート': [151.20, 150.85, 151.50]
}
df_rates = pd.DataFrame(exchange_rates)

# 2. 外貨建て取引データ（USD）
# ※ 2行目は「150.85」で計算すべきところが間違っている想定
foreign_tx = {
    '伝票No': ['FX001', 'FX002', 'FX003'],
    '日付': ['2024-04-01', '2024-04-02', '2024-04-03'],
    '外貨額_USD': [100.00, 200.00, 150.00],
    '入力済邦貨額_JPY': [15120, 30100, 22725]
}
df_tx = pd.DataFrame(foreign_tx)

# 3. レートを紐付けて理論上の邦貨額を計算
df_val = pd.merge(df_tx, df_rates, on='日付', how='left')
df_val['理論上の邦貨額'] = (df_val['外貨額_USD'] * df_val['レート']).round(0).astype(int)

# 4. 誤差の抽出
df_val['換算誤差'] = df_val['入力済邦貨額_JPY'] - df_val['理論上の邦貨額']
fx_error = df_val[df_val['換算誤差'] != 0]

print("--- 外貨換算（USD -> JPY）整合性チェック ---")
if len(fx_error) > 0:
    print(f"⚠️ 警告: 為替換算に誤りがある可能性が高い伝票が {len(fx_error)} 件あります。")
    display(fx_error[['伝票No', '日付', '外貨額_USD', 'レート', '理論上の邦貨額', '入力済邦貨額_JPY']])
else:
    print("✅ すべての外貨取引が適切なレートで換算されています。")

In [ ]:
import pandas as pd
import math

def calculate_monthly_depreciation(acquisition_cost, salvage_value, useful_life_years):
    """
    定額法に基づき、月次の償却スケジュールを生成する関数
    """
    # 総償却月数
    total_months = useful_life_years * 12
    # 総償却対象額
    total_depreciable_amount = acquisition_cost - salvage_value
    # 月次の標準償却額（端数は切り捨て、最終月で調整）
    monthly_depreciation = math.floor(total_depreciable_amount / total_months)

    schedule = []
    accumulated_depreciation = 0 # 累計償却額

    for month in range(1, total_months + 1):
        # 最終月の判定（残りの償却対象額をすべて計上して端数を調整）
        if month == total_months:
            depreciation_amount = total_depreciable_amount - accumulated_depreciation
        else:
            depreciation_amount = monthly_depreciation

        accumulated_depreciation += depreciation_amount
        current_book_value = acquisition_cost - accumulated_depreciation

        schedule.append({
            "月": month,
            "当月償却額": int(depreciation_amount),
            "累計償却額": int(accumulated_depreciation),
            "期末帳簿価額": int(current_book_value)
        })

    return pd.DataFrame(schedule)

# --- テスト実行 ---
cost = 1200000
salvage = 1
years = 5

monthly_df = calculate_monthly_depreciation(cost, salvage, years)

print(f"【月次】減価償却シミュレーション ({years}年 = {years*12}ヶ月)")
# 全件表示すると長いので、最初と最後だけ表示
display(monthly_df.head(12)) # 1年目
print("...")
display(monthly_df.tail(3))  # 最終月付近

In [ ]:
import pandas as pd

# 1. 銀行からの明細データ（外部データ）
bank_statement_data = {
    '取引日': ['2024-04-01', '2024-04-05', '2024-04-10', '2024-04-15'],
    '金額': [50000, 120000, 3000, 45000],
    '銀行参照番号': ['BK001', 'BK002', 'BK003', 'BK004']
}
df_bank = pd.DataFrame(bank_statement_data)

# 2. D365上の元帳データ（内部データ）
# ※4/10の3000円が未入力、逆にD365には20000円の未達仕訳がある想定
d365_ledger_data = {
    '仕訳日': ['2024-04-01', '2024-04-05', '2024-04-20'],
    '金額': [50000, 120000, 20000],
    '伝票番号': ['V-001', 'V-002', 'V-003']
}
df_ledger = pd.DataFrame(d365_ledger_data)

# --- 照合ロジック (Inner Joinを利用) ---
# 「日付」と「金額」が一致するものを抽出
matched_df = pd.merge(
    df_bank,
    df_ledger,
    left_on=['取引日', '金額'],
    right_on=['仕訳日', '金額'],
    how='inner'
)

# --- 未照合（不一致）の抽出 ---
# 銀行にはあるが、D365にないもの
unmatched_bank = df_bank[~df_bank['銀行参照番号'].isin(matched_df['銀行参照番号'])]

print("✅ 照合成功（日付と金額が一致）:")
display(matched_df[['取引日', '金額', '銀行参照番号', '伝票番号']])

print("\n⚠️ 銀行明細のみに存在（D365未入力の可能性）:")
display(unmatched_bank)

In [ ]:
import pandas as pd

# 1. 銀行からの明細データ（外部データ）
bank_statement_data = {
    '取引日': ['2024-04-01', '2024-04-05', '2024-04-10', '2024-04-15'],
    '金額': [50000, 120000, 3000, 45000],
    '銀行参照番号': ['BK001', 'BK002', 'BK003', 'BK004']
}
df_bank = pd.DataFrame(bank_statement_data)

# 2. D365上の元帳データ（内部データ）
# 「1日のズレ」を許容する照合
# ※4/10の3000円が未入力、逆にD365には20000円の未達仕訳がある想定
d365_ledger_data = {
    '仕訳日': ['2024-04-01', '2024-04-06', '2024-04-20'],
    '金額': [50000, 120000, 20000],
    '伝票番号': ['V-001', 'V-002', 'V-003']
}
df_ledger = pd.DataFrame(d365_ledger_data)

# --- 照合ロジック (Inner Joinを利用) ---
# 「日付」と「金額」が一致するものを抽出
matched_df = pd.merge(
    df_bank,
    df_ledger,
    left_on=['金額'],
    right_on=['金額'],
    how='inner'
)

# --- 未照合（不一致）の抽出 ---
# 銀行にはあるが、D365にないもの
unmatched_bank = df_bank[~df_bank['銀行参照番号'].isin(matched_df['銀行参照番号'])]

print("✅ 照合成功（日付と金額が一致）:")
display(matched_df[['取引日', '金額', '銀行参照番号', '伝票番号']])

print("\n⚠️ 銀行明細のみに存在（D365未入力の可能性）:")
display(unmatched_bank)

In [ ]:
import pandas as pd

# 1. 銀行からの明細データ（外部データ）
bank_statement_data = {
    '取引日': ['2024-04-01', '2024-04-05', '2024-04-10', '2024-04-15'],
    '金額': [50000, 120000, 3000, 45000],
    '銀行参照番号': ['BK001', 'BK002', 'BK003', 'BK004']
}
df_bank = pd.DataFrame(bank_statement_data)

# 2. D365上の元帳データ（内部データ）
# 「1日のズレ」を許容する照合
# ※4/10の3000円が未入力、逆にD365には20000円の未達仕訳がある想定
d365_ledger_data = {
    '仕訳日': ['2024-04-01', '2024-04-06', '2024-04-20'],
    '金額': [50000, 120000, 20000],
    '伝票番号': ['V-001', 'V-002', 'V-003']
}
df_ledger = pd.DataFrame(d365_ledger_data)

# --- 照合ロジック (Inner Joinを利用) ---
# 「日付」と「金額」が一致するものを抽出
matched_df = pd.merge(
    df_bank,
    df_ledger,
    left_on=['金額'],
    right_on=['金額'],
    how='inner'
)

# --- 未照合（不一致）の抽出 ---
# 銀行にはあるが、D365にないもの
unmatched_bank = df_bank[~df_bank['銀行参照番号'].isin(matched_df['銀行参照番号'])]

print("✅ 照合成功（日付と金額が一致）:")
display(matched_df[['取引日', '金額', '銀行参照番号', '伝票番号']])

print("\n⚠️ 銀行明細のみに存在（D365未入力の可能性）:")
display(unmatched_bank)


import pandas as pd

# 1. 銀行からの明細データ（外部データ）
bank_statement_data = {
    '取引日': ['2024-04-01', '2024-04-05', '2024-04-10', '2024-04-15'],
    '金額': [50000, 120000, abcd, 45000],
    '銀行参照番号': ['BK001', 'BK002', 'BK003', 'BK004']
}
df_bank = pd.DataFrame(bank_statement_data)

# 2. D365上の元帳データ（内部データ）
# 「金額に数字ではない文字」が入っている場合は警告を出す処理
# ※4/10の3000円が未入力、逆にD365には20000円の未達仕訳がある想定
d365_ledger_data = {
    '仕訳日': ['2024-04-01', '2024-04-05', '2024-04-20'],
    '金額': [50000, 120000, 20000],
    '伝票番号': ['V-001', 'V-002', 'V-003']
}
df_ledger = pd.DataFrame(d365_ledger_data)

# --- 照合ロジック (Inner Joinを利用) ---
# 「日付」と「金額」が一致するものを抽出
matched_df = pd.merge(
    df_bank,
    df_ledger,
    left_on=['取引日', '金額'],
    right_on=['仕訳日', '金額'],
    how='inner'
)

# --- エラーハンドリング処理 ---
# 金額を数値に変換。変換できない文字は強制的に「NaN（欠損値）」にする
df_bank['金額'] = pd.to_numeric(df_bank['金額'], errors='coerce')

# NaN（数字じゃなかった行）を探して警告を出す
invalid_rows = df_bank[df_bank['金額'].isnull()]

print("--- データ不正チェック ---")
if len(invalid_rows) > 0:
    print(f"⚠️ 警告: 数字ではないデータが {len(invalid_rows)} 件見つかりました。")
    display(invalid_rows)
else:
    print("✅ 全ての金額データが正常な数値です。")

# --- 未照合（不一致）の抽出 ---
# 銀行にはあるが、D365にないもの
unmatched_bank = df_bank[~df_bank['銀行参照番号'].isin(matched_df['銀行参照番号'])]

print("✅ 照合成功（日付と金額が一致）:")
display(matched_df[['取引日', '金額', '銀行参照番号', '伝票番号']])
display(unmatched_bank)

print("\n⚠️ 銀行明細のみに存在（D365未入力の可能性）:")
display(unmatched_bank)

In [ ]:
# テストデータ（金額に「あいうえお」という不正な値が混じっている想定）
data = {
    '取引日': ['2024-04-01', '2024-04-02', '2024-04-03'],
    '金額': [50000, 'あいうえお', 120000] # 文字列が混入
}
df_bank = pd.DataFrame(data)

# --- エラーハンドリングの核心 ---
# 金額を数値に変換。変換できない文字は強制的に「NaN（欠損値）」にする
df_bank['金額'] = pd.to_numeric(df_bank['金額'], errors='coerce')

# NaN（数字じゃなかった行）を探して警告を出す
invalid_rows = df_bank[df_bank['金額'].isnull()]

print("--- データ不正チェック ---")
if len(invalid_rows) > 0:
    print(f"⚠️ 警告: 数字ではないデータが {len(invalid_rows)} 件見つかりました。")
    display(invalid_rows)
else:
    print("✅ 全ての金額データが正常な数値です。")

# 正常なデータだけで処理を続ける
df_valid = df_bank.dropna(subset=['金額'])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

print("Current working directory files:")
for dirname, _, filenames in os.walk('.'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import pandas as pd

# Excelファイルを読み込む
df = pd.read_excel('test_data_2.xlsx')

# 現在読み込んでいるデータの列名をすべて表示する
print("--- 現在のExcelの列名一覧 ---")
print(df.columns.tolist())

# 列のデータ型や欠損値もまとめて確認する
print("\n--- データの詳細情報 ---")

# 1. 貸借一致チェック（会計コンサルの基本！）
debit_total = df['借方金額'].sum()
credit_total = df['貸方金額'].sum()

print(f"--- 貸借チェック結果 ---")
if debit_total == credit_total:
    print(f"✅ OK: 貸借が一致しています（合計: {debit_total}）")
else:
    print(f"❌ NG: 貸借が不一致です（借方: {debit_total}, 貸方: {credit_total}）")

# 2. 必須項目の空欄（欠損値）チェック
print("\n--- 必須項目チェック ---")
missing_accounts = df['勘定科目コード'].isnull().sum()
if missing_accounts > 0:
    print(f"⚠️ 警告: 勘定科目コードが入力されていない行が {missing_accounts} 件あります。")
else:
    print("✅ OK: すべての行に勘定科目コードが入っています。")